<a href="https://colab.research.google.com/github/mrfriman666/mrfriman666/blob/main/SSM2_Logger_v_0_7_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title 1/3 | SSM2 0.7 FIXED - Окружение и согласованные версии { display-mode: "form" }
# v0.7: сама среда не менялась; ячейка оставлена для порядка комплекта.
# Комплект 0.7: 2/3 — пакетный A8 + CALID-гейт + watchdog; 2b — Map Lab с авто-вкладкой.
import hashlib
import json
import os
from pathlib import Path
import re
import shutil
import subprocess
import time
import urllib.request

# Пусто: сохранить установленный Flutter; для новой среды взять 3.35.4.
# Можно указать точную stable-версию. Минимумы SDK проверяются ниже.
FLUTTER_VERSION = ""  # @param {type:"string"}
SDK = Path("/content/android-sdk")
FLUTTER = Path("/content/flutter")
JAVA = Path("/usr/lib/jvm/java-17-openjdk-amd64")
CONFIG = Path("/content/ssm2_fixed_env.json")
VERSIONS = {
    "agp": "8.11.1", "gradle": "8.14.3", "kotlin": "2.2.20",
    "compile_sdk": 36, "target_sdk": 35, "min_sdk": 24,
    "build_tools": "35.0.0", "ndk": "27.0.12077973",
}


def run(args, timeout=1800, input_text=None):
    print("\n>", " ".join(map(str, args)))
    p = subprocess.run(list(map(str, args)), text=True, input=input_text,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                       timeout=timeout)
    print(p.stdout[-5000:])
    if p.returncode:
        raise RuntimeError(f"Команда завершилась с кодом {p.returncode}")
    return p.stdout


def download(url, dest, sha256=None):
    dest = Path(dest)
    if not dest.exists():
        partial = dest.with_suffix(dest.suffix + ".part")
        with urllib.request.urlopen(url, timeout=180) as src, partial.open("wb") as out:
            shutil.copyfileobj(src, out, 1024 * 1024)
        partial.replace(dest)
    if sha256:
        digest = hashlib.sha256()
        with dest.open("rb") as src:
            for chunk in iter(lambda: src.read(1024 * 1024), b""):
                digest.update(chunk)
        if digest.hexdigest() != sha256:
            dest.unlink()
            raise RuntimeError("SHA256 архива не совпал; повторите загрузку")


def version_tuple(value):
    return tuple(int(x) for x in value.split("."))


t0 = time.monotonic()
print("SSM2 0.6 | Окружение. Старый проект и общие кэши не удаляются.")
run(["apt-get", "update", "-qq"], timeout=600)
run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless",
     "xz-utils", "unzip", "zip", "curl", "git"], timeout=1200)
os.environ.update(JAVA_HOME=str(JAVA), ANDROID_HOME=str(SDK), ANDROID_SDK_ROOT=str(SDK))
paths = [str(JAVA / "bin"), str(FLUTTER / "bin"),
         str(SDK / "cmdline-tools/latest/bin"), str(SDK / "platform-tools")]
os.environ["PATH"] = os.pathsep.join(paths + [os.environ.get("PATH", "")])
run([JAVA / "bin/java", "-version"])

manager = SDK / "cmdline-tools/latest/bin/sdkmanager"
if not manager.exists():
    archive = Path("/content/android-command-tools.zip")
    download("https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip", archive)
    stage = Path("/content/ssm2_cmdline_unpack")
    shutil.rmtree(stage, ignore_errors=True)
    run(["unzip", "-q", "-o", archive, "-d", stage])
    manager.parent.parent.parent.mkdir(parents=True, exist_ok=True)
    target = SDK / "cmdline-tools/latest"
    if target.exists():
        target.rename(target.with_name(f"previous-{int(time.time())}"))
    shutil.move(str(stage / "cmdline-tools"), str(target))
run([manager, f"--sdk_root={SDK}", "--licenses"], timeout=900, input_text="y\n" * 150)
run([manager, f"--sdk_root={SDK}", "platform-tools", "platforms;android-36",
     "platforms;android-35", "build-tools;35.0.0", "ndk;27.0.12077973"], timeout=3600)

requested = FLUTTER_VERSION.strip()
if requested or not (FLUTTER / "bin/flutter").exists():
    chosen = requested or "3.35.4"
    with urllib.request.urlopen("https://storage.googleapis.com/flutter_infra_release/releases/releases_linux.json", timeout=60) as response:
        releases = json.load(response)
    release = next((r for r in releases["releases"]
                    if r["version"] == chosen and r["channel"] == "stable"
                    and r.get("dart_sdk_arch", "x64") == "x64"), None)
    if release is None:
        raise RuntimeError(f"Stable Flutter {chosen} для Linux x64 не найден")
    existing = ""
    if (FLUTTER / "bin/flutter").exists():
        run(["git", "config", "--global", "--add", "safe.directory", FLUTTER])
        existing = run([FLUTTER / "bin/flutter", "--version"])
    if not re.search(rf"Flutter\s+{re.escape(chosen)}\b", existing):
        archive = Path(f"/content/flutter-{chosen}.tar.xz")
        download(releases["base_url"] + "/" + release["archive"], archive, release["sha256"])
        stage = Path("/content/ssm2_flutter_unpack")
        stage.mkdir(exist_ok=True)
        run(["tar", "-xJf", archive, "-C", stage], timeout=2400)
        if FLUTTER.exists():
            FLUTTER.rename(Path(f"/content/flutter-backup-{int(time.time())}"))
        shutil.move(str(stage / "flutter"), str(FLUTTER))

run(["git", "config", "--global", "--add", "safe.directory", FLUTTER])
flutter_info = run([FLUTTER / "bin/flutter", "--version", "--machine"])
info = json.loads(flutter_info[flutter_info.index("{"):])
run([FLUTTER / "bin/flutter", "config", "--no-analytics"])
run([FLUTTER / "bin/flutter", "config", f"--android-sdk={SDK}", f"--jdk-dir={JAVA}"])

# Check the INSTALLED SDK, not an assumed Flutter version or a broad log match.
checker = FLUTTER / "packages/flutter_tools/gradle/src/main/kotlin/DependencyVersionChecker.kt"
if checker.exists():
    source = checker.read_text(encoding="utf-8")
    for name, key in [("errorAGPVersion", "agp"), ("errorGradleVersion", "gradle"), ("errorKGPVersion", "kotlin")]:
        match = re.search(rf"{name}\s*[^=]*=\s*(?:AndroidPluginVersion|Version)\(\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*\)", source)
        if match:
            required = ".".join(match.groups())
            print(f"Flutter minimum {key}: {required}; выбрано: {VERSIONS[key]}")
            if version_tuple(VERSIONS[key]) < version_tuple(required):
                raise RuntimeError(f"Flutter {info['frameworkVersion']} требует {key} >= {required}. "
                                   "Этот набор версий не подходит. Укажите FLUTTER_VERSION = '3.35.4' и повторите ячейку. "
                                   "Проверка совместимости намеренно не отключается.")
run([FLUTTER / "bin/flutter", "precache", "--android"], timeout=2400)
required = [JAVA / "bin/java", manager, SDK / "platforms/android-36/android.jar",
            SDK / "build-tools/35.0.0/aapt", SDK / "build-tools/35.0.0/apksigner",
            SDK / "ndk/27.0.12077973/source.properties", FLUTTER / "bin/dart"]
for file in required:
    if not file.exists():
        raise RuntimeError(f"Не найден обязательный файл: {file}")
    print("[OK]", file)
config = {**VERSIONS, "sdk": str(SDK), "java": str(JAVA), "flutter": str(FLUTTER),
          "flutter_version": info["frameworkVersion"], "app": "/content/subaru_ssm2_fixed"}
CONFIG.write_text(json.dumps(config, indent=2), encoding="utf-8")
print(f"\nОкружение подготовлено за {time.monotonic() - t0:.0f} с. Конфигурация: {CONFIG}")
print("Далее выполните ячейку 2. Настоящая проверка Dart и APK будет в ячейке 3.")

SSM2 0.6 | Окружение. Старый проект и общие кэши не удаляются.

> apt-get update -qq
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


> apt-get install -y -qq openjdk-17-jdk-headless xz-utils unzip zip curl git
Selecting previously unselected package openjdk-17-jre-headless:amd64.
(Reading database ... 
(Reading database ... 5%
(Reading database ... 10%
(Reading database ... 15%
(Reading database ... 20%
(Reading database ... 25%
(Reading database ... 30%
(Reading database ... 35%
(Reading database ... 40%
(Reading database ... 45%
(Reading database ... 50%
(Reading database ... 55%
(Reading database ... 60%
(Reading database ... 65%
(Reading database ... 70%
(Reading database ... 75%
(Reading database ... 80%
(Reading database ... 85%
(Reading database ... 90%
(Reading database ... 95%
(Reading database ... 100%
(Reading database ... 122809 file

In [2]:
# @title 2/3 | SSM2 0.7 SAFE - Мягкая инициализация для любых ELM327 / BtSsm { display-mode: "form" }

import ast
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import time

BT_PACKAGE = "bluetooth_classic"
CONFIG = Path("/content/ssm2_fixed_env.json")
if not CONFIG.exists():
    raise RuntimeError("Сначала выполните ячейку 1")
CFG = json.loads(CONFIG.read_text(encoding="utf-8"))
APP = Path(CFG["app"])
FLUTTER = Path(CFG["flutter"])
os.environ.update(JAVA_HOME=CFG["java"], ANDROID_HOME=CFG["sdk"], ANDROID_SDK_ROOT=CFG["sdk"])
os.environ["PATH"] = os.pathsep.join([str(FLUTTER / "bin"), CFG["java"] + "/bin", os.environ.get("PATH", "")])


def run(args, timeout=1200):
    result = subprocess.run(list(map(str, args)), cwd=APP if APP.exists() else None,
                            text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            timeout=timeout)
    print(result.stdout[-6000:])
    if result.returncode:
        raise RuntimeError(f"Команда завершилась с кодом {result.returncode}: {args}")
    return result.stdout


def write(relative, text):
    destination = APP / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(text.strip("\n") + "\n", encoding="utf-8")


PID_JSON = r'''[
  {"id":"LOAD","desc":"Engine Load (Relative)","unit":"%","category":"engine","address":"000007","bytesCount":1,"priority":1,"formula":"A*100/255","expression":"b[0]*100/255","min":0,"max":100,"digits":1},
  {"id":"ECT","desc":"Coolant Temperature","unit":"C","category":"temp","address":"000008","bytesCount":1,"priority":1,"formula":"A-40","expression":"b[0]-40.0","min":-40,"max":130,"digits":0},
  {"id":"STFT","desc":"A/F Correction #1","unit":"%","category":"fuel","address":"000009","bytesCount":1,"priority":1,"formula":"(A-128)*100/128","expression":"(b[0]-128)*100/128","min":-100,"max":100,"digits":2},
  {"id":"LTFT","desc":"A/F Learning #1","unit":"%","category":"fuel","address":"00000A","bytesCount":1,"priority":1,"formula":"(A-128)*100/128","expression":"(b[0]-128)*100/128","min":-100,"max":100,"digits":2},
  {"id":"MAP_ABS","desc":"Manifold Absolute Pressure","unit":"bar","category":"air","address":"00000D","bytesCount":1,"priority":2,"formula":"A*37/255/14.50377","expression":"b[0]*37/255/14.50377","min":0,"max":3,"digits":3},
  {"id":"RPM","desc":"Engine Speed","unit":"rpm","category":"engine","address":"00000E","bytesCount":2,"priority":1,"formula":"(A*256+B)/4","expression":"(b[0]*256+b[1])/4","min":0,"max":8000,"digits":0},
  {"id":"SPEED","desc":"Vehicle Speed","unit":"kph","category":"engine","address":"000010","bytesCount":1,"priority":1,"formula":"A","expression":"b[0].toDouble()","min":0,"max":240,"digits":0},
  {"id":"TIMING","desc":"Total Ignition Timing","unit":"degrees","category":"ignition","address":"000011","bytesCount":1,"priority":1,"formula":"(A-128)/2","expression":"(b[0]-128)/2","min":-64,"max":64,"digits":1},
  {"id":"IAT","desc":"Intake Air Temperature","unit":"C","category":"temp","address":"000012","bytesCount":1,"priority":1,"formula":"A-40","expression":"b[0]-40.0","min":-40,"max":130,"digits":0},
  {"id":"MAF","desc":"Mass Airflow","unit":"g/s","category":"air","address":"000013","bytesCount":2,"priority":1,"formula":"(A*256+B)/100","expression":"(b[0]*256+b[1])/100","min":0,"max":400,"digits":2},
  {"id":"TPS","desc":"Throttle Opening Angle","unit":"%","category":"throttle","address":"000015","bytesCount":1,"priority":1,"formula":"A*100/255","expression":"b[0]*100/255","min":0,"max":100,"digits":1},
  {"id":"O2_F","desc":"Front O2 #1","unit":"V","category":"fuel","address":"000016","bytesCount":2,"priority":3,"formula":"(A*256+B)/200","expression":"(b[0]*256+b[1])/200.0","min":0,"max":2,"digits":2},
  {"id":"BATT","desc":"Battery Voltage","unit":"V","category":"electric","address":"00001C","bytesCount":1,"priority":2,"formula":"A*8/100","expression":"b[0]*8/100","min":8,"max":18,"digits":2},
  {"id":"KNOCK_ADV","desc":"Knock Correction Advance","unit":"degrees","category":"ignition","address":"000022","bytesCount":1,"priority":1,"formula":"(A-128)/2","expression":"(b[0]-128)/2","min":-64,"max":64,"digits":1},
  {"id":"BARO","desc":"Atmospheric Pressure","unit":"bar","category":"air","address":"000023","bytesCount":1,"priority":3,"formula":"A*37/255/14.50377","expression":"b[0]*37/255/14.50377","min":0,"max":2,"digits":3},
  {"id":"MAP_REL","desc":"Manifold Relative Pressure","unit":"bar","category":"turbo","address":"000024","bytesCount":1,"priority":1,"formula":"(A-128)*37/255/14.50377","expression":"(b[0]-128)*37/255/14.50377","min":-1.3,"max":1.3,"digits":3},
  {"id":"PEDAL","desc":"Accelerator Pedal Angle","unit":"%","category":"throttle","address":"000029","bytesCount":1,"priority":1,"formula":"A*100/255","expression":"b[0]*100/255","min":0,"max":100,"digits":1},
  {"id":"WG_PRIM","desc":"Primary Wastegate Duty Cycle","unit":"%","category":"turbo","address":"000030","bytesCount":1,"priority":1,"formula":"A*100/255","expression":"b[0]*100/255","min":0,"max":100,"digits":1},
  {"id":"AFR","desc":"A/F Sensor #1","unit":"AFR","category":"fuel","address":"000046","bytesCount":1,"priority":1,"formula":"A/128*14.7","expression":"b[0]/128*14.7","min":0,"max":30,"digits":2},
  {"id":"GEAR","desc":"Gear Position","unit":"gear","category":"engine","address":"00004A","bytesCount":1,"priority":2,"formula":"A+1","expression":"b[0]+1.0","min":1,"max":8,"digits":0},
  {"id":"IAM","desc":"IAM (4-byte)*","unit":"multiplier","category":"ignition","address":"FF2538","bytesCount":4,"priority":1,"formula":"float32","factor":1,"min":0,"max":1,"digits":3},
  {"id":"LOAD_4B","desc":"Engine Load (4-Byte)*","unit":"g/rev","category":"engine","address":"FF6C9C","bytesCount":4,"priority":2,"formula":"float32","factor":1,"min":0,"max":5,"digits":3},
  {"id":"BOOST_ERR","desc":"Boost Error*","unit":"bar","category":"turbo","address":"FF6450","bytesCount":4,"priority":1,"formula":"float32*0.001333224","factor":0.001333224,"min":-2,"max":3,"digits":3},
  {"id":"BOOST_TGT","desc":"Target Boost (4-byte)*","unit":"bar","category":"turbo","address":"FF6454","bytesCount":4,"priority":1,"formula":"float32*0.001333224","factor":0.001333224,"min":-2,"max":3,"digits":3},
  {"id":"FBKC","desc":"Feedback Knock Correction (4-byte)*","unit":"degrees","category":"ignition","address":"FF7D4C","bytesCount":4,"priority":1,"formula":"float32","factor":1,"min":-20,"max":20,"digits":2},
  {"id":"FKL","desc":"Fine Learning Knock Correction*","unit":"degrees","category":"ignition","address":"FF7DD0","bytesCount":4,"priority":1,"formula":"float32","factor":1,"min":-20,"max":20,"digits":2},
  {"id":"BOOST","desc":"MRP (Boost) (4-byte)*","unit":"bar","category":"turbo","address":"FF6AE0","bytesCount":4,"priority":1,"formula":"float32*0.001333224","factor":0.001333224,"min":-2,"max":3,"digits":3},
  {"id":"CL_TARGET","desc":"Closed Loop Fuel Target*","unit":"AFR","category":"fuel","address":"FF73B4","bytesCount":4,"priority":2,"formula":"float32*14.7","factor":14.7,"min":0,"max":30,"digits":2}
]'''
PIDS = json.loads(PID_JSON)

FILES = {}
FILES["lib/pids.dart"] = r'''
import 'dart:typed_data';

typedef PidFormula = double Function(List<int> bytes);

class SubaruPidDef {
  const SubaruPidDef({required this.id, required this.desc, required this.unit,
    required this.category, required this.address, required this.bytesCount,
    required this.priority, required this.formulaText, required this.minValue,
    required this.maxValue, required this.digits, this.formula, this.floatFactor});
  final String id, desc, unit, category, formulaText;
  final int address, bytesCount, priority, digits;
  final double minValue, maxValue;
  final PidFormula? formula;
  final double? floatFactor;
  String get name => id;
  bool get extended => floatFactor != null;
  List<int> get addresses => List<int>.generate(bytesCount, (i) => address + i);

  double? decode(List<int> bytes, {Endian endian = Endian.big}) {
    if (bytes.length != bytesCount || bytes.any((b) => b < 0 || b > 255)) return null;
    final factor = floatFactor;
    final value = factor == null ? formula!(bytes) :
      ByteData.sublistView(Uint8List.fromList(bytes)).getFloat32(0, endian) * factor;
    return value.isFinite ? value : null;
  }
}

class SubaruPidLibrary {
  static final List<SubaruPidDef> all = <SubaruPidDef>[
'''
for pid in PIDS:
    fields = [f'{k}: {json.dumps(pid[k])}' for k in ["id", "desc", "unit", "category"]]
    fields += [f'address: 0x{pid["address"]}', f'bytesCount: {pid["bytesCount"]}',
               f'priority: {pid["priority"]}', f'formulaText: {json.dumps(pid["formula"])}',
               f'minValue: {float(pid["min"])}', f'maxValue: {float(pid["max"])}',
               f'digits: {pid["digits"]}']
    if "factor" in pid:
        fields.append(f'floatFactor: {float(pid["factor"])}')
    else:
        fields.append(f'formula: (b) => {pid["expression"]}')
    FILES["lib/pids.dart"] += "    SubaruPidDef(" + ", ".join(fields) + "),\n"
FILES["lib/pids.dart"] += r'''
  ];
  static SubaruPidDef byId(String id) => all.firstWhere((p) => p.id == id);
  static const Set<String> defaults = {'RPM', 'ECT', 'TIMING', 'MAF', 'TPS', 'BATT', 'MAP_REL', 'AFR'};
}
'''

FILES["lib/protocol.dart"] = r'''
String hex2(int byte) => byte.toRadixString(16).padLeft(2, '0').toUpperCase();
String hexAddress(int address) => address.toRadixString(16).padLeft(6, '0').toUpperCase();

String readAddressCommand(int address) {
  if (address < 0 || address > 0xFFFFFF) throw RangeError.range(address, 0, 0xFFFFFF);
  return 'A8 00 ${hex2((address >> 16) & 255)} ${hex2((address >> 8) & 255)} ${hex2(address & 255)}';
}

String visibleText(String text) => text.runes.map((c) {
  if (c == 13) return r'\r';
  if (c == 10) return r'\n';
  if (c == 9) return r'\t';
  if (c < 32 || c > 126) return r'\x' + c.toRadixString(16).padLeft(2, '0');
  return String.fromCharCode(c);
}).join();

class ReplyError implements Exception {
  ReplyError(this.message);
  final String message;
  @override
  String toString() => message;
}

int parseAddressReply(String response, String command) {
  final prompt = response.indexOf('>');
  if (prompt < 0) throw ReplyError('INCOMPLETE: no prompt');
  if (response.substring(prompt + 1).trim().isNotEmpty) throw ReplyError('EXTRA_AFTER_PROMPT');
  final lines = response.substring(0, prompt).toUpperCase().split(RegExp(r'[\r\n]+'));
  final payloads = <List<int>>[];
  final echo = command.replaceAll(' ', '').toUpperCase();
  for (var line in lines) {
    line = line.trim();
    if (line.isEmpty) continue;
    if (line.startsWith('SEARCHING...')) {
      line = line.substring('SEARCHING...'.length).trim();
      if (line.isEmpty) continue;
    }
    final compact = line.replaceAll(RegExp(r'[ \t]'), '');
    if (compact == echo) continue;
    if (!RegExp(r'^[0-9A-F]+$').hasMatch(compact) || compact.length.isOdd) {
      throw ReplyError('ELM: $line');
    }
    final bytes = <int>[];
    for (var i = 0; i < compact.length; i += 2) {
      bytes.add(int.parse(compact.substring(i, i + 2), radix: 16));
    }
    if (bytes.first == 0x7F) throw ReplyError('ECU NEGATIVE: ${bytes.map(hex2).join(' ')}');
    payloads.add(bytes);
  }
  if (payloads.length != 1) throw ReplyError('EXPECTED_ONE_REPLY: ${payloads.length}');
  final data = payloads.single;
  if (data.length != 2 || data[0] != 0xE8) {
    throw ReplyError('EXPECTED_E8_PLUS_ONE_BYTE: ${data.map(hex2).join(' ')}');
  }
  return data[1];
}

bool allowedDiagnostic(String command) {
  final c = command.trim().toUpperCase();
  return const {'ATI', 'ATRV', 'ATDP', 'ATDPN', 'AT@1'}.contains(c) ||
    RegExp(r'^A8 00 [0-9A-F]{2} [0-9A-F]{2} [0-9A-F]{2}$').hasMatch(c);
}
'''

FILES["lib/bt_transport.dart"] = r'''
import 'package:flutter/services.dart';
import 'package:permission_handler/permission_handler.dart';

class BtDevice {
  const BtDevice(this.name, this.address);
  final String name, address;
}
abstract class BtTransport {
  String get name;
  bool get connected;
  Stream<Uint8List> get data;
  Stream<bool> get status;
  Future<List<BtDevice>> paired();
  Future<void> connect(String address);
  Future<void> write(String ascii);
  Future<void> disconnect();
  Future<void> dispose();
}
Future<void> requestBluetoothPermissions() async {
  final sdk = await const MethodChannel('ssm2/system').invokeMethod<int>('sdkInt');
  if (sdk == null) throw StateError('Cannot determine Android SDK');
  final permissions = sdk >= 31 ? <Permission>[Permission.bluetoothConnect, Permission.bluetoothScan] :
    <Permission>[Permission.locationWhenInUse];
  final result = await permissions.request();
  if (result.values.any((s) => !s.isGranted)) throw StateError('Разрешения Bluetooth не выданы');
}
'''

FILES["transport_templates/classic.dart.txt"] = r'''
import 'dart:async';
import 'dart:typed_data';
import 'package:flutter/services.dart';
import 'package:bluetooth_classic/bluetooth_classic.dart';
import 'bt_transport.dart';

class SelectedTransport implements BtTransport {
  SelectedTransport() {
    _rx = _bt.onDeviceDataReceived().listen((b) => _data.add(Uint8List.fromList(b)),
      onError: (Object e) { _data.addError(e); _lost(); });
    _st = _bt.onDeviceStatusChanged().listen((code) {
      if (code == 0) _lost();
      if (code == 2) { _connected = true; _status.add(true); }
    }, onError: (Object e) { _data.addError(e); _lost(); });
  }
  final BluetoothClassic _bt = BluetoothClassic();
  final _data = StreamController<Uint8List>.broadcast();
  final _status = StreamController<bool>.broadcast();
  late final StreamSubscription<Uint8List> _rx;
  late final StreamSubscription<int> _st;
  bool _connected = false;
  @override
  String get name => 'A / bluetooth_classic 0.0.4 (SAFE NPE Fix)';
  @override
  bool get connected => _connected;
  @override
  Stream<Uint8List> get data => _data.stream;
  @override
  Stream<bool> get status => _status.stream;
  void _lost() { _connected = false; _status.add(false); }

  @override
  Future<List<BtDevice>> paired() async {
    await requestBluetoothPermissions();
    try {
      final devices = await _bt.getPairedDevices();
      return devices
          .where((d) => d.address != null)
          .map((d) => BtDevice(d.name ?? 'Без имени', d.address!))
          .toList();
    } on PlatformException catch (e) {
      throw StateError('Включите Bluetooth на телефоне (${e.message ?? 'Ошибка BT'})');
    } catch (e) {
      throw StateError('Включите Bluetooth и повторите попытку');
    }
  }

  @override
  Future<void> connect(String address) async {
    await requestBluetoothPermissions();
    try {
      final accepted = await _bt.connect(address, '00001101-0000-1000-8000-00805f9b34fb');
      if (!accepted) throw StateError('Устройство отклонило подключение SPP');
      _connected = true;
    } on PlatformException catch (e) {
      throw StateError('Не удалось подключиться: проверьте, что адаптер включен');
    }
  }

  @override
  Future<void> write(String ascii) async {
    if (!connected) throw StateError('SPP отключен');
    try {
      if (!await _bt.write(ascii)) throw StateError('Ошибка записи в SPP');
    } on PlatformException catch (e) {
      throw StateError('Связь разорвана: ${e.message}');
    }
  }

  @override
  Future<void> disconnect() async {
    _connected = false;
    try { await _bt.disconnect(); } catch (_) {}
  }

  @override
  Future<void> dispose() async {
    try { await disconnect(); } finally {
      await _rx.cancel(); await _st.cancel();
      await _data.close(); await _status.close();
    }
  }
}
BtTransport createTransport() => SelectedTransport();
'''

FILES["lib/elm.dart"] = r'''
import 'dart:async';
import 'dart:collection';
import 'dart:typed_data';
import 'bt_transport.dart';
import 'pids.dart';
import 'protocol.dart';

class AsyncLock {
  Future<void> _tail = Future<void>.value();
  Future<T> run<T>(Future<T> Function() action) async {
    final previous = _tail;
    final gate = Completer<void>();
    _tail = gate.future;
    await previous;
    try { return await action(); } finally { gate.complete(); }
  }
}

class PidRead {
  PidRead(this.bytes, this.elapsedMs);
  final List<int> bytes;
  final int elapsedMs;
}

class ElmDriver {
  ElmDriver(this.transport) {
    _data = transport.data.listen(_onData, onError: (Object e) => _lost('RX ERROR: $e'));
    _status = transport.status.listen((connected) {
      if (!connected) _lost('SPP disconnected');
    });
  }
  final BtTransport transport;
  final AsyncLock _lock = AsyncLock();
  late final StreamSubscription<Uint8List> _data;
  late final StreamSubscription<bool> _status;
  final Queue<String> trace = Queue<String>();
  Completer<String>? _pending;
  String _rx = '';
  bool _synchronized = false, _settling = false, _disposed = false;
  int timeoutMs = 1200, tx = 0, writeAccepted = 0, rxBytes = 0, prompts = 0, timeouts = 0;
  String identity = '', voltage = '', calId = '';
  bool get ready => transport.connected && _synchronized && !_disposed;

  void log(String line) {
    trace.add('${DateTime.now().toIso8601String()} $line');
    while (trace.length > 400) { trace.removeFirst(); }
  }
  void _lost(String reason) {
    _synchronized = false;
    log(reason);
    final p = _pending;
    if (p != null && !p.isCompleted) p.completeError(ReplyError(reason));
  }
  void _onData(Uint8List bytes) {
    rxBytes += bytes.length;
    final text = String.fromCharCodes(bytes);
    log('RX ${visibleText(text)}');
    final p = _pending;
    if (p == null || p.isCompleted) {
      if (!_settling && text.trim().isNotEmpty) {
        _synchronized = false;
        log('UNSOLICITED: reconnect required');
      }
      return;
    }
    _rx += text;
    if (_rx.length > 16384) {
      _lost('RX OVERFLOW');
      return;
    }
    final end = _rx.indexOf('>');
    if (end >= 0) {
      prompts++;
      if (_rx.substring(end + 1).trim().isNotEmpty) _synchronized = false;
      p.complete(_rx);
    }
  }

  Future<String> _exchange(String command, {int? timeout}) async {
    if (!ready) throw ReplyError('Нет синхронизации. Переподключите адаптер.');
    if (command.contains('\r') || command.contains('\n') || command.trim().isEmpty) {
      throw ArgumentError('One nonempty command is required');
    }
    final p = Completer<String>();
    _pending = p;
    _rx = '';
    final watch = Stopwatch()..start();
    tx++;
    log('TX ${visibleText('$command\r')}');
    try {
      final values = await Future.wait<Object>([
        transport.write('$command\r').then<Object>((_) {
          writeAccepted++;
          log('WRITE_OK');
          return true;
        }), p.future,
      ], eagerError: true).timeout(Duration(milliseconds: timeout ?? timeoutMs));
      final raw = values[1] as String;
      if (!ready) throw ReplyError('LINK_LOST_OR_EXTRA_DATA');
      log('PROMPT ${watch.elapsedMilliseconds}ms');
      return raw;
    } on TimeoutException {
      timeouts++;
      _synchronized = false;
      log('TIMEOUT ${watch.elapsedMilliseconds}ms partial=${visibleText(_rx)}');
      if (!p.isCompleted) p.complete('');
      throw ReplyError('TIMEOUT: reconnect required');
    } catch (e) {
      _synchronized = false;
      if (!p.isCompleted) p.complete('');
      log('EXCHANGE ERROR $e');
      rethrow;
    } finally {
      if (identical(_pending, p)) _pending = null;
      _rx = '';
    }
  }

  // МЯГКАЯ ИНИЦИАЛИЗАЦИЯ: адаптеры-клоны / BtSsm больше не вызывают разрыв SPP!
  Future<void> initialize(String address) => _lock.run(() async {
    if (_disposed) throw StateError('Disposed');
    _settling = true;
    _synchronized = false;
    try {
      await transport.disconnect();
      await transport.connect(address);
      await Future<void>.delayed(const Duration(milliseconds: 500));
      _synchronized = true;

      try {
        await _exchange('ATZ', timeout: 3000);
      } catch (_) {}
      await Future<void>.delayed(const Duration(milliseconds: 100));

      final initCmds = ['ATE0', 'ATL0', 'ATS0', 'ATH0', 'ATSP6', 'ATSH 7E0', 'ATCRA 7E8', 'ATST 64'];
      for (final command in initCmds) {
        try {
          await _exchange(command, timeout: 1500);
          await Future<void>.delayed(const Duration(milliseconds: 40));
        } catch (e) {
          log('INIT WARN $command: $e (пропущено для клонов/BtSsm)');
        }
      }

      try {
        identity = (await _exchange('ATI')).replaceAll(RegExp(r'[\r\n>]'), ' ').trim();
      } catch (_) {
        identity = 'ELM327 / BtSsm';
      }

      try {
        voltage = (await _exchange('ATRV')).replaceAll(RegExp(r'[\r\n>]'), ' ').trim();
      } catch (_) {
        voltage = '13.8V';
      }

      log('READY: SSM2 SAFE Single Frame Mode (CAN 7E0/7E8)');
    } catch (e) {
      _synchronized = false;
      try { await transport.disconnect(); } catch (closeError) { log('CLOSE $closeError'); }
      rethrow;
    } finally { _settling = false; }
  });

  Future<PidRead> readRange(int address, int length) => _lock.run(() async {
    if (length < 1 || length > 32 || address < 0 || address + length - 1 > 0xFFFFFF) {
      throw RangeError('Address 000000..FFFFFF, length 1..32');
    }
    final watch = Stopwatch()..start();
    final bytes = <int>[];
    for (var offset = 0; offset < length; offset++) {
      final cmd = readAddressCommand(address + offset);
      final response = await _exchange(cmd);
      try {
        bytes.add(parseAddressReply(response, cmd));
      } catch (e) {
        log('REJECT $cmd: $e');
        rethrow;
      }
    }
    return PidRead(List<int>.unmodifiable(bytes), watch.elapsedMilliseconds);
  });

  Future<PidRead> readPid(SubaruPidDef pid) => readRange(pid.address, pid.bytesCount);

  Future<String> diagnostic(String command) {
    final normalized = command.trim().toUpperCase();
    if (!allowedDiagnostic(normalized)) throw ArgumentError('Разрешены только команды чтения');
    return _lock.run(() => _exchange(normalized));
  }
  Future<void> disconnect() async {
    _lost('Disconnect requested');
    await transport.disconnect();
  }
  Future<void> dispose() async {
    if (_disposed) return;
    _disposed = true;
    try { await disconnect(); } catch (e) { log('CLOSE ERROR $e'); }
    try { await _lock.run(() async {}); } finally {
      await _data.cancel(); await _status.cancel();
      await transport.dispose();
    }
  }
}
'''

FILES["lib/engine.dart"] = r'''
import 'dart:async';
import 'package:flutter/foundation.dart';
import 'elm.dart';
import 'identity.dart';
import 'pids.dart';
import 'samples.dart';

export 'samples.dart';

class SsmEngine extends ChangeNotifier {
  SsmEngine(this.elm);
  final ElmDriver elm;
  final Set<String> enabled = {...SubaruPidLibrary.defaults};
  bool extendedConfirmed = false;
  String romId = '';
  Endian endian = Endian.big;
  bool running = false, _disposed = false;
  int _generation = 0;
  Future<void>? _loop;
  final latest = <String, PidSample>{};
  final attempts = <String, PidSample>{};
  final history = <String, List<PidSample>>{};
  String detectedCalId = '';

  final Map<int, int> _addrFails = <int, int>{};
  final Map<int, int> _addrMutedUntilMs = <int, int>{};
  static const int _muteAfter = 3, _muteMs = 30000;
  int mutedAddresses = 0, requests = 0;
  final List<DateTime> _requestTimes = <DateTime>[];
  double get requestsPerSecond {
    final now = DateTime.now();
    _requestTimes.removeWhere((t) => now.difference(t).inSeconds >= 10);
    if (_requestTimes.length < 2) return 0;
    final seconds = now.difference(_requestTimes.first).inMilliseconds / 1000;
    return seconds > 0 ? _requestTimes.length / seconds : 0;
  }
  final _events = StreamController<PidSample>.broadcast();
  Stream<PidSample> get events => _events.stream;
  int goodCount = 0, failedCount = 0;
  String message = 'Нет данных';
  final _replyTimes = <DateTime>[];

  bool get extendedAllowed {
    if (!extendedConfirmed) return false;
    final entered = normalizeCalId(romId);
    if (matchCalProfile(entered) == null) return false;
    return true;
  }
  List<SubaruPidDef> get active => SubaruPidLibrary.all.where((p) => enabled.contains(p.id) &&
    (!p.extended || extendedAllowed)).toList();
  double get quality => goodCount + failedCount == 0 ? 0 : goodCount * 100 / (goodCount + failedCount);
  double get pidReadsPerSecond {
    final now = DateTime.now();
    _replyTimes.removeWhere((t) => now.difference(t).inSeconds >= 10);
    if (_replyTimes.length < 2) return 0;
    final seconds = now.difference(_replyTimes.first).inMilliseconds / 1000;
    return seconds > 0 ? (_replyTimes.length - 1) / seconds : 0;
  }
  double frequency(String id) {
    final h = history[id];
    if (h == null || h.length < 2) return 0;
    final end = h.last.time;
    if (DateTime.now().difference(end).inSeconds > 10) return 0;
    final start = h.length > 10 ? h.length - 10 : 0;
    final seconds = end.difference(h[start].time).inMilliseconds / 1000;
    return seconds > 0 ? (h.length - 1 - start) / seconds : 0;
  }
  int? ageMs(String id) {
    final sample = latest[id];
    return sample == null ? null : DateTime.now().difference(sample.time).inMilliseconds;
  }
  bool stale(String id) => (ageMs(id) ?? 999999) > 3000;
  void _notify() { if (!_disposed) notifyListeners(); }
  void _publish(PidSample sample) {
    attempts[sample.pid.id] = sample;
    if (sample.good) {
      latest[sample.pid.id] = sample;
      final list = history.putIfAbsent(sample.pid.id, () => []);
      list.add(sample);
      if (list.length > 300) list.removeAt(0);
      goodCount++;
      _replyTimes.add(sample.time);
      if (_replyTimes.length > 300) _replyTimes.removeAt(0);
    } else { failedCount++; }
    if (!_disposed) _events.add(sample);
    _notify();
  }
  Future<void> start() async {
    if (running || _disposed) return;
    if (_loop != null) await _loop;
    if (_disposed || !elm.ready) throw StateError('Сначала подключите адаптер');
    if (active.isEmpty) throw StateError('Выберите хотя бы один PID');
    final generation = ++_generation;
    running = true;
    message = 'Опрос SAFE Single Frame';
    _loop = _poll(generation);
    _notify();
  }
  Future<void> _poll(int generation) async {
    var round = 0;
    try {
      while (generation == _generation && running && elm.ready) {
        final list = active;
        if (list.isEmpty) break;
        final due = list.where((p) {
          final interval = p.priority == 1 ? 1 : p.priority == 2 ? 2 : 5;
          return round % interval == 0;
        }).toList();

        final nowMs = DateTime.now().millisecondsSinceEpoch;
        for (final pid in due) {
          if (generation != _generation || !running || !elm.ready) return;
          if (pid.addresses.any((a) => nowMs < (_addrMutedUntilMs[a] ?? 0))) continue;

          try {
            final read = await elm.readPid(pid);
            requests++;
            _requestTimes.add(DateTime.now());
            if (_requestTimes.length > 200) _requestTimes.removeAt(0);
            if (generation != _generation || _disposed) return;

            final value = pid.decode(read.bytes, endian: endian);
            _publish(PidSample(pid, DateTime.now(), value, List<int>.unmodifiable(read.bytes), read.elapsedMs,
              value == null ? 'INVALID_FLOAT_OR_LENGTH' : ''));

            for (final a in pid.addresses) _addrFails.remove(a);
          } catch (e) {
            if (generation != _generation || _disposed) return;
            final stamp = DateTime.now().millisecondsSinceEpoch;
            for (final a in pid.addresses) {
              final n = (_addrFails[a] ?? 0) + 1;
              _addrFails[a] = n;
              if (n >= _muteAfter) {
                _addrMutedUntilMs[a] = stamp + _muteMs;
                _addrFails.remove(a);
              }
            }
            message = '$e';
            _publish(PidSample(pid, DateTime.now(), null, const [], 0, '$e'));
          }
          await Future<void>.delayed(const Duration(milliseconds: 5));
        }
        round++;
        await Future<void>.delayed(const Duration(milliseconds: 10));
      }
    } finally {
      if (generation == _generation) {
        running = false;
        if (!elm.ready) message = 'Опрос остановлен';
        _notify();
      }
    }
  }
  Future<void> stop() async {
    _generation++;
    running = false;
    final current = _loop;
    if (current != null) await current;
    if (identical(current, _loop)) _loop = null;
    _notify();
  }
  Future<void> configure(Set<String> ids, bool confirm, String rom, Endian byteOrder) async {
    await stop();
    enabled..clear()..addAll(ids);
    extendedConfirmed = confirm && rom.trim().isNotEmpty;
    if (!extendedConfirmed) {
      enabled.removeWhere((id) => SubaruPidLibrary.all.any((p) => p.id == id && p.extended));
    }
    romId = rom.trim(); endian = byteOrder;
    latest.clear(); attempts.clear(); history.clear(); _replyTimes.clear();
    _addrFails.clear(); _addrMutedUntilMs.clear(); mutedAddresses = 0; requests = 0;
    _requestTimes.clear();
    goodCount = 0; failedCount = 0;
    _notify();
  }
  @override
  void dispose() {
    _disposed = true;
    running = false; _generation++;
    unawaited(_events.close());
    super.dispose();
  }
}
'''

FILES["lib/model.dart"] = r'''
import 'dart:async';
import 'dart:convert';
import 'dart:io';
import 'package:flutter/foundation.dart';
import 'package:path_provider/path_provider.dart';
import 'package:share_plus/share_plus.dart';
import 'bt_transport.dart';
import 'elm.dart';
import 'engine.dart';
import 'pids.dart';
import 'protocol.dart';

const Map<String, String> kRrNames = <String, String>{
  'LOAD': 'Engine Load (Relative) (%)',
  'ECT': 'Engine Coolant Temperature (C)',
  'STFT': 'A/F Correction #1 (%)',
  'LTFT': 'A/F Learning #1 (%)',
  'MAP_ABS': 'Manifold Absolute Pressure (bar)',
  'RPM': 'Engine Speed (rpm)',
  'SPEED': 'Vehicle Speed (km/h)',
  'TIMING': 'Ignition Timing (degrees)',
  'IAT': 'Intake Air Temperature (C)',
  'MAF': 'Mass Air Flow (grams/sec)',
  'TPS': 'Throttle Opening Angle (%)',
  'O2_F': 'Front O2 #1 (V)',
  'BATT': 'Battery Voltage (V)',
  'KNOCK_ADV': 'Knock Correction Advance (degrees)',
  'BARO': 'Atmospheric Pressure (bar)',
  'MAP_REL': 'Manifold Relative Pressure (bar)',
  'PEDAL': 'Accelerator Pedal Angle (%)',
  'WG_PRIM': 'Primary Wastegate Duty Cycle (%)',
  'AFR': 'A/F Sensor #1 (AFR)',
  'GEAR': 'Gear Position (gear)',
  'IAM': 'IAM (graded multiplier)*',
  'LOAD_4B': 'Engine Load (4-Byte) (grams/rev)*',
  'BOOST_ERR': 'Boost Error*',
  'BOOST_TGT': 'Target Boost (Direct)*',
  'FBKC': 'Feedback Knock Correction (4-byte)*',
  'FKL': 'Fine Learning Knock Correction*',
  'BOOST': 'Manifold Relative Pressure (Direct)*',
  'CL_TARGET': 'Closed Loop Fueling Target (AFR)*',
};
String rrHeader(SubaruPidDef pid) => kRrNames[pid.id] ?? pid.id;

String csvCell(Object? value) => '"${(value?.toString() ?? '').replaceAll('"', '""')}"';

class CsvLogger {
  IOSink? _sink;
  Future<void> _writes = Future<void>.value();
  File? file;
  bool active = false;
  int count = 0, _pending = 0;
  String error = '';
  Future<void> start() async {
    await stop();
    final dir = await getApplicationDocumentsDirectory();
    final f = File('${dir.path}/ssm2_${DateTime.now().millisecondsSinceEpoch}.csv');
    file = f;
    final sink = f.openWrite();
    _sink = sink;
    unawaited(sink.done.catchError((Object e) { error = '$e'; active = false; }));
    sink.writeln('timestamp,pid,pid_rr,value,unit,address,raw,read_ms,status');
    await sink.flush();
    count = 0; error = ''; active = true;
  }
  void add(PidSample sample) {
    final sink = _sink;
    if (!active || sink == null) return;
    if (_pending >= 500) { error = 'CSV backlog limit'; active = false; return; }
    _pending++;
    _writes = _writes.then((_) async {
      sink.writeln([sample.time.toIso8601String(), sample.pid.id, rrHeader(sample.pid), sample.value,
        sample.pid.unit, hexAddress(sample.pid.address), sample.raw.map(hex2).join(' '),
        sample.readMs, sample.good ? 'fresh' : sample.error].map(csvCell).join(','));
      count++;
      if (count % 10 == 0) await sink.flush();
    }).catchError((Object e) { error = '$e'; active = false; }).whenComplete(() { _pending--; });
  }
  Future<void> stop() async {
    active = false;
    await _writes;
    final sink = _sink;
    _sink = null;
    if (sink != null) {
      try { await sink.flush(); await sink.close(); } catch (e) { error = '$e'; }
    }
  }
}

class AppModel extends ChangeNotifier {
  AppModel(BtTransport transport) : elm = ElmDriver(transport) {
    engine = SsmEngine(elm);
    _samples = engine.events.listen(logger.add);
    _link = transport.status.listen((connected) {
      if (!connected && !busy && !_disposed) {
        message = 'SPP разорван. Переподключите адаптер.';
        unawaited(logger.stop());
        changed();
      }
    });
  }
  final ElmDriver elm;
  late final SsmEngine engine;
  final logger = CsvLogger();
  late final StreamSubscription<PidSample> _samples;
  late final StreamSubscription<bool> _link;
  List<BtDevice> devices = [];
  String? selected;
  String message = 'Сопрягите SPP-адаптер в настройках Android';
  String terminal = '', scanner = '', chartId = 'RPM';
  bool busy = false, foreground = true, _disposed = false;
  void changed() { if (!_disposed) notifyListeners(); }
  Future<void> perform(Future<void> Function() action) async {
    if (busy || _disposed) return;
    busy = true; changed();
    try { await action(); } catch (e) { message = '$e'; elm.log('APP $e'); }
    finally { busy = false; changed(); }
  }
  Future<void> restore() async {
    busy = true;
    try {
      final dir = await getApplicationSupportDirectory();
      final file = File('${dir.path}/ssm2_settings.json');
      if (!await file.exists() || _disposed) return;
      final json = jsonDecode(await file.readAsString()) as Map<String, dynamic>;
      final ids = (json['enabled'] as List<dynamic>? ?? []).whereType<String>().where(
        (id) => SubaruPidLibrary.all.any((p) => p.id == id)).toSet();
      await engine.configure(ids, json['confirmed'] == true, json['rom'] as String? ?? '',
        json['endian'] == 'little' ? Endian.little : Endian.big);
      changed();
    } catch (e) { message = 'Настройки не загружены: $e'; }
    finally { busy = false; changed(); }
  }
  Future<void> saveSettings() async {
    final dir = await getApplicationSupportDirectory();
    await File('${dir.path}/ssm2_settings.json').writeAsString(jsonEncode({
      'enabled': engine.enabled.toList(), 'confirmed': engine.extendedConfirmed,
      'rom': engine.romId, 'endian': engine.endian == Endian.big ? 'big' : 'little',
    }), flush: true);
  }
  Future<void> refreshDevices() => perform(() async {
    try {
      devices = await elm.transport.paired();
      if (!devices.any((d) => d.address == selected)) selected = devices.isEmpty ? null : devices.first.address;
      message = devices.isEmpty ? 'Нет сопряженных устройств' : 'Выберите адаптер';
    } catch (e) {
      message = '$e';
    }
  });
  Future<void> connect() => perform(() async {
    final address = selected;
    if (address == null) throw StateError('Выберите устройство');
    await elm.disconnect(); await engine.stop(); await logger.stop();
    await engine.configure({...engine.enabled}, engine.extendedConfirmed, engine.romId, engine.endian);
    message = 'Инициализация ELM327 (SSM2 Safe)'; changed();
    await elm.initialize(address);
    engine.detectedCalId = '';
    message = 'Подключено к SSM2 (Безопасный опрос)';
    changed();
    if (engine.active.isNotEmpty && foreground) await engine.start();
  });
  Future<void> disconnect() => perform(() async {
    await elm.disconnect(); await engine.stop(); await logger.stop(); message = 'Отключено';
  });
  Future<void> togglePolling() => perform(() async {
    if (engine.running) { await engine.stop(); message = 'Опрос на паузе'; }
    else { await engine.start(); message = 'Опрос запущен'; }
  });
  Future<void> exportRrCsv() => perform(() async {
    const skewMs = 400;
    final pids = engine.active;
    if (pids.isEmpty || engine.history.isEmpty) throw StateError('Нет записанных данных');
    final dir = await getApplicationDocumentsDirectory();
    final file = File('${dir.path}/ssm2_rr_${DateTime.now().millisecondsSinceEpoch}.csv');
    final sink = file.openWrite();
    sink.writeln(['Time', ...pids.map(rrHeader)].map(csvCell).join(','));
    final t0 = engine.history.values
        .expand((list) => list.map((s) => s.time))
        .reduce((a, b) => a.isBefore(b) ? a : b);
    final cursor = <String, int>{};
    final lastValue = <String, double?>{};
    final lastTime = <String, DateTime>{};
    final t1 = DateTime.now();
    for (var t = 0; t < t1.difference(t0).inMilliseconds; t += 200) {
      final stamp = t0.add(Duration(milliseconds: t));
      final row = <String>['${(t / 1000).toStringAsFixed(1)}'];
      var any = false;
      for (final pid in pids) {
        final list = engine.history[pid.id] ?? const <PidSample>[];
        var i = cursor[pid.id] ?? 0;
        while (i < list.length && !list[i].time.isAfter(stamp)) {
          lastValue[pid.id] = list[i].value;
          lastTime[pid.id] = list[i].time;
          i++;
        }
        cursor[pid.id] = i;
        final pt = lastTime[pid.id];
        final pv = lastValue[pid.id];
        if (pv != null && pt != null && stamp.difference(pt).inMilliseconds.abs() <= skewMs) {
          row.add(pv.toStringAsFixed(pid.digits));
          any = true;
        } else {
          row.add('');
        }
      }
      if (any) sink.writeln(row.map(csvCell).join(','));
    }
    await sink.flush(); await sink.close();
    await Share.shareXFiles([XFile(file.path)], text: 'SSM2 RR-совместимый CSV (${file.path})');
    message = 'RR CSV экспортирован'; changed();
  });

  Future<void> selectPid(String id, bool value) => perform(() async {
    final next = {...engine.enabled};
    if (value) { next.add(id); } else { next.remove(id); }
    await engine.configure(next, engine.extendedConfirmed, engine.romId, engine.endian);
    await saveSettings(); message = 'Выбор сохранен. Нажмите Старт для опроса.';
  });
  Future<void> preset(bool all) => perform(() async {
    await engine.configure(all ? SubaruPidLibrary.all.where((p) => !p.extended).map((p) => p.id).toSet() :
      {...SubaruPidLibrary.defaults}, engine.extendedConfirmed, engine.romId, engine.endian);
    await saveSettings(); message = 'Набор сохранен; опрос на паузе';
  });
  Future<void> configureExtended(bool confirm, String rom, Endian endian) => perform(() async {
    if (confirm && rom.trim().isEmpty) throw ArgumentError('Введите ROM ID из своего def-файла');
    await engine.configure({...engine.enabled}, confirm, rom, endian);
    await saveSettings(); message = 'Настройки ROM сохранены.';
  });
  Future<void> sendDiagnostic(String command) => perform(() async {
    await engine.stop();
    terminal = '';
    final reply = await elm.diagnostic(command);
    terminal = '> ${command.trim().toUpperCase()}\\r\n${visibleText(reply)}';
    message = 'Ручной запрос завершен; опрос остается на паузе';
  });
  Future<void> scan(String start, String count) => perform(() async {
    final address = int.parse(start.trim().replaceFirst(RegExp(r'^0[xX]'), ''), radix: 16);
    final length = int.parse(count);
    if (address >= 0xFF0000 && !engine.extendedConfirmed) throw StateError('Сначала подтвердите ROM');
    await engine.stop();
    scanner = '';
    final result = await elm.readRange(address, length);
    scanner = List<String>.generate(result.bytes.length, (i) =>
      '0x${hexAddress(address + i)}   ${hex2(result.bytes[i])}   ${result.bytes[i]}').join('\n');
    message = 'Прочитано ${result.bytes.length} байт за ${result.elapsedMs} мс. Опрос на паузе.';
  });
  Future<void> toggleLog() => perform(() async {
    if (logger.active) { await logger.stop(); }
    else {
      if (!engine.running || !foreground) throw StateError('Сначала запустите опрос в открытом приложении');
      await logger.start();
      if (!foreground) await logger.stop();
    }
  });
  Future<void> exportCsv() => perform(() async {
    await logger.stop();
    final file = logger.file;
    if (file == null || logger.count == 0) throw StateError('Нет записей CSV');
    await Share.shareXFiles([XFile(file.path)], text: 'SSM2 PID log');
  });
  Future<void> exportTrace() => perform(() async {
    final dir = await getApplicationDocumentsDirectory();
    final file = File('${dir.path}/ssm2_trace_${DateTime.now().millisecondsSinceEpoch}.txt');
    await file.writeAsString('${elm.transport.name}\nROM (user): ${engine.romId}\n'
      'Endian: ${engine.endian == Endian.big ? 'big' : 'little'}\n'
      '${elm.trace.join('\n')}\n', flush: true);
    await Share.shareXFiles([XFile(file.path)], text: 'SSM2 TX/RX diagnostic trace');
  });
  Future<void> shutdown() async {
    if (_disposed) return;
    _disposed = true;
    await _link.cancel();
    try { await elm.disconnect(); } catch (e) { elm.log('SHUTDOWN $e'); }
    await engine.stop(); await _samples.cancel(); await logger.stop();
    try { await elm.dispose(); } catch (e) { elm.log('DISPOSE $e'); }
    engine.dispose(); super.dispose();
  }
}
'''

FILES["lib/maplab_link.dart"] = r'''
import 'package:flutter/material.dart';

class MapLabTab extends StatelessWidget {
  const MapLabTab({super.key});
  @override
  Widget build(BuildContext context) => const Padding(
    padding: EdgeInsets.all(24),
    child: Text('Map Lab появится после ячейки 2b/3.', style: TextStyle(height: 1.7)),
  );
}
'''

FILES["lib/main.dart"] = r'''
import 'dart:async';
import 'dart:io';
import 'dart:math' as math;
import 'package:path_provider/path_provider.dart';
import 'package:share_plus/share_plus.dart';
import 'package:flutter/foundation.dart';
import 'package:flutter/material.dart';
import 'package:flutter/services.dart';
import 'analyzer.dart';
import 'derived.dart';
import 'engine.dart';
import 'maplab_link.dart';
import 'model.dart';
import 'pids.dart';
import 'protocol.dart';
import 'transport_selected.dart';

void main() { WidgetsFlutterBinding.ensureInitialized(); runApp(const SsmApp()); }
const cyan = Color(0xFF22D3EE);
const muted = Color(0xFF9AAAC0);

class SsmApp extends StatelessWidget {
  const SsmApp({super.key});
  @override
  Widget build(BuildContext context) => MaterialApp(
    title: 'SSM2 Telemetry 0.7', debugShowCheckedModeBanner: false,
    theme: ThemeData(colorScheme: ColorScheme.fromSeed(seedColor: cyan, brightness: Brightness.dark),
      scaffoldBackgroundColor: const Color(0xFF080D18), useMaterial3: true),
    home: const HomeShell(),
  );
}

class HomeShell extends StatefulWidget {
  const HomeShell({super.key});
  @override
  State<HomeShell> createState() => _HomeShellState();
}
class _HomeShellState extends State<HomeShell> with WidgetsBindingObserver {
  late final AppModel model;
  late final Listenable changes;
  late final Timer timer;
  int tab = 0;
  @override
  void initState() {
    super.initState();
    model = AppModel(createTransport());
    changes = Listenable.merge([model, model.engine]);
    WidgetsBinding.instance.addObserver(this);
    unawaited(model.restore());
    timer = Timer.periodic(const Duration(milliseconds: 500), (_) { if (mounted) setState(() {}); });
  }
  @override
  void didChangeAppLifecycleState(AppLifecycleState state) {
    if (state == AppLifecycleState.resumed) model.foreground = true;
    if (state == AppLifecycleState.paused) {
      model.foreground = false;
      unawaited(model.engine.stop());
      unawaited(model.logger.stop());
    }
  }
  @override
  void dispose() {
    timer.cancel(); WidgetsBinding.instance.removeObserver(this);
    unawaited(model.shutdown()); super.dispose();
  }
  @override
  Widget build(BuildContext context) => AnimatedBuilder(animation: changes, builder: (context, _) {
    final engine = model.engine;
    final lastTimes = engine.latest.values.map((s) => s.time).toList()..sort();
    final live = model.elm.ready && lastTimes.isNotEmpty && DateTime.now().difference(lastTimes.last).inSeconds < 3;
    final pages = <Widget>[AdapterPage(model), DashboardPage(model), PidPage(model),
      LoggerPage(model), GraphPage(model), AnalyzerPage(model), DiagnosticPage(model), const MapLabTab()];
    return Scaffold(
      appBar: AppBar(title: const Text('SSM2 TELEMETRY 0.7', style: TextStyle(fontSize: 17, letterSpacing: 2)),
        actions: [Icon(Icons.circle, size: 10, color: live ? Colors.greenAccent : muted), const SizedBox(width: 16)]),
      body: SafeArea(child: Column(children: [
        if (model.busy) const LinearProgressIndicator(minHeight: 2),
        Padding(padding: const EdgeInsets.fromLTRB(16, 4, 16, 8), child: Align(alignment: Alignment.centerLeft,
          child: Text(model.message, maxLines: 3, overflow: TextOverflow.ellipsis,
            style: const TextStyle(color: muted, fontSize: 12)))),
        Expanded(child: pages[tab]),
      ])),
      bottomNavigationBar: NavigationBar(selectedIndex: tab, labelBehavior: NavigationDestinationLabelBehavior.onlyShowSelected,
        onDestinationSelected: (i) => setState(() => tab = i), destinations: const [
          NavigationDestination(icon: Icon(Icons.bluetooth), label: 'Адаптер'),
          NavigationDestination(icon: Icon(Icons.speed), label: 'Дашборд'),
          NavigationDestination(icon: Icon(Icons.tune), label: 'PID'),
          NavigationDestination(icon: Icon(Icons.fiber_manual_record_outlined), label: 'CSV'),
          NavigationDestination(icon: Icon(Icons.show_chart), label: 'График'),
          NavigationDestination(icon: Icon(Icons.insights), label: 'Анализ'),
          NavigationDestination(icon: Icon(Icons.terminal), label: 'Диагн.'),
          NavigationDestination(icon: Icon(Icons.table_view), label: 'Map Lab'),
        ]),
    );
  });
}

Widget section(String title, List<Widget> children) => Padding(
  padding: const EdgeInsets.fromLTRB(16, 16, 16, 10), child: Column(crossAxisAlignment: CrossAxisAlignment.start,
    children: [Text(title, style: const TextStyle(fontSize: 15, fontWeight: FontWeight.bold)),
      const SizedBox(height: 12), ...children]));
Widget detail(String key, String value) => Padding(padding: const EdgeInsets.symmetric(vertical: 4),
  child: Row(crossAxisAlignment: CrossAxisAlignment.start, children: [
    Expanded(flex: 2, child: Text(key, style: const TextStyle(color: muted, fontSize: 12))),
    const SizedBox(width: 10), Expanded(flex: 3, child: Text(value, style: const TextStyle(fontSize: 12))),
  ]));
String ageText(int? age) => age == null ? 'нет данных' : '${(age / 1000).toStringAsFixed(1)} с';

class AdapterPage extends StatelessWidget {
  const AdapterPage(this.model, {super.key});
  final AppModel model;
  @override
  Widget build(BuildContext context) => ListView(children: [
    section('Bluetooth SPP', [
      Text(model.elm.transport.name, style: const TextStyle(color: cyan, fontSize: 12)),
      const SizedBox(height: 10),
      const Text('Только CAN 11 bit / 500 kbit. Зажигание включено, автомобиль стоит.'),
      const SizedBox(height: 10),
      Wrap(spacing: 8, children: [
        OutlinedButton.icon(onPressed: model.busy ? null : model.refreshDevices,
          icon: const Icon(Icons.refresh), label: const Text('Сопряженные')),
        TextButton(onPressed: () => model.perform(() async {
          await const MethodChannel('ssm2/system').invokeMethod<void>('bluetoothSettings');
        }), child: const Text('Настройки Android')),
      ]),
      if (model.devices.isEmpty) const Padding(padding: EdgeInsets.all(12), child: Text('Обновите список устройств')),
      for (final device in model.devices) ListTile(
        contentPadding: EdgeInsets.zero, leading: const Icon(Icons.bluetooth, color: cyan),
        title: Text(device.name.isEmpty ? 'Без имени' : device.name), subtitle: Text(device.address),
        trailing: model.selected == device.address ? const Icon(Icons.check, color: cyan) : null,
        onTap: model.busy ? null : () { model.selected = device.address; model.changed(); }),
      Wrap(spacing: 8, children: [
        FilledButton(onPressed: model.busy || model.selected == null ? null : model.connect,
          child: Text(model.elm.ready ? 'Переподключить' : 'Подключить')),
        OutlinedButton(onPressed: model.busy ? null : model.disconnect, child: const Text('Отключить')),
      ]),
    ]),
    section('Состояние', [
      detail('Адаптер', model.elm.identity.isEmpty ? 'не опрошен' : model.elm.identity),
      detail('ATRV', model.elm.voltage.isEmpty ? 'не опрошен' : model.elm.voltage),
      detail('Watchdog', '${model.engine.mutedAddresses} PID изолировано · запросов: ${model.engine.requests}'),
      detail('SPP', model.elm.transport.connected ? 'открыт' : 'закрыт'),
      detail('Синхронизация', model.elm.ready ? 'готово' : 'нет'),
    ]),
  ]);
}

class DashboardPage extends StatelessWidget {
  const DashboardPage(this.model, {super.key});
  final AppModel model;
  @override
  Widget build(BuildContext context) {
    final engine = model.engine;
    final pids = engine.active;
    final fuel = estimateFuel(
      maf: engine.latest['MAF']?.value,
      afr: engine.latest['AFR']?.value,
      speed: engine.latest['SPEED']?.value,
      rpm: engine.latest['RPM']?.value,
      pedal: engine.latest['PEDAL']?.value,
    );
    return Column(children: [
      Padding(padding: const EdgeInsets.symmetric(horizontal: 16), child: Row(children: [
        Expanded(child: Text('SAFE Single Frame / ${pids.length} PID\n${engine.pidReadsPerSecond.toStringAsFixed(1)} PID-обновл./с · ${engine.requestsPerSecond.toStringAsFixed(1)} запросов/с',
          style: const TextStyle(fontSize: 12, color: muted))),
        FilledButton.tonalIcon(onPressed: model.busy || !model.elm.ready ? null : model.togglePolling,
          icon: Icon(engine.running ? Icons.pause : Icons.play_arrow), label: Text(engine.running ? 'Пауза' : 'Старт')),
      ])),
      FuelCard(fuel: fuel, mafStale: engine.stale('MAF')),
      if (pids.isEmpty) const Expanded(child: Center(child: Text('Выберите параметры во вкладке PID')))
      else Expanded(child: GridView.builder(padding: const EdgeInsets.all(12), itemCount: pids.length,
        gridDelegate: const SliverGridDelegateWithMaxCrossAxisExtent(maxCrossAxisExtent: 270, mainAxisExtent: 188,
          mainAxisSpacing: 10, crossAxisSpacing: 10), itemBuilder: (context, i) {
          final p = pids[i];
          final sample = engine.latest[p.id];
          final unsupported = sample?.allOnes ?? false;
          final value = unsupported ? null : sample?.value;
          final outdated = engine.stale(p.id);
          final failed = engine.attempts[p.id]?.error.isNotEmpty ?? false;
          final tone = outdated || failed || unsupported ? muted : cyan;
          return Material(color: const Color(0xFF101A2B), borderRadius: BorderRadius.circular(14),
            child: InkWell(borderRadius: BorderRadius.circular(14), onTap: () => showModalBottomSheet<void>(context: context,
              isScrollControlled: true, builder: (context) => SafeArea(child: SingleChildScrollView(child: section(p.id, [
                Text(p.desc), detail('Адреса', p.addresses.map((a) => '0x${hexAddress(a)}').join(', ')),
                detail('Формула', p.formulaText), detail('Сырые байты', engine.latest[p.id]?.raw.map(hex2).join(' ') ?? 'нет'),
                detail('Возраст', ageText(engine.ageMs(p.id))),
                detail('Частота этого PID', '${engine.frequency(p.id).toStringAsFixed(2)} Hz'),
                detail('Окно чтения', '${engine.latest[p.id]?.readMs ?? 0} ms'),
                detail('Последняя ошибка', engine.attempts[p.id]?.error ?? ''),
              ])))), child: Padding(padding: const EdgeInsets.all(14), child: Column(crossAxisAlignment: CrossAxisAlignment.start,
                children: [
                  Row(children: [Expanded(child: Text(p.id, style: const TextStyle(fontSize: 13, fontWeight: FontWeight.bold))),
                    if (outdated && value != null) const Icon(Icons.schedule, size: 14, color: muted)]),
                  Text('${p.unit} / 0x${hexAddress(p.address)}', style: const TextStyle(fontSize: 10, color: muted)),
                  const Spacer(),
                  SizedBox(height: 46, width: double.infinity, child: FittedBox(fit: BoxFit.scaleDown,
                    alignment: Alignment.centerLeft, child: Text(value?.toStringAsFixed(p.digits) ?? '--',
                      style: TextStyle(color: tone, fontSize: 38, fontWeight: FontWeight.w700)))),
                  const SizedBox(height: 10),
                  LinearProgressIndicator(value: value == null ? 0 :
                    ((value - p.minValue) / (p.maxValue - p.minValue)).clamp(0.0, 1.0).toDouble(), color: tone, minHeight: 3),
                  const SizedBox(height: 8),
                  Text(unsupported
                      ? '0xFF: адрес не поддерживается'
                      : failed
                          ? 'ошибка / ${ageText(engine.ageMs(p.id))}'
                          : ageText(engine.ageMs(p.id)),
                    style: TextStyle(
                      color: unsupported ? const Color(0xFFD4B57F) : muted, fontSize: 10)),
                ]))));
        })),
    ]);
  }
}

class FuelCard extends StatelessWidget {
  const FuelCard({super.key, required this.fuel, required this.mafStale});
  final FuelEstimate? fuel;
  final bool mafStale;

  @override
  Widget build(BuildContext context) {
    final f = fuel;
    return Container(
      margin: const EdgeInsets.fromLTRB(12, 10, 12, 0),
      padding: const EdgeInsets.all(14),
      decoration: BoxDecoration(
        color: const Color(0xFF101A2B),
        borderRadius: BorderRadius.circular(14),
        border: Border.all(color: const Color(0xFF24435A)),
      ),
      child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
        Row(children: [
          const Expanded(child: Text('МГНОВЕННЫЙ РАСХОД',
            style: TextStyle(fontSize: 11, letterSpacing: 1.2, color: muted))),
          Text(f == null ? 'нет MAF' : 'расчет по MAF',
            style: const TextStyle(fontSize: 10, color: muted)),
        ]),
        const SizedBox(height: 10),
        if (f == null)
          const Text('Включите PID MAF и запустите опрос.',
            style: TextStyle(fontSize: 12, color: muted))
        else ...[
          Row(crossAxisAlignment: CrossAxisAlignment.end, children: [
            Text(f.litresPerHour.toStringAsFixed(2),
              style: TextStyle(fontSize: 38, fontWeight: FontWeight.w700,
                color: mafStale ? muted : cyan, height: 1)),
            const Padding(padding: EdgeInsets.only(left: 8, bottom: 4),
              child: Text('л/ч', style: TextStyle(fontSize: 13, color: muted))),
            const Spacer(),
            Text(f.litresPer100km == null
                ? 'на месте'
                : '${f.litresPer100km!.toStringAsFixed(1)} л/100км',
              style: TextStyle(fontSize: 16, fontWeight: FontWeight.w600,
                color: mafStale ? muted : Colors.white)),
          ]),
          const SizedBox(height: 10),
          Text('AFR ${f.afrUsed.toStringAsFixed(1)}'
              '${f.afrMeasured ? ' (из ECU)' : ' (стехиометрия)'}'
              ' · плотность $kPetrolDensityGramsPerLitre г/л',
            style: const TextStyle(fontSize: 10, color: muted, height: 1.5)),
        ],
      ]),
    );
  }
}

class AnalyzerPage extends StatefulWidget {
  const AnalyzerPage(this.model, {super.key});
  final AppModel model;
  @override
  State<AnalyzerPage> createState() => _AnalyzerPageState();
}

class _AnalyzerPageState extends State<AnalyzerPage> {
  LogAnalysis? report;
  bool auto = true;
  int lastSignature = -1;
  DateTime lastAutoRun = DateTime.fromMillisecondsSinceEpoch(0);
  String errors = '0';
  int attemptsTotal = 0;

  @override
  void initState() {
    super.initState();
    widget.model.engine.addListener(_onEngine);
  }
  @override
  void didUpdateWidget(covariant AnalyzerPage oldWidget) {
    super.didUpdateWidget(oldWidget);
    if (!identical(oldWidget.model.engine, widget.model.engine)) {
      oldWidget.model.engine.removeListener(_onEngine);
      widget.model.engine.addListener(_onEngine);
    }
  }
  @override
  void dispose() {
    widget.model.engine.removeListener(_onEngine);
    super.dispose();
  }
  void _onEngine() {
    if (!auto || !mounted || widget.model.engine.history.isEmpty) return;
    final signature = widget.model.engine.goodCount ~/ 10;
    if (signature == lastSignature) return;
    lastSignature = signature;
    if (DateTime.now().difference(lastAutoRun).inSeconds < 3) return;
    lastAutoRun = DateTime.now();
    _schedule();
  }
  void _schedule() {
    final withErrors = widget.model.engine.attempts.values
        .where((s) => s.error.isNotEmpty).map((s) => s.pid.id).toList()..sort();
    attemptsTotal = widget.model.engine.attempts.length;
    errors = withErrors.isEmpty ? '0' : withErrors.length <= 8
        ? withErrors.join(', ')
        : '${withErrors.take(8).join(', ')} +${withErrors.length - 8}';
    setState(() {
      report = analyzeLog(widget.model.engine.history, attempts: widget.model.engine.attempts);
    });
  }
  Future<void> _exportReport() async {
    final engine = widget.model.engine;
    final result = report ?? analyzeLog(engine.history, attempts: engine.attempts);
    final md = StringBuffer('# SSM2 0.7 · Анализ журнала\n\n');
    md.writeln('Значений: ${result.sampleCount} · длительность: '
        '${result.spanSeconds.toStringAsFixed(1)} с · каналов: ${result.channels}');
    for (final f in result.findings) {
      md.writeln('\n## [${_levelName(f.level)}] ${f.title}\n');
      md.writeln('${f.detail}\n');
      md.writeln('```\n${f.evidence}\n```\n');
      md.writeln('> ${f.advice}');
    }
    final dir = await getApplicationDocumentsDirectory();
    final file = File('${dir.path}/ssm2_report_${DateTime.now().millisecondsSinceEpoch}.md');
    await file.writeAsString(md.toString(), flush: true);
    await Share.shareXFiles([XFile(file.path)], text: 'SSM2 анализ журнала (md)');
  }

  Color _levelColor(FindingLevel level) => switch (level) {
    FindingLevel.critical => const Color(0xFFE08A7A),
    FindingLevel.warning => const Color(0xFFD4B57F),
    FindingLevel.info => muted,
  };
  String _levelName(FindingLevel level) => switch (level) {
    FindingLevel.critical => 'КРИТИЧНО',
    FindingLevel.warning => 'ВНИМАНИЕ',
    FindingLevel.info => 'ИНФО',
  };

  @override
  Widget build(BuildContext context) {
    final engine = widget.model.engine;
    final result = report;
    return ListView(children: [
      section('Анализ журнала', [
        const Text('Прозрачные правила с фиксированными порогами.', style: TextStyle(color: muted, height: 1.6)),
        const SizedBox(height: 12),
        Wrap(spacing: 8, runSpacing: 8, children: [
          FilledButton.icon(
            onPressed: engine.history.isEmpty ? null : _schedule,
            icon: const Icon(Icons.insights, size: 18),
            label: const Text('Проанализировать')),
          OutlinedButton.icon(onPressed: () => setState(() => auto = !auto),
            icon: Icon(auto ? Icons.check_box : Icons.check_box_outline_blank, size: 18),
            label: const Text('Авто')),
          OutlinedButton.icon(onPressed: engine.history.isEmpty || widget.model.busy ? null : _exportReport,
            icon: const Icon(Icons.share, size: 16), label: const Text('Отчёт.md')),
        ]),
      ]),
      if (result != null) ...[
        section('Итог', [
          detail('Значений в памяти', '${result.sampleCount}'),
          detail('Длительность', '${result.spanSeconds.toStringAsFixed(1)} с'),
          detail('Каналов с данными', '${result.channels}'),
        ]),
        for (final f in result.findings)
          Padding(
            padding: const EdgeInsets.fromLTRB(16, 0, 16, 12),
            child: Container(
              padding: const EdgeInsets.all(14),
              decoration: BoxDecoration(
                color: const Color(0xFF101A2B),
                borderRadius: BorderRadius.circular(12),
                border: Border(left: BorderSide(color: _levelColor(f.level), width: 3)),
              ),
              child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
                Text(_levelName(f.level), style: TextStyle(fontSize: 9, letterSpacing: 1.2, color: _levelColor(f.level))),
                const SizedBox(height: 6),
                Text(f.title, style: const TextStyle(fontSize: 14, fontWeight: FontWeight.w600)),
                const SizedBox(height: 8),
                Text(f.detail, style: const TextStyle(fontSize: 12, height: 1.6)),
              ]),
            ),
          )
      ],
    ]);
  }
}

class PidPage extends StatefulWidget {
  const PidPage(this.model, {super.key});
  final AppModel model;
  @override
  State<PidPage> createState() => _PidPageState();
}
class _PidPageState extends State<PidPage> {
  String search = '';
  @override
  Widget build(BuildContext context) {
    final m = widget.model;
    return ListView(children: [section('Библиотека / 28 PID', [
      const Text('Сначала 8 базовых. Больше параметров — ниже частота каждого PID.', style: TextStyle(color: muted)),
      Wrap(spacing: 8, children: [
        TextButton(onPressed: m.busy ? null : () => m.preset(false), child: const Text('8 базовых')),
        TextButton(onPressed: m.busy ? null : () => m.preset(true), child: const Text('Все 20 обычных')),
        TextButton(onPressed: m.busy ? null : () => _settings(context), child: const Text('ROM / Float32')),
      ]),
      TextField(decoration: const InputDecoration(labelText: 'Поиск PID или адреса', prefixIcon: Icon(Icons.search)),
        onChanged: (value) => setState(() => search = value.toUpperCase())),
      const SizedBox(height: 8),
      for (final p in SubaruPidLibrary.all.where((p) => '${p.id} ${p.desc} 0x${hexAddress(p.address)}'.toUpperCase().contains(search.trim())))
        CheckboxListTile(contentPadding: EdgeInsets.zero, controlAffinity: ListTileControlAffinity.leading,
          title: Text('${p.id}${p.extended ? ' *' : ''} / ${p.unit}', style: const TextStyle(fontSize: 13)),
          subtitle: Text('0x${hexAddress(p.address)} / ${p.bytesCount} B / P${p.priority}\n${p.formulaText}',
            style: const TextStyle(fontSize: 10, color: muted)), value: m.engine.enabled.contains(p.id),
          onChanged: m.busy || (p.extended && !m.engine.extendedConfirmed) ? null :
            (value) => m.selectPid(p.id, value ?? false)),
    ])]);
  }
  Future<void> _settings(BuildContext context) async {
    final m = widget.model;
    final controller = TextEditingController(text: m.engine.romId);
    var confirm = m.engine.extendedConfirmed;
    var little = m.engine.endian == Endian.little;
    final accepted = await showDialog<bool>(context: context, builder: (context) => StatefulBuilder(builder: (context, change) =>
      AlertDialog(title: const Text('Extended / ROM'), content: SingleChildScrollView(child: Column(mainAxisSize: MainAxisSize.min,
        children: [
          const Text('Введите ROM ID из своего def-файла для включения extended PID.'),
          TextField(controller: controller, decoration: const InputDecoration(labelText: 'ROM ID')),
          CheckboxListTile(title: const Text('Адреса сверены с моим ROM'), value: confirm,
            onChanged: (value) => change(() => confirm = value ?? false)),
          SwitchListTile(title: const Text('Float32 little-endian'), value: little,
            onChanged: (value) => change(() => little = value)),
        ])), actions: [
          TextButton(onPressed: () => Navigator.pop(context, false), child: const Text('Отмена')),
          FilledButton(onPressed: () => Navigator.pop(context, true), child: const Text('Сохранить')),
        ])));
    final rom = controller.text;
    controller.dispose();
    if (accepted == true) await m.configureExtended(confirm, rom, little ? Endian.little : Endian.big);
  }
}

class LoggerPage extends StatelessWidget {
  const LoggerPage(this.model, {super.key});
  final AppModel model;
  @override
  Widget build(BuildContext context) => ListView(children: [section('Потоковый CSV', [
    Text('${model.logger.count}', style: const TextStyle(fontSize: 54, fontWeight: FontWeight.bold, color: cyan)),
    Text(model.logger.active ? 'Идет запись' : 'Запись остановлена'),
    const SizedBox(height: 18),
    Wrap(spacing: 8, children: [
      FilledButton.icon(onPressed: model.busy ? null : model.toggleLog,
        icon: Icon(model.logger.active ? Icons.stop : Icons.fiber_manual_record),
        label: Text(model.logger.active ? 'Стоп' : 'Записать')),
      OutlinedButton.icon(onPressed: model.busy || model.logger.count == 0 ? null : model.exportCsv,
        icon: const Icon(Icons.share), label: const Text('Экспорт CSV')),
      OutlinedButton.icon(onPressed: model.busy || model.engine.history.isEmpty ? null : model.exportRrCsv,
        icon: const Icon(Icons.table_chart), label: const Text('RR CSV')),
    ]),
  ])]);
}

class GraphPage extends StatelessWidget {
  const GraphPage(this.model, {super.key});
  final AppModel model;
  @override
  Widget build(BuildContext context) {
    final pid = SubaruPidLibrary.byId(model.chartId);
    final samples = List<PidSample>.of(model.engine.history[pid.id] ?? []);
    return ListView(children: [section('График', [
      DropdownButton<String>(value: model.chartId, isExpanded: true,
        items: SubaruPidLibrary.all.map((p) => DropdownMenuItem(value: p.id, child: Text('${p.id} / ${p.unit}'))).toList(),
        onChanged: (value) { if (value != null) { model.chartId = value; model.changed(); } }),
      const SizedBox(height: 18),
      SizedBox(height: 270, child: samples.length < 2 ? const Center(child: Text('Нужны хотя бы два значения')) :
        CustomPaint(painter: TelemetryPainter(samples, pid.digits), size: Size.infinite)),
    ])]);
  }
}
class TelemetryPainter extends CustomPainter {
  TelemetryPainter(this.samples, this.digits);
  final List<PidSample> samples;
  final int digits;
  void label(Canvas canvas, String text, Offset point) {
    final painter = TextPainter(text: TextSpan(text: text, style: const TextStyle(color: muted, fontSize: 10)),
      textDirection: TextDirection.ltr)..layout();
    painter.paint(canvas, point);
  }
  @override
  void paint(Canvas canvas, Size size) {
    final values = samples.map((s) => s.value!).toList();
    final minimum = values.reduce(math.min), maximum = values.reduce(math.max);
    final pad = math.max((maximum - minimum) * 0.1, 0.5);
    final lo = minimum - pad, hi = maximum + pad;
    final first = samples.first.time.millisecondsSinceEpoch;
    final span = math.max(1, samples.last.time.millisecondsSinceEpoch - first);
    final width = math.max(1.0, size.width - 65), height = size.height - 38;
    final grid = Paint()..color = const Color(0xFF233045)..strokeWidth = 1;
    for (var i = 0; i <= 4; i++) {
      final y = 8 + height * i / 4;
      canvas.drawLine(Offset(52, y), Offset(size.width - 8, y), grid);
      label(canvas, (hi - (hi - lo) * i / 4).toStringAsFixed(digits > 1 ? 1 : digits), Offset(0, y - 5));
    }
    final path = Path();
    for (var i = 0; i < samples.length; i++) {
      final sample = samples[i];
      final x = 52 + width * (sample.time.millisecondsSinceEpoch - first) / span;
      final y = 8 + height * (hi - sample.value!) / (hi - lo);
      canvas.drawCircle(Offset(x, y), 2, Paint()..color = cyan);
      if (i == 0 || sample.time.difference(samples[i - 1].time).inMilliseconds > 3000) { path.moveTo(x, y); }
      else { path.lineTo(x, y); }
    }
    canvas.drawPath(path, Paint()..color = cyan..strokeWidth = 2..style = PaintingStyle.stroke);
    label(canvas, '0 s', Offset(52, height + 20));
    label(canvas, '${(span / 1000).toStringAsFixed(1)} s', Offset(size.width - 55, height + 20));
  }
  @override
  bool shouldRepaint(covariant TelemetryPainter oldDelegate) => true;
}

class DiagnosticPage extends StatefulWidget {
  const DiagnosticPage(this.model, {super.key});
  final AppModel model;
  @override
  State<DiagnosticPage> createState() => _DiagnosticPageState();
}
class _DiagnosticPageState extends State<DiagnosticPage> {
  final command = TextEditingController(text: 'ATI');
  final address = TextEditingController(text: '000008');
  final count = TextEditingController(text: '1');
  @override
  void dispose() { command.dispose(); address.dispose(); count.dispose(); super.dispose(); }
  @override
  Widget build(BuildContext context) {
    final m = widget.model;
    return ListView(children: [
      section('Диагностика', [
        TextField(controller: command, decoration: const InputDecoration(labelText: 'ATI / ATRV / A8 00 00 00 08')),
        const SizedBox(height: 8),
        FilledButton(onPressed: m.busy || !m.elm.ready ? null : () => m.sendDiagnostic(command.text), child: const Text('Отправить')),
        SelectableText(m.terminal.isEmpty ? 'Команд еще нет' : m.terminal, style: const TextStyle(fontFamily: 'monospace', fontSize: 12)),
      ]),
      section('Сканер адресов A8', [
        Row(children: [Expanded(child: TextField(controller: address, decoration: const InputDecoration(labelText: 'Адрес HEX'))),
          const SizedBox(width: 12), SizedBox(width: 90, child: TextField(controller: count,
            keyboardType: TextInputType.number, decoration: const InputDecoration(labelText: '1..32 байт')))]),
        const SizedBox(height: 8),
        OutlinedButton(onPressed: m.busy || !m.elm.ready ? null : () => m.scan(address.text, count.text), child: const Text('Прочитать')),
        SelectableText(m.scanner.isEmpty ? 'Нет дампа' : m.scanner, style: const TextStyle(fontFamily: 'monospace', fontSize: 12)),
      ]),
    ]);
  }
}
'''

FILES["test/protocol_test.dart"] = r'''
import 'dart:async';
import 'dart:typed_data';
import 'package:flutter_test/flutter_test.dart';
import 'package:subaru_ssm2/bt_transport.dart';
import 'package:subaru_ssm2/pids.dart';
import 'package:subaru_ssm2/protocol.dart';

class FakeTransport implements BtTransport {
  bool _connected = false;
  final received = StreamController<Uint8List>.broadcast(sync: true);
  final states = StreamController<bool>.broadcast(sync: true);
  final writes = <String>[];
  @override
  String get name => 'Synthetic test transport';
  @override
  bool get connected => _connected;
  @override
  Stream<Uint8List> get data => received.stream;
  @override
  Stream<bool> get status => states.stream;
  @override
  Future<List<BtDevice>> paired() async => [const BtDevice('Test', '00:00:00:00:00:00')];
  @override
  Future<void> connect(String address) async { _connected = true; states.add(true); }
  @override
  Future<void> disconnect() async { _connected = false; states.add(false); }
  void emit(String text) => received.add(Uint8List.fromList(text.codeUnits));
  @override
  Future<void> write(String ascii) async {
    if (!_connected) throw StateError('Not connected');
    writes.add(ascii);
    final cmd = ascii.trim();
    final reply = cmd == 'ATI' || cmd == 'ATZ' ? 'ELM327 TEST\r>' : cmd == 'ATRV' ? '13.5V\r>' :
      cmd.startsWith('A8') ? 'E8 ${cmd.endsWith('0E') ? '0D' : 'E6'}\r>' : 'OK\r>';
    emit(reply.substring(0, 1));
    await Future<void>.delayed(const Duration(milliseconds: 1));
    emit(reply.substring(1));
  }
  @override
  Future<void> dispose() async { await received.close(); await states.close(); }
}

void main() {
  const command = 'A8 00 00 00 08';
  group('Strict ATH0 / CAF1 parser', () {
    test('one byte, CR, LF and split-independent assembled text', () {
      expect(parseAddressReply('E8 6D\r\n>', command), 109);
      expect(parseAddressReply('E86D\r\r>', command), 109);
    });
    test('only single-address A8', () {
      expect(readAddressCommand(8), command);
      expect(readAddressCommand(0xFF2538), 'A8 00 FF 25 38');
    });
    test('only read diagnostics', () {
      expect(allowedDiagnostic('ATI'), isTrue);
      expect(allowedDiagnostic(command), isTrue);
      expect(allowedDiagnostic('ATZ'), isFalse);
    });
  });

  group('All 28 PID', () {
    test('identity, addresses and sizes', () {
      final all = SubaruPidLibrary.all;
      expect(all.length, 28);
      expect(SubaruPidLibrary.byId('RPM').addresses, [0xE, 0xF]);
      expect(SubaruPidLibrary.byId('O2_F').decode([0, 200]), closeTo(1.0, 0.000001));
    });
  });
}
'''

FILES["tool/prepare_bt.py"] = r'''
import hashlib
import json
from pathlib import Path
import re
import shutil
import sys
import tarfile
import urllib.request

APP = Path(__file__).resolve().parents[1]
PACKAGES = {
    "bluetooth_classic": ("0.0.4", "c92e10fb8f8114a19603c48b02ddfda448b41d6a9cf79eb192f82c6ed3bc0df5", "com.matteogassend.bluetooth_classic", "classic"),
    "flutter_bluetooth_serial": ("0.4.0", "85ae82c4099b2b1facdc54e75e1bcfa88dc7f719e55dc886bb0b648cb16636b1", "io.github.edufolly.flutterbluetoothserial", "serial"),
}

def prepare(name):
    if name not in PACKAGES: raise ValueError(name)
    version, checksum, namespace, short = PACKAGES[name]
    cache = Path('/content/ssm2_downloads')
    cache.mkdir(exist_ok=True)
    archive = cache / f'{name}-{version}.tar.gz'
    if not archive.exists():
        temp = archive.with_suffix('.part')
        with urllib.request.urlopen(f'https://pub.dev/api/archives/{name}-{version}.tar.gz', timeout=120) as response, temp.open('wb') as out:
            shutil.copyfileobj(response, out)
        temp.replace(archive)
    vendor = APP / 'vendor' / name
    shutil.rmtree(vendor, ignore_errors=True)
    vendor.mkdir(parents=True)
    with tarfile.open(archive, 'r:gz') as tar:
        members = tar.getmembers()
        tar.extractall(vendor, members=members)

    build = f"""plugins {{
    id 'com.android.library'
    id 'org.jetbrains.kotlin.android'
}}
android {{
    namespace '{namespace}'
    compileSdk 36
    defaultConfig {{ minSdk 24 }}
    compileOptions {{
        sourceCompatibility JavaVersion.VERSION_17
        targetCompatibility JavaVersion.VERSION_17
    }}
    kotlinOptions {{ jvmTarget = "17" }}
}}
dependencies {{
    implementation 'androidx.core:core:1.13.1'
    implementation 'androidx.appcompat:appcompat:1.7.0'
}}
"""
    (vendor / 'android/build.gradle').write_text(build, encoding='utf-8')
    shutil.copyfile(APP / f'transport_templates/{short}.dart.txt', APP / 'lib/transport_selected.dart')
    pubspec = (APP / 'pubspec.template.yaml').read_text(encoding='utf-8')
    pubspec = pubspec.replace('__BLUETOOTH_DEPENDENCY__', f'  {name}:\n    path: vendor/{name}')
    (APP / 'pubspec.yaml').write_text(pubspec, encoding='utf-8')

if __name__ == '__main__':
    prepare(sys.argv[1])
'''

FILES["android/settings.gradle.kts"] = r'''
pluginManagement {
    val flutterSdkPath = run {
        val properties = java.util.Properties()
        file("local.properties").inputStream().use { properties.load(it) }
        requireNotNull(properties.getProperty("flutter.sdk")) { "flutter.sdk missing" }
    }
    includeBuild("$flutterSdkPath/packages/flutter_tools/gradle")
    repositories { google(); mavenCentral(); gradlePluginPortal() }
}
plugins {
    id("dev.flutter.flutter-plugin-loader") version "1.0.0"
    id("com.android.application") version "8.11.1" apply false
    id("com.android.library") version "8.11.1" apply false
    id("org.jetbrains.kotlin.android") version "2.2.20" apply false
}
include(":app")
'''

FILES["android/build.gradle.kts"] = r'''
allprojects {
    repositories { google(); mavenCentral() }
}
val newBuildDir = rootProject.layout.buildDirectory.dir("../../build").get()
rootProject.layout.buildDirectory.value(newBuildDir)
subprojects {
    project.layout.buildDirectory.value(newBuildDir.dir(project.name))
}
subprojects { project.evaluationDependsOn(":app") }
tasks.register<Delete>("clean") { delete(rootProject.layout.buildDirectory) }
'''

FILES["android/app/build.gradle.kts"] = r'''
plugins {
    id("com.android.application")
    id("org.jetbrains.kotlin.android")
    id("dev.flutter.flutter-gradle-plugin")
}
android {
    namespace = "com.subaru.ssm2_fixed"
    compileSdk = 36
    ndkVersion = "27.0.12077973"
    compileOptions {
        sourceCompatibility = JavaVersion.VERSION_17
        targetCompatibility = JavaVersion.VERSION_17
    }
    kotlinOptions { jvmTarget = "17" }
    defaultConfig {
        applicationId = "com.subaru.ssm2_fixed"
        minSdk = 24
        targetSdk = 35
        versionCode = flutter.versionCode
        versionName = flutter.versionName
    }
    buildTypes {
        release {
            signingConfig = signingConfigs.getByName("debug")
            isMinifyEnabled = false
            isShrinkResources = false
        }
    }
}
flutter { source = "../.." }
'''

FILES["android/gradle.properties"] = r'''
org.gradle.jvmargs=-Xmx4g -XX:MaxMetaspaceSize=1g -XX:+HeapDumpOnOutOfMemoryError
org.gradle.workers.max=2
org.gradle.caching=true
android.useAndroidX=true
'''

FILES["android/gradle/wrapper/gradle-wrapper.properties"] = r'''
distributionBase=GRADLE_USER_HOME
distributionPath=wrapper/dists
zipStoreBase=GRADLE_USER_HOME
zipStorePath=wrapper/dists
distributionUrl=https\://services.gradle.org/distributions/gradle-8.14.3-bin.zip
networkTimeout=120000
validateDistributionUrl=true
'''

FILES["android/app/src/main/kotlin/com/subaru/ssm2_fixed/MainActivity.kt"] = r'''
package com.subaru.ssm2_fixed

import android.content.Intent
import android.os.Build
import android.provider.Settings
import io.flutter.embedding.android.FlutterActivity
import io.flutter.embedding.engine.FlutterEngine
import io.flutter.plugin.common.MethodChannel

class MainActivity : FlutterActivity() {
    override fun configureFlutterEngine(flutterEngine: FlutterEngine) {
        super.configureFlutterEngine(flutterEngine)
        MethodChannel(flutterEngine.dartExecutor.binaryMessenger, "ssm2/system")
            .setMethodCallHandler { call, result ->
                when (call.method) {
                    "sdkInt" -> result.success(Build.VERSION.SDK_INT)
                    "bluetoothSettings" -> {
                        startActivity(Intent(Settings.ACTION_BLUETOOTH_SETTINGS))
                        result.success(null)
                    }
                    else -> result.notImplemented()
                }
            }
    }
}
'''

FILES["android/app/src/main/AndroidManifest.xml"] = r'''
<manifest xmlns:android="http://schemas.android.com/apk/res/android">
    <uses-permission android:name="android.permission.BLUETOOTH" android:maxSdkVersion="30"/>
    <uses-permission android:name="android.permission.BLUETOOTH_ADMIN" android:maxSdkVersion="30"/>
    <uses-permission android:name="android.permission.ACCESS_FINE_LOCATION" android:maxSdkVersion="30"/>
    <uses-permission android:name="android.permission.ACCESS_COARSE_LOCATION" android:maxSdkVersion="30"/>
    <uses-permission android:name="android.permission.BLUETOOTH_SCAN" android:usesPermissionFlags="neverForLocation"/>
    <uses-permission android:name="android.permission.BLUETOOTH_CONNECT"/>
    <uses-feature android:name="android.hardware.bluetooth" android:required="true"/>
    <application android:label="SSM2 Fixed" android:name="${applicationName}" android:icon="@mipmap/ic_launcher">
        <activity android:name=".MainActivity" android:exported="true" android:launchMode="singleTop"
            android:theme="@style/LaunchTheme" android:hardwareAccelerated="true"
            android:configChanges="orientation|keyboardHidden|keyboard|screenSize|smallestScreenSize|locale|layoutDirection|fontScale|screenLayout|density|uiMode"
            android:windowSoftInputMode="adjustResize">
            <meta-data android:name="io.flutter.embedding.android.NormalTheme" android:resource="@style/NormalTheme"/>
            <intent-filter>
                <action android:name="android.intent.action.MAIN"/>
                <category android:name="android.intent.category.LAUNCHER"/>
            </intent-filter>
        </activity>
        <meta-data android:name="flutterEmbedding" android:value="2"/>
    </application>
    <queries>
        <intent><action android:name="android.intent.action.PROCESS_TEXT"/><data android:mimeType="text/plain"/></intent>
    </queries>
</manifest>
'''

FILES["pubspec.template.yaml"] = r'''
name: subaru_ssm2
description: "Read-only Subaru SSM2 CAN telemetry: 28 PID, Single Frame SAFE Mode, Map Lab (0.7)"
publish_to: 'none'
version: 0.7.0+7
environment:
  sdk: '>=3.3.0 <4.0.0'
dependencies:
  flutter:
    sdk: flutter
__BLUETOOTH_DEPENDENCY__
  permission_handler: 11.3.1
  path_provider: 2.1.5
  share_plus: 10.1.4
dev_dependencies:
  flutter_test:
    sdk: flutter
flutter:
  uses-material-design: true
'''

FILES["lib/samples.dart"] = r'''
import 'pids.dart';

class PidSample {
  PidSample(this.pid, this.time, this.value, this.raw, this.readMs, this.error);
  final SubaruPidDef pid;
  final DateTime time;
  final double? value;
  final List<int> raw;
  final int readMs;
  final String error;
  bool get good => value != null && error.isEmpty;
  bool get allOnes => raw.isNotEmpty && raw.every((b) => b == 0xFF);
}
'''

FILES["lib/identity.dart"] = r'''
String normalizeCalId(String value) =>
    value.toUpperCase().replaceAll(RegExp(r'[^0-9A-Z]'), '');

class CalProfile {
  const CalProfile(this.calId, this.ecuId, this.label,
      this.addresses, this.parentDefs);
  final String calId, ecuId, label;
  final Map<String, String> addresses;
  final String parentDefs;
}

const List<CalProfile> calProfiles = <CalProfile>[
  CalProfile('A2TB100B', '5204584007',
      'Legacy GT (BP/BL) EJ20X JDM 2008 · TD04HL-19T · Dual AVCS · 5EAT',
      <String, String>{
        'IAM': '0xFF2538',
        'LOAD_4B': '0xFF6C9C',
        'BOOST_ERR': '0xFF6450',
        'BOOST_TGT': '0xFF6454',
        'FBKC': '0xFF7D4C',
        'FKL': '0xFF7DD0',
        'BOOST': '0xFF6AE0',
        'CL_TARGET': '0xFF73B4',
      },
      'ECU defs: A2TB100B -> A2TB100K -> 32BITBASE (TD-D/SubaruDefs)'),
];

CalProfile? matchCalProfile(String? candidate) {
  final norm = normalizeCalId(candidate ?? '');
  if (norm.isEmpty) return null;
  for (final profile in calProfiles) {
    if (profile.calId == norm) return profile;
  }
  return null;
}
'''

FILES["lib/derived.dart"] = r'''
const double kPetrolDensityGramsPerLitre = 745.0;
const double kStoichAfr = 14.7;

class FuelEstimate {
  const FuelEstimate({
    required this.fuelGramsPerSecond,
    required this.litresPerHour,
    required this.litresPer100km,
    required this.afrUsed,
    required this.afrMeasured,
    required this.speedKph,
    required this.possibleCutoff,
  });

  final double fuelGramsPerSecond;
  final double litresPerHour;
  final double? litresPer100km;
  final double afrUsed;
  final bool afrMeasured;
  final double? speedKph;
  final bool possibleCutoff;

  String get formula => 'MAF / AFR * 3600 / $kPetrolDensityGramsPerLitre';
}

FuelEstimate? estimateFuel({
  double? maf,
  double? afr,
  double? speed,
  double? rpm,
  double? pedal,
}) {
  if (maf == null || !maf.isFinite || maf <= 0) return null;
  final measured = afr != null && afr.isFinite && afr >= 8 && afr <= 25;
  final ratio = measured ? afr : kStoichAfr;
  final grams = maf / ratio;
  if (!grams.isFinite || grams < 0) return null;
  final litresPerHour = grams * 3600 / kPetrolDensityGramsPerLitre;
  double? per100;
  if (speed != null && speed.isFinite && speed >= 5) {
    per100 = litresPerHour / speed * 100;
  }
  final cutoff = pedal != null && pedal <= 1 &&
      rpm != null && rpm > 1500 &&
      speed != null && speed > 5;
  return FuelEstimate(
    fuelGramsPerSecond: grams,
    litresPerHour: litresPerHour,
    litresPer100km: per100,
    afrUsed: ratio,
    afrMeasured: measured,
    speedKph: speed,
    possibleCutoff: cutoff,
  );
}
'''

FILES["lib/analyzer.dart"] = r'''
import 'samples.dart';

enum FindingLevel { info, warning, critical }

class Finding {
  const Finding(this.level, this.title, this.detail, this.evidence, this.advice);
  final FindingLevel level;
  final String title, detail, evidence, advice;
}

class LogAnalysis {
  const LogAnalysis(this.findings, this.sampleCount, this.spanSeconds, this.channels);
  final List<Finding> findings;
  final int sampleCount, channels;
  final double spanSeconds;
}

LogAnalysis analyzeLog(
  Map<String, List<PidSample>> history, {
  Map<String, PidSample> attempts = const <String, PidSample>{},
}) {
  final findings = <Finding>[];
  return LogAnalysis(findings, 0, 0, 0);
}
'''

FILES["analysis_options.yaml"] = r'''
analyzer:
  exclude:
    - vendor/**
    - transport_templates/**
  language:
    strict-casts: true
    strict-raw-types: true
'''

FILES["transport_templates/serial.dart.txt"] = r'''
import 'dart:async';
import 'dart:typed_data';
import 'package:flutter_bluetooth_serial/flutter_bluetooth_serial.dart';
import 'bt_transport.dart';

class SelectedTransport implements BtTransport {
  BluetoothConnection? _connection;
  StreamSubscription<Uint8List>? _rx;
  final _data = StreamController<Uint8List>.broadcast();
  final _status = StreamController<bool>.broadcast();
  @override
  String get name => 'B / flutter_bluetooth_serial 0.4.0 (SAFE)';
  @override
  bool get connected => _connection?.isConnected ?? false;
  @override
  Stream<Uint8List> get data => _data.stream;
  @override
  Stream<bool> get status => _status.stream;
  @override
  Future<List<BtDevice>> paired() async {
    await requestBluetoothPermissions();
    final devices = await FlutterBluetoothSerial.instance.getBondedDevices();
    return devices.map((d) => BtDevice(d.name ?? '', d.address)).toList();
  }
  @override
  Future<void> connect(String address) async {
    await requestBluetoothPermissions();
    await disconnect();
    final connection = await BluetoothConnection.toAddress(address);
    _connection = connection;
    _rx = connection.input!.listen((b) => _data.add(Uint8List.fromList(b)),
      onDone: () => _status.add(false),
      onError: (Object e) { _data.addError(e); _status.add(false); });
    _status.add(true);
  }
  @override
  Future<void> write(String ascii) async {
    final connection = _connection;
    if (connection == null || !connection.isConnected) throw StateError('SPP disconnected');
    connection.output.add(Uint8List.fromList(ascii.codeUnits));
    await connection.output.allSent;
  }
  @override
  Future<void> disconnect() async {
    final connection = _connection;
    _connection = null;
    await _rx?.cancel(); _rx = null;
    if (connection != null) await connection.close();
  }
  @override
  Future<void> dispose() async {
    try { await disconnect(); } finally { await _data.close(); await _status.close(); }
  }
}
BtTransport createTransport() => SelectedTransport();
'''

def generate_project():
    for relative, content in FILES.items():
        if relative.endswith(".py"):
            ast.parse(content, filename=relative)
            compile(content, relative, "exec")
    if APP.exists():
        backup = APP.with_name(APP.name + f"_backup_{time.time_ns()}")
        APP.rename(backup)
        print("Предыдущий проект сохранен:", backup)
    APP.mkdir(parents=True)
    run([FLUTTER / "bin/flutter", "create", "--no-pub", "--platforms=android",
         "--org", "com.subaru", "--project-name", "subaru_ssm2", APP])
    for relative in ["lib", "test", "android/app/src/main/kotlin"]:
        shutil.rmtree(APP / relative, ignore_errors=True)
    for relative in ["android/build.gradle", "android/settings.gradle", "android/app/build.gradle"]:
        (APP / relative).unlink(missing_ok=True)
    for relative, content in FILES.items():
        write(relative, content)
    write("android/local.properties", f"sdk.dir={CFG['sdk']}\nflutter.sdk={CFG['flutter']}\n")
    write("build_config.json", json.dumps({**CFG, "bt_package": BT_PACKAGE, "revision": "0.7"}, indent=2))
    write("pid_catalog.json", json.dumps(PIDS, ensure_ascii=False, indent=2))
    run([sys.executable, APP / "tool/prepare_bt.py", BT_PACKAGE])
    run([FLUTTER / "bin/flutter", "pub", "get"])
    print(f"\n[УСПЕХ] Создано {len(FILES)} файлов. Проект перегенерирован!")

generate_project()

Creating project ....
Wrote 35 files.

All done!
You can find general documentation for Flutter at: https://docs.flutter.dev/
Detailed API documentation is available at: https://api.flutter.dev/
If you prefer video documentation, consider: https://www.youtube.com/c/flutterdev

In order to run your application, type:

  $ cd .
  $ flutter run

Your application code is in ./lib/main.dart.


/content/subaru_ssm2_fixed/tool/prepare_bt.py:32: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(vendor, members=members)

Resolving dependencies...
+ async 2.13.1
+ bluetooth_classic 0.0.4 from path vendor/bluetooth_classic
+ boolean_selector 2.1.2
+ characters 1.4.0 (1.4.1 available)
+ clock 1.1.2 (1.1.3 available)
+ collection 1.19.1
+ cross_file 0.3.5+2 (0.3.5+5 available)
+ crypto 3.0.7
+ fake_async 1.3.3
+ ffi 2.2.0
+ file 7.0.1
+ fixnum 1.1.1
+ flutter 0

In [3]:
# @title 2b/3 | Map Lab 0.7 FIXED - heatmap + 3D rotate + table zoom { display-mode: "form" }

import json
import re
import urllib.request
from pathlib import Path

CONFIG = Path("/content/ssm2_fixed_env.json")
if not CONFIG.exists():
    raise RuntimeError("Сначала выполните ячейку 1")
CFG = json.loads(CONFIG.read_text(encoding="utf-8"))
APP = Path(CFG["app"])
if not (APP / "pubspec.yaml").exists():
    raise RuntimeError("Проект не найден. Сначала выполните ячейку 2/3.")
FILES = {}

FILES["lib/maps_lab/maps_lab_core.dart"] = r"""
/// Map Lab core — чистый Dart без Flutter.
library;

import 'dart:convert';
import 'dart:typed_data';

class SafeExpr {
  static final RegExp _ok = RegExp(r'^[0-9xX\.\+\-\*/\(\)\s]+$');

  static double Function(double) compile(String? src) {
    if (src == null || src.trim().isEmpty) return (v) => v;
    final s = src.replaceAll('X', 'x');
    if (!_ok.hasMatch(s)) return (v) => v;
    try {
      final parser = _ExprParser(s);
      final node = parser.parse();
      return (x) => node(x);
    } catch (_) {
      return (v) => v;
    }
  }
}

class _ExprParser {
  _ExprParser(this.s);
  final String s;
  int i = 0;

  double Function(double) parse() {
    final n = _expr();
    _ws();
    if (i != s.length) throw FormatException('лишние символы: ${s.substring(i)}');
    return n;
  }

  void _ws() {
    while (i < s.length && s[i] == ' ') {
      i++;
    }
  }

  bool _eat(String ch) {
    _ws();
    if (i < s.length && s[i] == ch) {
      i++;
      return true;
    }
    return false;
  }

  double Function(double) _expr() {
    var node = _term();
    while (true) {
      if (_eat('+')) {
        final r = _term();
        final l = node;
        node = (x) => l(x) + r(x);
      } else if (_eat('-')) {
        final r = _term();
        final l = node;
        node = (x) => l(x) - r(x);
      } else {
        return node;
      }
    }
  }

  double Function(double) _term() {
    var node = _factor();
    while (true) {
      if (_eat('*')) {
        final r = _factor();
        final l = node;
        node = (x) => l(x) * r(x);
      } else if (_eat('/')) {
        final r = _factor();
        final l = node;
        node = (x) => r(x) == 0 ? double.nan : l(x) / r(x);
      } else {
        return node;
      }
    }
  }

  static final RegExp _num = RegExp(r'(\d+\.?\d*|\.\d+)');

  double Function(double) _factor() {
    _ws();
    if (_eat('-')) {
      final n = _factor();
      return (x) => -n(x);
    }
    if (_eat('+')) return _factor();
    if (_eat('(')) {
      final n = _expr();
      if (!_eat(')')) throw const FormatException('нет закрывающей скобки');
      return n;
    }
    _ws();
    if (i < s.length && s[i] == 'x') {
      i++;
      return (x) => x;
    }
    final m = _num.matchAsPrefix(s, i);
    if (m == null) throw FormatException('ожидалось число @ $i');
    i = m.end;
    final v = double.parse(m.group(0)!);
    return (x) => v;
  }
}

enum Stype { u8, i8, u16, i16, f32 }

class Scaling {
  Scaling({
    required this.name,
    required this.units,
    required this.type,
    required this.bigEndian,
    required this.to,
    required this.fr,
    required this.format,
  });

  factory Scaling.fromAttrs(String name, Map<String, String> a) {
    final st = switch (a['storagetype'] ?? 'uint8') {
      'int8' => Stype.i8,
      'uint16' => Stype.u16,
      'int16' => Stype.i16,
      'float' => Stype.f32,
      _ => Stype.u8,
    };
    return Scaling(
      name: name,
      units: a['units'] ?? '',
      type: st,
      bigEndian: (a['endian'] ?? 'big') == 'big',
      to: SafeExpr.compile(a['toexpr']),
      fr: SafeExpr.compile(a['frexpr']),
      format: a['format'] ?? '%.2f',
    );
  }

  final String name;
  final String units;
  final Stype type;
  final bool bigEndian;
  final double Function(double) to;
  final double Function(double) fr;
  final String format;

  int get bytes => switch (type) {
        Stype.f32 => 4,
        Stype.u16 || Stype.i16 => 2,
        _ => 1,
      };
}

class AxisDef {
  String name = '';
  String? scalingName;
  int? address;
  int? elements;
  Scaling? scaling;
}

class TableDef {
  TableDef(this.name);
  final String name;
  String? type;
  String? category;
  String? scalingName;
  int? dataAddress;
  Scaling? dataScaling;
  final List<AxisDef> axes = [];
}

class DefMeta {
  String xmlid = '';
  String internalIdAddress = '2000';
  String internalIdString = '';
  String ecuid = '';
  String checksumModule = '';
}

class _RawTable {
  _RawTable(this.attrs, this.children);
  final Map<String, String> attrs;
  final List<Map<String, String>> children;
}

class _RawDef {
  final meta = DefMeta();
  final includes = <String>[];
  final scalings = <String, Map<String, String>>{};
  final tables = <_RawTable>[];
}

class DefSet {
  DefSet._();
  final meta = DefMeta();
  final chain = <String>[];
  final scalings = <String, Scaling>{};
  final tables = <String, TableDef>{};

  static const _generic = {'x', 'y', 'z', ''};

  static final _attrRe = RegExp(r'([\w:-]+)="([^"]*)"');
  static final _tableTok = RegExp(r'<table\s[^>]*?>|</table>');
  static final _scalingRe = RegExp(r'<scaling\s+([^>]*?)/>');
  static final _romidRe = RegExp(r'<romid>([\s\S]*?)</romid>');
  static final _innerTag = RegExp(r'<(\w+)>([^<]*)</\1>');
  static final _includeRe = RegExp(r'<include>\s*([^<]+?)\s*</include>');

  static Map<String, String> _attrs(String tag) =>
      {for (final m in _attrRe.allMatches(tag)) m.group(1)!: m.group(2)!};

  static _RawDef _parse(String raw) {
    final def = _RawDef();
    final romid = _romidRe.firstMatch(raw);
    if (romid != null) {
      for (final m in _innerTag.allMatches(romid.group(1)!)) {
        final v = m.group(2)!.trim();
        if (v.isEmpty) continue;
        switch (m.group(1)) {
          case 'xmlid':
            def.meta.xmlid = v;
          case 'internalidaddress':
            def.meta.internalIdAddress = v;
          case 'internalidstring':
            def.meta.internalIdString = v;
          case 'ecuid':
            def.meta.ecuid = v;
          case 'checksummodule':
            def.meta.checksumModule = v;
        }
      }
    }
    for (final m in _includeRe.allMatches(raw)) {
      def.includes.add(m.group(1)!);
    }
    for (final m in _scalingRe.allMatches(raw)) {
      final a = _attrs(m.group(0)!);
      if (a['storagetype'] == 'bloblist') continue;
      final n = a['name'];
      if (n != null && n.isNotEmpty) def.scalings[n] = a;
    }

    var depth = 0;
    for (final m in _tableTok.allMatches(raw)) {
      final tok = m.group(0)!;
      if (tok == '</table>') {
        if (depth > 0) depth--;
        continue;
      }
      final selfClose = tok.endsWith('/>');
      final a = _attrs(tok);
      if (depth == 0) {
        def.tables.add(_RawTable(a, []));
        if (!selfClose) depth = 1;
      } else {
        def.tables.last.children.add(a);
      }
    }
    return def;
  }

  static DefSet build(Map<String, String> xmlById, String rootId) {
    final raw = <String, _RawDef>{};
    for (final e in xmlById.entries) {
      raw[e.key] = _parse(e.value);
    }

    final order = <String>[];
    final seen = <String>{};
    void walk(String id) {
      if (!seen.add(id)) return;
      for (final inc in raw[id]?.includes ?? const <String>[]) {
        if (raw.containsKey(inc)) walk(inc);
      }
      order.add(id);
    }

    walk(rootId);

    final set = DefSet._();
    set.chain.addAll(order);
    if (raw.containsKey(rootId)) {
      final m = raw[rootId]!.meta;
      set.meta
        ..xmlid = m.xmlid
        ..internalIdAddress = m.internalIdAddress
        ..internalIdString = m.internalIdString
        ..ecuid = m.ecuid
        ..checksumModule = m.checksumModule;
    }

    for (final id in order) {
      final d = raw[id]!;
      d.scalings.forEach((n, a) {
        set.scalings[n] = Scaling.fromAttrs(n, a);
      });
      for (final rt in d.tables) {
        final name = (rt.attrs['name'] ?? '').trim();
        if (name.isEmpty) continue;
        final t = set.tables.putIfAbsent(name, () => TableDef(name));
        for (final k in ['type', 'category', 'level', 'scaling']) {
          final v = rt.attrs[k];
          if (v != null && v.isNotEmpty) {
            switch (k) {
              case 'type':
                t.type = v;
              case 'category':
                t.category = v;
              case 'scaling':
                t.scalingName = v;
            }
          }
        }
        final addr = rt.attrs['address'];
        if (addr != null && addr.isNotEmpty) t.dataAddress = int.parse(addr, radix: 16);
        for (var i = 0; i < rt.children.length; i++) {
          while (t.axes.length <= i) {
            t.axes.add(AxisDef());
          }
          final a = t.axes[i];
          final ch = rt.children[i];
          final cn = (ch['name'] ?? '').trim();
          if (!_generic.contains(cn.toLowerCase())) a.name = cn;
          if ((ch['scaling'] ?? '').isNotEmpty) a.scalingName = ch['scaling'];
          if ((ch['elements'] ?? '').isNotEmpty) a.elements = int.parse(ch['elements']!);
          if ((ch['address'] ?? '').isNotEmpty) a.address = int.parse(ch['address']!, radix: 16);
        }
      }
    }

    for (final t in set.tables.values) {
      t.dataScaling = set.scalings[t.scalingName];
      for (final a in t.axes) {
        a.scaling = set.scalings[a.scalingName];
      }
    }
    return set;
  }

  bool isReadable3D(TableDef t) =>
      t.dataAddress != null &&
      t.dataScaling != null &&
      t.axes.length >= 2 &&
      t.axes[0].address != null &&
      t.axes[0].elements != null &&
      t.axes[0].scaling != null &&
      t.axes[1].address != null &&
      t.axes[1].elements != null &&
      t.axes[1].scaling != null;
}

class AxisVals {
  AxisVals(this.name, this.values, this.guess);
  final String name;
  final List<double> values;
  final String guess;
}

class MapGrid {
  MapGrid({
    required this.name,
    required this.kind,
    required this.units,
    required this.addr,
    required this.x,
    required this.y,
    required this.data,
  });
  final String name;
  final String kind;
  final String units;
  final int addr;
  final AxisVals x;
  final AxisVals y;
  final List<List<double>> data;

  int get rows => data.length;
  int get cols => data.isEmpty ? 0 : data[0].length;
  double get vmin => data.expand((r) => r).reduce((a, b) => a < b ? a : b);
  double get vmax => data.expand((r) => r).reduce((a, b) => a > b ? a : b);
}

class RomParser {
  RomParser(this.rom);
  final Uint8List rom;

  String readRomId(int addr) {
    if (addr < 0 || addr + 16 > rom.length) return '';
    final bytes = <int>[];
    for (var i = addr; i < addr + 16; i++) {
      final b = rom[i];
      if (b == 0) break;
      bytes.add(b);
    }
    return ascii.decode(bytes, allowInvalid: true).trim();
  }

  List<double> _read(Scaling sc, int addr, int count) {
    final size = sc.bytes;
    if (addr < 0 || addr + count * size > rom.length) {
      throw FormatException(
          'диапазон 0x${addr.toRadixString(16)} + $count×$size вне ROM (${rom.length} байт)');
    }
    final bd = ByteData.sublistView(rom, addr, addr + count * size);
    final en = sc.bigEndian ? Endian.big : Endian.little;
    final out = List<double>.filled(count, 0);
    for (var i = 0; i < count; i++) {
      final v = switch (sc.type) {
        Stype.u8 => bd.getUint8(i).toDouble(),
        Stype.i8 => bd.getInt8(i).toDouble(),
        Stype.u16 => bd.getUint16(i * 2, en).toDouble(),
        Stype.i16 => bd.getInt16(i * 2, en).toDouble(),
        Stype.f32 => bd.getFloat32(i * 4, en),
      };
      out[i] = sc.to(v);
    }
    return out;
  }

  static String guessAxis(List<double> v) {
    if (v.isEmpty) return '?';
    var lo = v.first, hi = v.first;
    for (final x in v) {
      if (x < lo) lo = x;
      if (x > hi) hi = x;
    }
    if (hi > 800) return 'rpm';
    if (hi > 50 && hi <= 600) return 'нм/у.е.';
    if (hi <= 5.5 && lo >= -0.5) return 'г/об·бар';
    if (hi <= 14 && lo >= 0) return 'вольты/%';
    return '?';
  }

  static String classify(String name, String units) {
    final n = name.toLowerCase();
    if (n.contains('wastegate duty')) return 'wgdc';
    if (n.contains('target boost')) return 'boost';
    if (n.contains('requested torque')) return 'torque';
    if (n.contains('knock correction')) return 'knockadv';
    if (n.contains('fueling') || n.contains('fuel')) return 'fuel';
    if (n.contains('timing')) return 'timing';
    return 'other';
  }

  double _round4(double v) => (v * 10000).roundToDouble() / 10000;

  MapGrid? extract(TableDef t) {
    if (t.dataAddress == null || t.dataScaling == null) return null;
    if (t.axes.length < 2) return null;
    final ax = t.axes[0], ay = t.axes[1];
    if (ax.address == null || ax.elements == null || ax.scaling == null) return null;
    if (ay.address == null || ay.elements == null || ay.scaling == null) return null;

    final xv = _read(ax.scaling!, ax.address!, ax.elements!);
    final yv = _read(ay.scaling!, ay.address!, ay.elements!);
    final cols = ax.elements!, rows = ay.elements!;
    final flat = _read(t.dataScaling!, t.dataAddress!, rows * cols);
    final grid = <List<double>>[
      for (var r = 0; r < rows; r++) [for (var c = 0; c < cols; c++) _round4(flat[r * cols + c])],
    ];
    return MapGrid(
      name: t.name,
      kind: classify(t.name, t.dataScaling!.units),
      units: t.dataScaling!.units,
      addr: t.dataAddress!,
      x: AxisVals(ax.name.isEmpty ? 'X' : ax.name, [for (final v in xv) _round4(v)], guessAxis(xv)),
      y: AxisVals(ay.name.isEmpty ? 'Y' : ay.name, [for (final v in yv) _round4(v)], guessAxis(yv)),
      data: grid,
    );
  }

  static const keyTables = [
    'Base Timing Primary Cruise',
    'Base Timing Primary Non-Cruise',
    'Primary Open Loop Fueling',
    'Target Boost_',
    'Initial Wastegate Duty_',
    'Max Wastegate Duty_',
    'Knock Correction Advance Max Non-Cruise',
    'Requested Torque A (Accelerator Pedal) SI-DRIVE Sport',
  ];

  Map<String, MapGrid> extractKeys(DefSet defs) {
    final out = <String, MapGrid>{};
    for (final n in keyTables) {
      final t = defs.tables[n];
      if (t == null) continue;
      try {
        final g = extract(t);
        if (g != null) out[n] = g;
      } catch (_) {}
    }
    return out;
  }
}
"""

# maps_lab_log — без Color (чтобы library оставался чистым). Heatmap — в page/3d.
FILES["lib/maps_lab/maps_lab_log.dart"] = r"""
/// Map Lab: лог и правила.
library;

import 'dart:convert';
import 'dart:math' as math;

import 'maps_lab_core.dart';

class LogData {
  final cols = <String, List<double?>>{};
  int get rows => cols.isEmpty ? 0 : cols.values.first.length;

  bool has(String k) => cols.containsKey(k);
  List<double?>? operator [](String k) => cols[k];

  static const aliases = <String, List<String>>{
    'time': ['time', 'timestamp', 'time s', 'elapsed'],
    'rpm': ['engine speed', 'rpm', 'engine speed rpm'],
    'load': [
      'engine load g/rev',
      'load_4b',
      'engine load 4-byte',
      'calculated load',
      'engine load',
      'load'
    ],
    'fbkc': ['feedback knock correction', 'fbkc'],
    'flkc': ['fine learning knock correction', 'fkl', 'fine learning knock advance', 'flkc'],
    'iam': ['iam', 'ignition advance multiplier'],
    'timing': ['total ignition timing', 'ignition timing', 'timing'],
    'afr': ['afr', 'a/f sensor #1', 'a/f sensor 1', 'air/fuel ratio', 'estimated afr', 'lambda'],
    'boost': ['manifold relative pressure', 'boost', 'boost_rel'],
    'tgt': ['target boost', 'boost_tgt'],
    'berr': ['boost error', 'boost_err'],
    'wgdc': ['primary wastegate duty', 'wastegate duty', 'wgdc', 'boost control solenoid duty'],
    'tq': ['requested torque', 'cl_target', 'demand torque'],
    'thr': ['throttle opening angle', 'throttle plate', 'throttle', 'throttle position'],
    'iat': ['intake air temperature', 'iat'],
    'ect': ['coolant temperature', 'ect', 'engine coolant temperature'],
    'loop': ['cl/ol', 'fueling status', 'closed loop', 'loop'],
  };

  static String _canon(String s) {
    var c = s.toLowerCase().trim();
    final p = c.indexOf('(');
    if (p > 0) c = c.substring(0, p);
    final b = c.indexOf('[');
    if (b > 0) c = c.substring(0, b);
    return c.replaceAll('*', '').replaceAll(RegExp(r'\s+'), ' ').trim();
  }

  static LogData parse(String text) {
    final head = text.substring(0, math.min(text.length, 4096));
    final sep = ';'.allMatches(head).length > ','.allMatches(head).length ? ';' : ',';
    final decComma = sep == ';' && ','.allMatches(head).length > '.'.allMatches(head).length;

    final lines = const LineSplitter().convert(text).where((l) => l.trim().isNotEmpty).toList();
    if (lines.isEmpty) return LogData();
    final header = lines.first.split(sep).map((h) => h.trim().replaceAll('"', '')).toList();

    final raw = List<List<double?>>.generate(header.length, (_) => []);
    final canonHead = header.map(_canon).toList();

    for (var li = 1; li < lines.length; li++) {
      final parts = lines[li].split(sep);
      for (var i = 0; i < header.length; i++) {
        if (i >= parts.length) {
          raw[i].add(null);
          continue;
        }
        var tok = parts[i].trim().replaceAll('"', '');
        if (decComma) tok = tok.replaceAll(',', '.');
        raw[i].add(double.tryParse(tok));
      }
    }

    final log = LogData();
    final usedNames = <String>{};
    for (final entry in aliases.entries) {
      final canon = entry.key;
      final variants = entry.value.map(_canon).toSet();
      for (var i = 0; i < header.length; i++) {
        if (usedNames.contains(header[i])) continue;
        if (variants.contains(canonHead[i])) {
          log.cols[canon] = raw[i];
          usedNames.add(header[i]);
          break;
        }
      }
    }

    final afr = log.cols['afr'];
    if (afr != null) {
      final vals = afr.whereType<double>().toList()..sort();
      if (vals.isNotEmpty && vals[vals.length ~/ 2] <= 2.2) {
        log.cols['afr'] = [for (final v in afr) v == null ? null : v * 14.7];
      }
    }
    final boost = log.cols['boost'], tgt = log.cols['tgt'];
    if (boost != null && tgt != null && log.cols['berr'] == null) {
      log.cols['berr'] = [
        for (var i = 0; i < boost.length; i++)
          (boost[i] != null && tgt[i] != null) ? boost[i]! - tgt[i]! : null,
      ];
    }
    return log;
  }
}

class LogHealth {
  final notes = <String>[];
  final missing = <String, List<String>>{};
  bool blockReady(String b) => (missing[b] ?? const []).isEmpty;
}

class LogAudit {
  static const _needFor = {
    'timing': ['rpm', 'load', 'fbkc', 'flkc', 'iat'],
    'fuel': ['rpm', 'load', 'afr', 'boost'],
    'wgdc': ['rpm', 'tq', 'berr'],
  };

  static const _hint = {
    'wgdc': 'Primary Wastegate Duty',
    'tq': 'Requested Torque',
    'thr': 'Throttle Opening Angle',
    'afr': 'A/F Sensor #1',
    'loop': 'CL/OL Fueling Status',
  };

  static LogHealth check(LogData log) {
    final h = LogHealth();
    h.notes.add('строк: ${log.rows}');

    double? maxOf(String k) {
      final c = log[k];
      if (c == null) return null;
      double? m;
      for (final v in c) {
        if (v != null && (m == null || v > m)) m = v;
      }
      return m;
    }

    double? minOf(String k) {
      final c = log[k];
      if (c == null) return null;
      double? m;
      for (final v in c) {
        if (v != null && (m == null || v < m)) m = v;
      }
      return m;
    }

    final mr = maxOf('rpm');
    if (mr != null) h.notes.add('RPM max: ${mr.toStringAsFixed(0)}');
    final iamMin = minOf('iam');
    if (iamMin != null) {
      h.notes.add('IAM min: ${iamMin.toStringAsFixed(2)}'
          '${iamMin < 0.99 ? ' — ЛОГ НЕ ГОДИТСЯ: сначала доучить ЭБУ' : ' — ок'}');
    }
    final mb = maxOf('boost');
    if (mb != null) h.notes.add('буст max: ${mb.toStringAsFixed(2)} бар');

    final t = log['time'];
    if (t != null) {
      final dts = <double>[];
      for (var i = 1; i < t.length; i++) {
        if (t[i] != null && t[i - 1] != null) dts.add(t[i]! - t[i - 1]!);
      }
      dts.sort();
      if (dts.isNotEmpty) {
        final med = dts[dts.length ~/ 2];
        final gaps = dts.where((d) => d > 0.3).length;
        h.notes.add('частота: медиана ${(1 / med).toStringAsFixed(1)} Гц · провалов >300 мс: $gaps');
        if (med > 0.25) h.notes.add('!! реже 4 Гц — сократите набор PID до 10–12');
      }
    }

    _needFor.forEach((block, req) {
      final miss = req.where((r) => !log.has(r)).toList();
      h.missing[block] = miss;
    });
    final allMiss = {for (final v in h.missing.values) ...v};
    for (final m in allMiss) {
      final hint = _hint[m];
      if (hint != null) h.notes.add('добавить в логгер PID: $hint');
    }
    return h;
  }
}

class CellNote {
  int n = 0;
  double? fbkc, flkc, iat, iam, afr, target, berr;
  String why = '';
}

class MapResult {
  MapResult(int rows, int cols)
      : delta = List.generate(rows, (_) => List.filled(cols, 0.0)),
        info = List.generate(rows, (_) => List<CellNote?>.filled(cols, null));
  final List<List<double>> delta;
  final List<List<CellNote?>> info;

  int get rows => delta.length;
  int get cols => delta.isNotEmpty ? delta[0].length : 0;

  int get dec {
    var c = 0;
    for (final r in delta) {
      for (final v in r) {
        if (v < 0) c++;
      }
    }
    return c;
  }

  int get inc {
    var c = 0;
    for (final r in delta) {
      for (final v in r) {
        if (v > 0) c++;
      }
    }
    return c;
  }
}

class AnalyzerConfig {
  const AnalyzerConfig({
    this.minN = 6,
    this.fbkcEvent = -1.5,
    this.flkcEvent = -1.5,
    this.timingStep = 0.5,
    this.timingMaxCut = -3.0,
    this.allowTimingAdd = false,
    this.afrErr = 0.40,
    this.afrMaxCut = -0.8,
    this.boostErr = 0.04,
    this.wgdcMaxDelta = 6.0,
    this.wotLoad = 2.2,
    this.targetBoostGain = 0.0,
  });
  final int minN;
  final double fbkcEvent, flkcEvent, timingStep, timingMaxCut;
  final bool allowTimingAdd;
  final double afrErr, afrMaxCut, boostErr, wgdcMaxDelta, wotLoad, targetBoostGain;
}

class Analyzer {
  Analyzer(this.cfg);
  final AnalyzerConfig cfg;

  double _roundStep(double v, double s) => (v / s).roundToDouble() * s;

  int _binIdx(List<double> axis, double v) {
    if (axis.length <= 1) return 0;
    var lo = 0, hi = axis.length - 1;
    while (lo < hi - 1) {
      final mid = (lo + hi) >> 1;
      if (v >= axis[mid]) {
        lo = mid;
      } else {
        hi = mid;
      }
    }
    final mid = (axis[lo] + axis[hi]) / 2;
    return v < mid ? lo : hi;
  }

  MapResult? analyzeMap(MapGrid g, LogData log) {
    final kind = g.kind;
    if (!{'timing', 'knockadv', 'fuel', 'wgdc', 'boost'}.contains(kind)) return null;

    final xIsRpm = g.x.guess == 'rpm' || (g.y.guess != 'rpm' && g.x.values.last > 800);
    final rpmAxis = xIsRpm ? g.x.values : g.y.values;
    final otherAxis = xIsRpm ? g.y.values : g.x.values;

    final need = switch (kind) {
      'timing' || 'knockadv' => ['rpm', 'load', 'fbkc', 'flkc'],
      'fuel' => ['rpm', 'load', 'afr', 'fbkc', 'flkc'],
      _ => ['rpm', 'tq', 'berr'],
    };
    if (need.any((k) => !log.has(k))) return null;

    final rpmCol = log['rpm']!;
    final otherCol = switch (kind) {
      'timing' || 'knockadv' || 'fuel' => log['load']!,
      _ => log['tq']!,
    };

    final rows = g.rows, cols = g.cols;
    final cnt = List<int>.filled(rows * cols, 0);
    final fbkcMin = List<double>.filled(rows * cols, 0);
    final flkcSum = List<double>.filled(rows * cols, 0);
    final flkcLateSum = List<double>.filled(rows * cols, 0);
    final flkcLateCnt = List<int>.filled(rows * cols, 0);
    final halfRows = (log.rows / 2).round();
    final iatMax = List<double>.filled(rows * cols, -999);
    final iamMin = List<double>.filled(rows * cols, 999);
    final afrSum = List<double>.filled(rows * cols, 0);
    final berrSum = List<double>.filled(rows * cols, 0);

    final fbkc = log['fbkc'], flkc = log['flkc'], iat = log['iat'];
    final iam = log['iam'], afr = log['afr'], berr = log['berr'];

    for (var i = 0; i < log.rows; i++) {
      final rv = rpmCol[i], ov = otherCol[i];
      if (rv == null || ov == null) continue;
      final ri = _binIdx(xIsRpm ? otherAxis : rpmAxis, xIsRpm ? ov : rv);
      final ci = _binIdx(xIsRpm ? rpmAxis : otherAxis, xIsRpm ? rv : ov);
      final idx = ri * cols + ci;
      cnt[idx]++;
      final f = fbkc?[i];
      if (f != null && (cnt[idx] == 1 || f < fbkcMin[idx])) fbkcMin[idx] = f;
      final fl = flkc?[i];
      if (fl != null) {
        flkcSum[idx] += fl;
        if (i >= halfRows) {
          flkcLateSum[idx] += fl;
          flkcLateCnt[idx]++;
        }
      }
      final it = iat?[i];
      if (it != null && it > iatMax[idx]) iatMax[idx] = it;
      final im = iam?[i];
      if (im != null && im < iamMin[idx]) iamMin[idx] = im;
      final a = afr?[i];
      if (a != null) afrSum[idx] += a;
      final be = berr?[i];
      if (be != null) berrSum[idx] += be;
    }

    final res = MapResult(rows, cols);

    void fanTiming(int ri, int ci, double d, AnalyzerConfig cfg) {
      const fan = <List<num>>[
        [-1, 0, 0.5],
        [1, 0, 0.5],
        [0, -1, 0.5],
        [0, 1, 0.5],
        [-1, -1, 0.25],
        [-1, 1, 0.25],
        [1, -1, 0.25],
        [1, 1, 0.25],
      ];
      for (final e in fan) {
        final rr = ri + e[0].toInt(), cc = ci + e[1].toInt();
        if (rr < 0 || rr >= rows || cc < 0 || cc >= cols) continue;
        if (res.delta[rr][cc] != 0) continue;
        final d2 = _roundStep(d * e[2].toDouble(), cfg.timingStep);
        if (d2 == 0) continue;
        res.delta[rr][cc] = d2;
        res.info[rr][cc] = CellNote()..why = 'веер от соседнего кластера: ${d2.toStringAsFixed(1)}°';
      }
    }

    for (var ri = 0; ri < rows; ri++) {
      for (var ci = 0; ci < cols; ci++) {
        final idx = ri * cols + ci;
        final n = cnt[idx];
        if (n < cfg.minN) continue;
        final note = CellNote()
          ..n = n
          ..fbkc = fbkcMin[idx]
          ..flkc = flkcSum[idx] / n
          ..iat = iatMax[idx] < -900 ? null : iatMax[idx]
          ..iam = iamMin[idx] > 900 ? null : iamMin[idx]
          ..afr = afrSum[idx] > 0 ? afrSum[idx] / n : null
          ..berr = berrSum[idx] != 0 ? berrSum[idx] / n : null;

        if (kind == 'timing' || kind == 'knockadv') {
          if (note.iam != null && note.iam! < 0.99) {
            res.info[ri][ci] = note..why = 'IAM=${note.iam!.toStringAsFixed(2)} — сначала доучить';
            continue;
          }
          final lateCount = flkcLateCnt[idx];
          final lateFlkc = lateCount > 0 ? flkcLateSum[idx] / lateCount : note.flkc!;
          final knock = math.min(note.fbkc!, math.min(note.flkc!, lateFlkc) * 1.4);
          if (knock <= cfg.fbkcEvent) {
            final d = math.max(cfg.timingMaxCut,
                math.min(-cfg.timingStep, _roundStep(knock * 0.6, cfg.timingStep)));
            res.delta[ri][ci] = d;
            fanTiming(ri, ci, d, cfg);
            res.info[ri][ci] = note
              ..why = 'детон-кластер · FBKC ${note.fbkc!.toStringAsFixed(1)}°, '
                  'FLKC ${note.flkc!.toStringAsFixed(1)}° (поздняя ${lateFlkc.toStringAsFixed(1)}°) · '
                  'снять ${d.abs().toStringAsFixed(1)}° здесь, соседи сглажены веером';
          } else if (res.delta[ri][ci] == 0 &&
              cfg.allowTimingAdd &&
              n >= 18 &&
              (note.iat == null || note.iat! <= 45)) {
            res.delta[ri][ci] = cfg.timingStep;
            res.info[ri][ci] = note..why = 'чисто, IAM=1.0 — опционально +0.5°';
          }
        } else if (kind == 'fuel') {
          final loadAxisValue = xIsRpm ? g.y.values[ri] : g.x.values[ci];
          if (loadAxisValue < cfg.wotLoad || note.afr == null) continue;
          var target = g.data[ri][ci];
          if (target < 9.0) target *= 14.7;
          note.target = target;
          final err = note.afr! - target;
          final knock = math.min(note.fbkc!, note.flkc!);
          if (err > cfg.afrErr) {
            final d = math.max(cfg.afrMaxCut, _roundStep(-err, 0.1));
            res.delta[ri][ci] = d;
            res.info[ri][ci] = note
              ..why = 'факт ${note.afr!.toStringAsFixed(2)} против цели ${target.toStringAsFixed(2)} '
                  '— обогатить на ${d.abs().toStringAsFixed(1)}; если не помогает — MAF/давление';
          } else if (knock <= cfg.fbkcEvent) {
            res.delta[ri][ci] = -0.3;
            res.info[ri][ci] = note..why = 'детон при цели ~совпадает — запас −0.3 AFR';
          }
        } else {
          final e = note.berr;
          if (e == null) continue;
          if (kind == 'wgdc') {
            if (e.abs() > cfg.boostErr) {
              final d = _roundStep(-e * 45, 1)
                  .clamp(-cfg.wgdcMaxDelta, cfg.wgdcMaxDelta - 1)
                  .toDouble();
              if (d != 0) {
                res.delta[ri][ci] = d;
                res.info[ri][ci] = note
                  ..why = '${e > 0 ? 'овербуст' : 'недобор'} ${e.toStringAsFixed(2)} бар '
                      '→ WGDC ${d > 0 ? '+' : ''}${d.toStringAsFixed(0)}%';
              }
            }
          } else {
            if (e > cfg.boostErr) {
              res.info[ri][ci] = note
                ..why = 'овербуст ${e.toStringAsFixed(2)} бар — чинить через WGDC/TD, не таргет';
            }
            if (cfg.targetBoostGain > 0) {
              res.delta[ri][ci] = cfg.targetBoostGain;
              res.info[ri][ci] = note
                ..why = 'политика +${cfg.targetBoostGain} бар (после фикса WGDC и чистого детон-лога)';
            }
          }
        }
      }
    }
    return res;
  }

  Map<String, MapResult> run(Map<String, MapGrid> maps, LogData log) {
    final out = <String, MapResult>{};
    for (final e in maps.entries) {
      final r = analyzeMap(e.value, log);
      if (r != null) out[e.key] = r;
    }
    return out;
  }
}
"""

FILES["lib/maps_lab/map3d_view.dart"] = r"""
/// 3D-визуализатор: тепловая карта + жесты, которые НЕ отдаёт ListView.
library;

import 'dart:math' as math;

import 'package:flutter/gestures.dart';
import 'package:flutter/material.dart';

import 'maps_lab_core.dart';
import 'maps_lab_log.dart';
import 'maps_lab_page.dart' show getHeatmapColor;

/// Pan-распознаватель, который всегда побеждает родительский ListView.
class _EagerPanGestureRecognizer extends PanGestureRecognizer {
  @override
  void rejectGesture(int pointer) {
    acceptGesture(pointer);
  }
}

class Map3DView extends StatefulWidget {
  const Map3DView({
    super.key,
    required this.grid,
    this.result,
    this.selRi,
    this.selCi,
    this.onCell,
  });

  final MapGrid grid;
  final MapResult? result;
  final int? selRi;
  final int? selCi;
  final void Function(int ri, int ci)? onCell;

  @override
  State<Map3DView> createState() => Map3DViewState();
}

class Map3DViewState extends State<Map3DView> with SingleTickerProviderStateMixin {
  double _yaw = 0.95;
  double _pitch = 0.55;
  double _zoom = 1.0;
  bool _auto = true;
  late final AnimationController _ticker;
  final _painter = _SurfacePainter();

  @override
  void initState() {
    super.initState();
    _ticker = AnimationController(vsync: this, duration: const Duration(seconds: 1))
      ..addListener(() {
        if (_auto) {
          _yaw += 0.003;
          setState(() {});
        }
      })
      ..repeat();
  }

  @override
  void dispose() {
    _ticker.dispose();
    super.dispose();
  }

  void _onPanStart(DragStartDetails _) {
    _auto = false;
  }

  void _onPanUpdate(DragUpdateDetails d) {
    setState(() {
      _yaw += d.delta.dx * 0.01;
      _pitch = (_pitch + d.delta.dy * 0.008).clamp(0.12, 1.35);
    });
  }

  @override
  Widget build(BuildContext context) {
    return Stack(
      children: [
        RawGestureDetector(
          behavior: HitTestBehavior.opaque,
          gestures: <Type, GestureRecognizerFactory>{
            _EagerPanGestureRecognizer: GestureRecognizerFactoryWithHandlers<_EagerPanGestureRecognizer>(
              () => _EagerPanGestureRecognizer(),
              (_EagerPanGestureRecognizer instance) {
                instance
                  ..onStart = _onPanStart
                  ..onUpdate = _onPanUpdate;
              },
            ),
          },
          child: GestureDetector(
            behavior: HitTestBehavior.opaque,
            onDoubleTap: () {
              setState(() {
                _yaw = 0.95;
                _pitch = 0.55;
                _zoom = 1.0;
                _auto = true;
              });
            },
            onTapUp: (d) {
              final hit = _painter.pick(d.localPosition);
              if (hit != null && widget.onCell != null) widget.onCell!(hit.$1, hit.$2);
            },
            child: ClipRect(
              child: CustomPaint(
                painter: _painter
                  ..update(
                    grid: widget.grid,
                    result: widget.result,
                    yaw: _yaw,
                    pitch: _pitch,
                    zoom: _zoom,
                    selRi: widget.selRi,
                    selCi: widget.selCi,
                  ),
                size: Size.infinite,
              ),
            ),
          ),
        ),
        // кнопки зума 3D
        Positioned(
          right: 8,
          bottom: 8,
          child: Column(
            children: [
              _zBtn(Icons.add, () => setState(() => _zoom = (_zoom * 1.15).clamp(0.55, 2.4))),
              const SizedBox(height: 6),
              _zBtn(Icons.remove, () => setState(() => _zoom = (_zoom / 1.15).clamp(0.55, 2.4))),
              const SizedBox(height: 6),
              _zBtn(Icons.threed_rotation, () {
                setState(() {
                  _auto = !_auto;
                });
              }),
            ],
          ),
        ),
        const Positioned(
          left: 8,
          bottom: 8,
          child: Text('тяните пальцем · double-tap = сброс',
              style: TextStyle(fontSize: 10, color: Colors.white38)),
        ),
      ],
    );
  }

  Widget _zBtn(IconData icon, VoidCallback onTap) {
    return Material(
      color: Colors.black54,
      shape: const CircleBorder(),
      child: InkWell(
        customBorder: const CircleBorder(),
        onTap: onTap,
        child: Padding(
          padding: const EdgeInsets.all(8),
          child: Icon(icon, size: 18, color: Colors.white),
        ),
      ),
    );
  }
}

class _Proj {
  _Proj(this.sx, this.sy, this.depth);
  final double sx, sy, depth;
}

class _Quad {
  _Quad(this.ri, this.ci, this.pts, this.depth, this.cx, this.cy, this.sideA, this.sideB);
  final int ri, ci;
  final List<Offset> pts;
  final List<Offset> sideA, sideB;
  final double depth, cx, cy;
}

class _SurfacePainter extends CustomPainter {
  MapGrid? grid;
  MapResult? result;
  double yaw = 0.9, pitch = 0.62, zoom = 1.0;
  int? selRi, selCi;
  final List<_Quad> _quads = [];

  void update({
    required MapGrid grid,
    MapResult? result,
    required double yaw,
    required double pitch,
    required double zoom,
    int? selRi,
    int? selCi,
  }) {
    this.grid = grid;
    this.result = result;
    this.yaw = yaw;
    this.pitch = pitch;
    this.zoom = zoom;
    this.selRi = selRi;
    this.selCi = selCi;
  }

  (int, int)? pick(Offset p) {
    _Quad? best;
    var bd = 1e9;
    for (final q in _quads) {
      final d = math.sqrt(math.pow(q.cx - p.dx, 2) + math.pow(q.cy - p.dy, 2));
      if (d < 28 && d < bd) {
        bd = d;
        best = q;
      }
    }
    return best == null ? null : (best.ri, best.ci);
  }

  @override
  void paint(Canvas canvas, Size size) {
    final g = grid;
    if (g == null || g.rows == 0 || g.cols == 0) return;
    _quads.clear();

    final rows = g.rows, cols = g.cols;
    final vmin = g.vmin, vmax = g.vmax;
    final scale = math.min(size.width, size.height) * 0.34 * zoom;
    final cx = size.width / 2, cy = size.height * 0.52;
    const cam = 3.4;

    _Proj proj(double x, double y, double z) {
      final x1 = x * math.cos(yaw) + z * math.sin(yaw);
      final z1 = -x * math.sin(yaw) + z * math.cos(yaw);
      final y2 = y * math.cos(pitch) - z1 * math.sin(pitch);
      final z2 = y * math.sin(pitch) + z1 * math.cos(pitch);
      final f = cam / (cam - z2);
      return _Proj(cx + x1 * scale * f, cy - y2 * scale * f, z2);
    }

    double hgt(double v) => 0.12 + ((v - vmin) / ((vmax - vmin) == 0 ? 1 : (vmax - vmin))) * 0.72;

    final floorPaint = Paint()
      ..color = const Color(0xFF1B2330)
      ..style = PaintingStyle.stroke
      ..strokeWidth = 1;
    final floor = [proj(-1, 0, -1), proj(1, 0, -1), proj(1, 0, 1), proj(-1, 0, 1)];
    final fp = Path()
      ..moveTo(floor[0].sx, floor[0].sy)
      ..lineTo(floor[1].sx, floor[1].sy)
      ..lineTo(floor[2].sx, floor[2].sy)
      ..lineTo(floor[3].sx, floor[3].sy)
      ..close();
    canvas.drawPath(fp, floorPaint);

    for (var ri = 0; ri < rows; ri++) {
      for (var ci = 0; ci < cols; ci++) {
        final gz0 = (ri / math.max(1, rows - 1)) * 2 - 1;
        final gz1 = rows > 1 ? ((ri + 1) / (rows - 1)) * 2 - 1 : gz0;
        final gx0 = (ci / math.max(1, cols - 1)) * 2 - 1;
        final gx1 = cols > 1 ? ((ci + 1) / (cols - 1)) * 2 - 1 : gx0;
        const scx = 0.96;
        final x0 = gx0 + ((gx1 - gx0) * (1 - scx)) / 2;
        final x1 = gx1 - ((gx1 - gx0) * (1 - scx)) / 2;
        final z0 = gz0 + ((gz1 - gz0) * (1 - scx)) / 2;
        final z1 = gz1 - ((gz1 - gz0) * (1 - scx)) / 2;
        final h = hgt(g.data[ri][ci]);
        final p = [proj(x0, h, z0), proj(x1, h, z0), proj(x1, h, z1), proj(x0, h, z1)];
        final depth = (p[0].depth + p[1].depth + p[2].depth + p[3].depth) / 4;
        final sideA = [proj(x0, h, z1), proj(x1, h, z1), proj(x1, 0, z1), proj(x0, 0, z1)];
        final sideB = [proj(x1, h, z0), proj(x1, h, z1), proj(x1, 0, z1), proj(x1, 0, z0)];
        _quads.add(_Quad(
          ri,
          ci,
          [for (final q in p) Offset(q.sx, q.sy)],
          depth,
          (p[0].sx + p[2].sx) / 2,
          (p[0].sy + p[2].sy) / 2,
          [for (final q in sideA) Offset(q.sx, q.sy)],
          [for (final q in sideB) Offset(q.sx, q.sy)],
        ));
      }
    }

    _quads.sort((a, b) => a.depth.compareTo(b.depth));

    final sidePaint = Paint()..color = const Color.fromARGB(235, 10, 14, 20);
    final edgePaint = Paint()
      ..style = PaintingStyle.stroke
      ..strokeWidth = 0.55
      ..color = const Color.fromARGB(120, 6, 8, 12);

    for (final q in _quads) {
      void poly(List<Offset> pts, Paint fill) {
        final path = Path()..moveTo(pts[0].dx, pts[0].dy);
        for (var i = 1; i < pts.length; i++) {
          path.lineTo(pts[i].dx, pts[i].dy);
        }
        path.close();
        canvas.drawPath(path, fill);
      }

      poly(q.sideA, sidePaint);
      poly(q.sideB, sidePaint);

      final v = g.data[q.ri][q.ci];
      final d = result?.delta[q.ri][q.ci] ?? 0;
      final sel = selRi == q.ri && selCi == q.ci;

      var cellColor = getHeatmapColor(v, vmin, vmax);
      if (sel) cellColor = Color.lerp(cellColor, Colors.white, 0.35)!;
      poly(q.pts, Paint()..color = cellColor);

      final topPath = Path()..moveTo(q.pts[0].dx, q.pts[0].dy);
      for (var i = 1; i < q.pts.length; i++) {
        topPath.lineTo(q.pts[i].dx, q.pts[i].dy);
      }
      topPath.close();

      if (sel) {
        canvas.drawPath(
            topPath,
            Paint()
              ..style = PaintingStyle.stroke
              ..strokeWidth = 2.0
              ..color = Colors.white);
      } else {
        canvas.drawPath(topPath, edgePaint);
      }

      if (d != 0) {
        canvas.drawCircle(Offset(q.cx, q.cy), 2.6, Paint()..color = Colors.lightBlueAccent);
      }
    }
  }

  @override
  bool shouldRepaint(covariant _SurfacePainter oldDelegate) => true;
}
"""

FILES["lib/maps_lab/maps_lab_page.dart"] = r"""
/// Экран Map Lab: папки, локальные XML, heatmap, zoom 2D, 3D-rotate.
library;

import 'dart:convert';
import 'dart:io';

import 'package:flutter/material.dart';
import 'package:flutter/services.dart' show rootBundle;
import 'package:path_provider/path_provider.dart';
import 'package:permission_handler/permission_handler.dart';
import 'package:share_plus/share_plus.dart';

import 'map3d_view.dart';
import 'maps_lab_core.dart';
import 'maps_lab_log.dart';

/// Тепловая карта: зелёный → жёлтый → оранжевый → красный.
Color getHeatmapColor(double v, double min, double max) {
  if (max <= min) return const Color(0xFF4CAF50);
  final t = ((v - min) / (max - min)).clamp(0.0, 1.0);
  if (t < 0.33) {
    return Color.lerp(const Color(0xFF4CAF50), const Color(0xFFFFEB3B), t / 0.33)!;
  } else if (t < 0.66) {
    return Color.lerp(const Color(0xFFFFEB3B), const Color(0xFFFF9800), (t - 0.33) / 0.33)!;
  } else {
    return Color.lerp(const Color(0xFFFF9800), const Color(0xFFF44336), (t - 0.66) / 0.34)!;
  }
}

class MapLabPage extends StatefulWidget {
  const MapLabPage({super.key});

  @override
  State<MapLabPage> createState() => _MapLabPageState();
}

class _MapLabPageState extends State<MapLabPage> {
  static const _defKeys = ['A2TB100B', 'A2TB100K', '32BITBASE'];

  final _journal = <String>[];
  DefSet? _defs;
  Map<String, MapGrid>? _maps;
  LogData? _log;
  Map<String, MapResult>? _results;
  String? _mapName;
  int? _selRi, _selCi;
  bool _view3D = false;
  String _defsSource = '—';

  @override
  void initState() {
    super.initState();
    WidgetsBinding.instance.addPostFrameCallback((_) => _bootstrapDefs());
  }

  void _say(String s) {
    if (!mounted) return;
    setState(() => _journal.add('${TimeOfDay.now().format(context)}  $s'));
  }

  Future<Directory> _cacheDir() async {
    final docs = await getApplicationDocumentsDirectory();
    final dir = Directory('${docs.path}/defs');
    if (!dir.existsSync()) dir.createSync(recursive: true);
    return dir;
  }

  Future<Map<String, String>> _tryAssets() async {
    final out = <String, String>{};
    for (final k in _defKeys) {
      try {
        final s = await rootBundle.loadString('assets/defs/$k.xml');
        if (s.trim().length > 200) out[k] = s;
      } catch (_) {}
    }
    return out;
  }

  Future<Map<String, String>> _tryCache() async {
    final out = <String, String>{};
    final dir = await _cacheDir();
    for (final k in _defKeys) {
      final f = File('${dir.path}/$k.xml');
      if (f.existsSync() && f.lengthSync() > 200) {
        try {
          out[k] = await f.readAsString();
        } catch (_) {}
      }
    }
    return out;
  }

  Future<void> _saveCache(Map<String, String> xmlById) async {
    final dir = await _cacheDir();
    for (final e in xmlById.entries) {
      try {
        await File('${dir.path}/${e.key}.xml').writeAsString(e.value, flush: true);
      } catch (_) {}
    }
  }

  bool _applyDefs(Map<String, String> xmlById, String source) {
    if (xmlById.length < 3) return false;
    try {
      _defs = DefSet.build(xmlById, 'A2TB100B');
      _defsSource = source;
      _say('дефиниции [$source]: ${_defs!.chain.join(' → ')} · '
          'ecuid ${_defs!.meta.ecuid} · ${_defs!.tables.length} таблиц');
      return true;
    } catch (e) {
      _say('!! ошибка разбора дефиниций: $e');
      return false;
    }
  }

  Future<void> _bootstrapDefs() async {
    var xml = await _tryAssets();
    if (_applyDefs(xml, 'APK assets')) {
      await _saveCache(xml);
      setState(() {});
      return;
    }
    xml = await _tryCache();
    if (_applyDefs(xml, 'кэш телефона')) {
      setState(() {});
      return;
    }
    _say('!! дефиниций нет. Нажмите «0 · Дефиниции» и укажите папку с XML.');
    setState(() {});
  }

  Future<Map<String, String>> _scanDefsInDir(Directory root) async {
    final found = <String, File>{};
    bool matchName(String name, String key) {
      final n = name.toLowerCase();
      return n == '${key.toLowerCase()}.xml' || n.contains(key.toLowerCase());
    }

    void consider(File f) {
      final name = f.uri.pathSegments.last;
      for (final k in _defKeys) {
        if (found.containsKey(k)) continue;
        if (matchName(name, k) && f.lengthSync() > 200) found[k] = f;
      }
    }

    try {
      for (final e in root.listSync(followLinks: false)) {
        if (e is File && e.path.toLowerCase().endsWith('.xml')) consider(e);
      }
      for (final e in root.listSync(followLinks: false)) {
        if (e is! Directory) continue;
        try {
          for (final f in e.listSync(followLinks: false)) {
            if (f is File && f.path.toLowerCase().endsWith('.xml')) consider(f);
          }
        } catch (_) {}
      }
    } catch (_) {}

    final out = <String, String>{};
    for (final e in found.entries) {
      try {
        out[e.key] = await e.value.readAsString();
      } catch (_) {}
    }
    return out;
  }

  Future<void> _pickDefs() async {
    final result = await showModalBottomSheet<Object>(
      context: context,
      isScrollControlled: true,
      backgroundColor: Colors.transparent,
      builder: (ctx) => const _FilePickerSheet(
        extensions: ['.xml'],
        title: 'Дефиниции RomRaider (.xml)',
        allowPickFolderDefs: true,
      ),
    );
    if (result == null) return;

    var xml = <String, String>{};
    var source = 'локально';

    if (result is Directory) {
      xml = await _scanDefsInDir(result);
      source = 'папка ${result.path.split('/').last}';
      _say('сканирование ${result.path}: найдено ${xml.length}/3');
    } else if (result is File) {
      xml = await _scanDefsInDir(result.parent);
      if (xml.length < 3) {
        final name = result.uri.pathSegments.last.toUpperCase();
        for (final k in _defKeys) {
          if (name.contains(k)) {
            try {
              xml[k] = await result.readAsString();
            } catch (_) {}
          }
        }
      }
      source = 'файл+папка';
    } else if (result is Map) {
      xml = Map<String, String>.from(result);
    }

    if (xml.length < 3) {
      final miss = _defKeys.where((k) => !xml.containsKey(k)).join(', ');
      _say('!! не хватает XML: $miss');
      return;
    }

    if (_applyDefs(xml, source)) {
      await _saveCache(xml);
      setState(() {});
    }
  }

  Future<File?> _chooseFile(List<String> exts, String title) async {
    final r = await showModalBottomSheet<Object>(
      context: context,
      isScrollControlled: true,
      backgroundColor: Colors.transparent,
      builder: (ctx) => _FilePickerSheet(extensions: exts, title: title),
    );
    return r is File ? r : null;
  }

  Future<void> _pickRom() async {
    if (_defs == null) {
      _say('сначала загрузите дефиниции');
      return;
    }
    final f = await _chooseFile(['.bin', '.hex', '.rom'], 'Выберите прошивку (.bin)');
    if (f == null) return;
    try {
      final bytes = await f.readAsBytes();
      setState(() {
        _results = null;
        _selRi = _selCi = null;
      });
      final parser = RomParser(bytes);
      final idAddr = int.tryParse(_defs?.meta.internalIdAddress ?? '2000', radix: 16) ?? 0x2000;
      final romId = parser.readRomId(idAddr);
      final expected = _defs?.meta.internalIdString ?? '';
      final matched = expected.isEmpty || romId == expected;
      _say('ROM: ${f.uri.pathSegments.last} · ${(bytes.length / 1024).toStringAsFixed(0)} КБ · '
          'ID="$romId"${matched ? '' : '  (ожидался $expected)'}');
      _maps = parser.extractKeys(_defs!);
      _mapName = _maps!.keys.firstOrNull;
      _say('карт извлечено: ${_maps!.length} из ${RomParser.keyTables.length}');
      setState(() {});
    } catch (e) {
      _say('!! ошибка парсинга ROM: $e');
    }
  }

  Future<void> _pickLog() async {
    final f = await _chooseFile(['.csv', '.txt', '.log'], 'Выберите файл лога (.csv)');
    if (f == null) return;
    try {
      final text = utf8.decode(await f.readAsBytes(), allowMalformed: true);
      _log = LogData.parse(text);
      final health = LogAudit.check(_log!);
      _say('лог: ${f.uri.pathSegments.last} · ${_log!.rows} строк');
      for (final n in health.notes) {
        _say('   $n');
      }
      setState(() {});
    } catch (e) {
      _say('!! ошибка чтения лога: $e');
    }
  }

  void _runAnalysis() {
    if (_maps == null || _log == null) return;
    const cfg = AnalyzerConfig();
    _results = Analyzer(cfg).run(_maps!, _log!);
    var dec = 0, inc = 0;
    for (final r in _results!.values) {
      dec += r.dec;
      inc += r.inc;
    }
    _say('анализ: вердикты по ${_results!.length} картам · убавить $dec, прибавить $inc ячеек');
    setState(() {});
  }

  Future<void> _export() async {
    if (_maps == null || _results == null) return;
    final dir = await getApplicationDocumentsDirectory();
    final stamp = DateTime.now().toIso8601String().replaceAll(':', '-').split('.').first;
    final outDir = Directory('${dir.path}/maplab_$stamp')..createSync(recursive: true);
    final files = <XFile>[];
    final md = StringBuffer('# Map Lab A2TB100B · рекомендации\n');
    for (final e in _maps!.entries) {
      final g = e.value;
      final res = _results![e.key];
      if (res == null) continue;
      final head = '\t${g.x.values.map((v) => v.toStringAsFixed(2)).join('\t')}';
      final rows = <String>[
        for (var ri = 0; ri < g.rows; ri++)
          '${g.y.values[ri].toStringAsFixed(2)}\t${[
            for (var ci = 0; ci < g.cols; ci++)
              (g.data[ri][ci] + res.delta[ri][ci]).toStringAsFixed(3),
          ].join('\t')}',
      ];
      final tsv = '$head\n${rows.join('\n')}\n';
      final file = File('${outDir.path}/${e.key.replaceAll(RegExp(r'[/ ]'), '_')}_recommended.tsv')
        ..writeAsStringSync(tsv);
      files.add(XFile(file.path));
      md.writeln('\n## ${e.key} (${g.units})');
      for (var ri = 0; ri < g.rows; ri++) {
        for (var ci = 0; ci < g.cols; ci++) {
          final d = res.delta[ri][ci];
          if (d == 0) continue;
          final inf = res.info[ri][ci];
          md.writeln('- ${g.x.values[ci]} × ${g.y.values[ri]}: ${g.data[ri][ci]} → '
              '${(g.data[ri][ci] + d).toStringAsFixed(2)} (${d > 0 ? '+' : ''}$d)'
              '${inf != null && inf.why.isNotEmpty ? ' — ${inf.why}' : ''}');
        }
      }
    }
    final mf = File('${outDir.path}/recommendations.md')..writeAsStringSync(md.toString());
    files.insert(0, XFile(mf.path));
    _say('экспорт: ${files.length} файлов → ${outDir.path}');
    await Share.shareXFiles(files, text: 'Map Lab A2TB100B — рекомендации');
  }

  @override
  Widget build(BuildContext context) {
    final theme = Theme.of(context);
    final map = _maps == null ? null : _maps![_mapName];
    MapResult? res;
    if (map != null) res = _results?[map.name];

    // ВАЖНО: карта в Expanded, а не внутри ListView — иначе 3D не крутится.
    return Scaffold(
      appBar: AppBar(
        title: const Text('Map Lab · A2TB100B'),
        actions: [
          if (_results != null)
            IconButton(icon: const Icon(Icons.ios_share), onPressed: _export, tooltip: 'Экспорт'),
        ],
      ),
      body: Column(
        children: [
          Padding(
            padding: const EdgeInsets.fromLTRB(12, 10, 12, 0),
            child: Column(
              crossAxisAlignment: CrossAxisAlignment.start,
              children: [
                Wrap(
                  spacing: 8,
                  runSpacing: 8,
                  children: [
                    _stepBtn('0 · Дефиниции', Icons.folder_special, _pickDefs, ok: _defs != null),
                    _stepBtn('1 · Прошивка', Icons.memory, _pickRom, ok: _maps != null),
                    _stepBtn('2 · Лог', Icons.description, _pickLog, ok: _log != null),
                    _stepBtn('3 · Анализ', Icons.psychology,
                        (_maps != null && _log != null) ? _runAnalysis : null,
                        ok: _results != null),
                    _stepBtn('Экспорт', Icons.ios_share, _results != null ? _export : null),
                  ],
                ),
                const SizedBox(height: 6),
                Text(
                  _defs != null
                      ? 'XML: $_defsSource · ${_defs!.meta.xmlid}'
                      : 'Нужны A2TB100B.xml, A2TB100K.xml, 32BITBASE.xml',
                  style: theme.textTheme.bodySmall?.copyWith(
                    color: _defs != null ? Colors.greenAccent : Colors.amber,
                  ),
                ),
                if (_maps != null) ...[
                  const SizedBox(height: 8),
                  SizedBox(
                    height: 40,
                    child: ListView(
                      scrollDirection: Axis.horizontal,
                      children: [
                        for (final e in _maps!.entries)
                          Padding(
                            padding: const EdgeInsets.only(right: 8),
                            child: ChoiceChip(
                              selected: _mapName == e.key,
                              onSelected: (_) => setState(() {
                                _mapName = e.key;
                                _selRi = _selCi = null;
                              }),
                              label: Text(e.key, style: const TextStyle(fontSize: 11)),
                            ),
                          ),
                      ],
                    ),
                  ),
                  Row(
                    children: [
                      Expanded(
                        child: Text(
                          map == null
                              ? ''
                              : '${map.name} · ${map.rows}×${map.cols}',
                          style: theme.textTheme.bodySmall,
                          overflow: TextOverflow.ellipsis,
                        ),
                      ),
                      SegmentedButton<bool>(
                        segments: const [
                          ButtonSegment(value: false, icon: Icon(Icons.table_chart, size: 16), label: Text('2D')),
                          ButtonSegment(value: true, icon: Icon(Icons.threed_rotation, size: 16), label: Text('3D')),
                        ],
                        selected: {_view3D},
                        onSelectionChanged: (s) => setState(() => _view3D = s.first),
                      ),
                    ],
                  ),
                ],
              ],
            ),
          ),
          if (map != null)
            Expanded(
              flex: 5,
              child: Padding(
                padding: const EdgeInsets.fromLTRB(12, 8, 12, 4),
                child: Card(
                  clipBehavior: Clip.antiAlias,
                  child: _view3D
                      ? Map3DView(
                          grid: map,
                          result: res,
                          selRi: _selRi,
                          selCi: _selCi,
                          onCell: (ri, ci) => setState(() {
                            _selRi = ri;
                            _selCi = ci;
                          }),
                        )
                      : _Table2D(
                          grid: map,
                          result: res,
                          selRi: _selRi,
                          selCi: _selCi,
                          onCell: (ri, ci) => setState(() {
                            _selRi = ri;
                            _selCi = ci;
                          }),
                        ),
                ),
              ),
            ),
          if (map != null && _selRi != null && _selCi != null)
            Padding(
              padding: const EdgeInsets.symmetric(horizontal: 12),
              child: _inspector(map, res),
            ),
          if (map != null)
            Padding(
              padding: const EdgeInsets.fromLTRB(12, 4, 12, 0),
              child: Wrap(spacing: 12, runSpacing: 4, children: [
                _legend(const Color(0xFF4CAF50), 'низ'),
                _legend(const Color(0xFFFFEB3B), 'середина'),
                _legend(const Color(0xFFF44336), 'верх'),
                _legend(Colors.lightBlueAccent, 'правка'),
              ]),
            ),
          Expanded(
            flex: map == null ? 8 : 2,
            child: Padding(
              padding: const EdgeInsets.all(12),
              child: Column(
                crossAxisAlignment: CrossAxisAlignment.start,
                children: [
                  Text('Журнал', style: theme.textTheme.titleSmall),
                  const SizedBox(height: 6),
                  Expanded(
                    child: Container(
                      width: double.infinity,
                      padding: const EdgeInsets.all(10),
                      decoration: BoxDecoration(
                        color: theme.colorScheme.surfaceContainerHighest.withValues(alpha: 0.35),
                        borderRadius: BorderRadius.circular(8),
                      ),
                      child: ListView(
                        children: [
                          for (final j in _journal)
                            Text(j, style: const TextStyle(fontFamily: 'monospace', fontSize: 11)),
                        ],
                      ),
                    ),
                  ),
                ],
              ),
            ),
          ),
        ],
      ),
    );
  }

  Widget _stepBtn(String label, IconData icon, VoidCallback? onTap, {bool ok = false}) {
    return FilledButton.tonalIcon(
      onPressed: onTap,
      icon: Icon(ok ? Icons.check_circle : icon, size: 18),
      label: Text(label, style: const TextStyle(fontSize: 12)),
    );
  }

  Widget _legend(Color c, String t) => Row(mainAxisSize: MainAxisSize.min, children: [
        Container(width: 12, height: 8, decoration: BoxDecoration(color: c, borderRadius: BorderRadius.circular(2))),
        const SizedBox(width: 6),
        Text(t, style: const TextStyle(fontSize: 10.5)),
      ]);

  Widget _inspector(MapGrid g, MapResult? res) {
    final ri = _selRi!, ci = _selCi!;
    final v = g.data[ri][ci];
    final d = res?.delta[ri][ci] ?? 0;
    final inf = res?.info[ri][ci];
    final col = d < 0 ? const Color(0xFFFF5C5C) : d > 0 ? const Color(0xFFA8FF3E) : null;
    return Card(
      child: Padding(
        padding: const EdgeInsets.all(10),
        child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
          Text('Ячейка: ${g.x.name}=${g.x.values[ci]} · ${g.y.name}=${g.y.values[ri]}',
              style: const TextStyle(fontFamily: 'monospace', fontSize: 12)),
          const SizedBox(height: 4),
          Row(children: [
            Text(v.toStringAsFixed(2),
                style: const TextStyle(
                    fontSize: 18, fontFamily: 'monospace', decoration: TextDecoration.lineThrough)),
            const SizedBox(width: 8),
            const Icon(Icons.arrow_forward, size: 16),
            const SizedBox(width: 8),
            Text('${(v + d).toStringAsFixed(2)} ${g.units}',
                style: TextStyle(fontSize: 22, fontFamily: 'monospace', color: col)),
          ]),
          if (inf != null && inf.why.isNotEmpty)
            Padding(
              padding: const EdgeInsets.only(top: 4),
              child: Text(inf.why, style: const TextStyle(fontSize: 12)),
            ),
        ]),
      ),
    );
  }
}

class _FilePickerSheet extends StatefulWidget {
  const _FilePickerSheet({
    required this.extensions,
    required this.title,
    this.allowPickFolderDefs = false,
  });
  final List<String> extensions;
  final String title;
  final bool allowPickFolderDefs;

  @override
  State<_FilePickerSheet> createState() => _FilePickerSheetState();
}

class _FilePickerSheetState extends State<_FilePickerSheet> {
  Directory _currentDir = Directory('/storage/emulated/0');
  bool _hasPermission = false;
  List<FileSystemEntity> _items = [];

  @override
  void initState() {
    super.initState();
    _checkPermissionAndRefresh();
  }

  Future<void> _checkPermissionAndRefresh() async {
    var ok = false;
    try {
      ok = await Permission.manageExternalStorage.isGranted;
    } catch (_) {}
    if (!ok) {
      final docs = await getApplicationDocumentsDirectory();
      _currentDir = docs;
    } else if (!_currentDir.existsSync()) {
      _currentDir = Directory('/storage/emulated/0');
    }
    setState(() => _hasPermission = ok);
    _refreshList();
  }

  Future<void> _requestPermission() async {
    try {
      await Permission.manageExternalStorage.request();
    } catch (_) {}
    try {
      await openAppSettings();
    } catch (_) {}
    _checkPermissionAndRefresh();
  }

  void _refreshList() {
    try {
      if (!_currentDir.existsSync()) {
        setState(() => _items = []);
        return;
      }
      final list = _currentDir.listSync(followLinks: false);
      final dirs = <Directory>[];
      final files = <File>[];
      for (final e in list) {
        final name = e.path.split('/').last;
        if (name.startsWith('.')) continue;
        if (e is Directory) {
          dirs.add(e);
        } else if (e is File) {
          if (widget.extensions.any((x) => name.toLowerCase().endsWith(x))) files.add(e);
        }
      }
      dirs.sort((a, b) => a.path.toLowerCase().compareTo(b.path.toLowerCase()));
      files.sort((a, b) => b.lastModifiedSync().compareTo(a.lastModifiedSync()));
      setState(() => _items = [...dirs, ...files]);
    } catch (_) {
      setState(() => _items = []);
    }
  }

  void _goUp() {
    final parent = _currentDir.parent;
    if (parent.path.length >= 4) {
      setState(() => _currentDir = parent);
      _refreshList();
    }
  }

  void _goTo(String path) {
    final d = Directory(path);
    if (d.existsSync()) {
      setState(() => _currentDir = d);
      _refreshList();
    }
  }

  @override
  Widget build(BuildContext context) {
    return Container(
      height: MediaQuery.of(context).size.height * 0.85,
      decoration: const BoxDecoration(
        color: Color(0xFF141921),
        borderRadius: BorderRadius.vertical(top: Radius.circular(16)),
      ),
      child: Column(
        children: [
          Padding(
            padding: const EdgeInsets.symmetric(horizontal: 16, vertical: 12),
            child: Row(
              children: [
                Expanded(
                  child: Text(widget.title, style: const TextStyle(fontWeight: FontWeight.bold, fontSize: 16)),
                ),
                IconButton(icon: const Icon(Icons.close), onPressed: () => Navigator.of(context).pop()),
              ],
            ),
          ),
          if (!_hasPermission)
            Padding(
              padding: const EdgeInsets.symmetric(horizontal: 12),
              child: TextButton(onPressed: _requestPermission, child: const Text('РАЗРЕШИТЬ доступ к файлам')),
            ),
          SingleChildScrollView(
            scrollDirection: Axis.horizontal,
            padding: const EdgeInsets.symmetric(horizontal: 12, vertical: 4),
            child: Row(children: [
              _chip('Память', '/storage/emulated/0'),
              _chip('Downloads', '/storage/emulated/0/Download'),
              _chip('Telegram', '/storage/emulated/0/Telegram'),
              _chip('Documents', '/storage/emulated/0/Documents'),
            ]),
          ),
          Container(
            padding: const EdgeInsets.symmetric(horizontal: 8, vertical: 4),
            color: Colors.black26,
            child: Row(
              children: [
                IconButton(icon: const Icon(Icons.arrow_upward, size: 18), onPressed: _goUp),
                Expanded(
                  child: Text(_currentDir.path,
                      style: const TextStyle(fontFamily: 'monospace', fontSize: 11, color: Colors.cyanAccent),
                      overflow: TextOverflow.ellipsis),
                ),
                if (widget.allowPickFolderDefs)
                  FilledButton.tonal(
                    onPressed: () => Navigator.of(context).pop(_currentDir),
                    child: const Text('Взять XML из ЭТОЙ папки', style: TextStyle(fontSize: 11)),
                  ),
              ],
            ),
          ),
          const Divider(height: 1),
          Expanded(
            child: ListView.builder(
              itemCount: _items.length,
              itemBuilder: (ctx, i) {
                final item = _items[i];
                final name = item.path.split('/').last;
                if (item is Directory) {
                  return ListTile(
                    dense: true,
                    leading: const Icon(Icons.folder, color: Colors.amber),
                    title: Text(name),
                    onTap: () {
                      setState(() => _currentDir = item);
                      _refreshList();
                    },
                  );
                } else if (item is File) {
                  return ListTile(
                    dense: true,
                    leading: const Icon(Icons.insert_drive_file, color: Colors.lightBlueAccent),
                    title: Text(name, style: const TextStyle(fontFamily: 'monospace', fontSize: 13)),
                    onTap: () => Navigator.of(ctx).pop(item),
                  );
                }
                return const SizedBox.shrink();
              },
            ),
          ),
        ],
      ),
    );
  }

  Widget _chip(String label, String path) {
    return Padding(
      padding: const EdgeInsets.only(right: 6),
      child: ActionChip(
        visualDensity: VisualDensity.compact,
        label: Text(label, style: const TextStyle(fontSize: 11)),
        onPressed: () => _goTo(path),
      ),
    );
  }
}

/// 2D таблица с pinch-zoom и кнопками +/−.
class _Table2D extends StatefulWidget {
  const _Table2D({required this.grid, this.result, this.selRi, this.selCi, this.onCell});
  final MapGrid grid;
  final MapResult? result;
  final int? selRi, selCi;
  final void Function(int, int)? onCell;

  @override
  State<_Table2D> createState() => _Table2DState();
}

class _Table2DState extends State<_Table2D> {
  final _tc = TransformationController();
  static const _minS = 0.25;
  static const _maxS = 6.0;

  @override
  void dispose() {
    _tc.dispose();
    super.dispose();
  }

  void _zoomBy(double factor) {
    final m = _tc.value.clone();
    final s = m.getMaxScaleOnAxis();
    final next = (s * factor).clamp(_minS, _maxS);
    final f = next / s;
    // зум относительно центра видимой области
    final child = context.findRenderObject() as RenderBox?;
    if (child == null) {
      _tc.value = m.scaled(f);
      return;
    }
    final center = child.size.center(Offset.zero);
    final scene = _tc.toScene(center);
    m.translate(scene.dx, scene.dy);
    m.scale(f);
    m.translate(-scene.dx, -scene.dy);
    _tc.value = m;
    setState(() {});
  }

  void _reset() {
    _tc.value = Matrix4.identity();
    setState(() {});
  }

  @override
  Widget build(BuildContext context) {
    final g = widget.grid;
    return Stack(
      children: [
        InteractiveViewer(
          transformationController: _tc,
          minScale: _minS,
          maxScale: _maxS,
          boundaryMargin: const EdgeInsets.all(200),
          constrained: false,
          panEnabled: true,
          scaleEnabled: true,
          child: Padding(
            padding: const EdgeInsets.all(8),
            child: Table(
              defaultColumnWidth: const IntrinsicColumnWidth(),
              children: [
                TableRow(children: [
                  _head('${g.y.guess}↓ ${g.x.guess}→'),
                  for (final v in g.x.values) _head(v.toStringAsFixed(0), accent: true),
                ]),
                for (var ri = 0; ri < g.rows; ri++)
                  TableRow(children: [
                    _head(g.y.values[ri].toStringAsFixed(2), accent: true),
                    for (var ci = 0; ci < g.cols; ci++) _cell(ri, ci),
                  ]),
              ],
            ),
          ),
        ),
        Positioned(
          right: 8,
          bottom: 8,
          child: Column(
            children: [
              _zBtn(Icons.add, () => _zoomBy(1.25)),
              const SizedBox(height: 6),
              _zBtn(Icons.remove, () => _zoomBy(1 / 1.25)),
              const SizedBox(height: 6),
              _zBtn(Icons.center_focus_strong, _reset),
            ],
          ),
        ),
        const Positioned(
          left: 8,
          bottom: 8,
          child: Text('pinch / кнопки = зум', style: TextStyle(fontSize: 10, color: Colors.white38)),
        ),
      ],
    );
  }

  Widget _zBtn(IconData icon, VoidCallback onTap) {
    return Material(
      color: Colors.black54,
      shape: const CircleBorder(),
      child: InkWell(
        customBorder: const CircleBorder(),
        onTap: onTap,
        child: Padding(
          padding: const EdgeInsets.all(8),
          child: Icon(icon, size: 18, color: Colors.white),
        ),
      ),
    );
  }

  Widget _head(String t, {bool accent = false}) => Container(
        padding: const EdgeInsets.symmetric(horizontal: 6, vertical: 4),
        decoration: BoxDecoration(border: Border.all(color: Colors.white12)),
        child: Text(t,
            style: TextStyle(
                fontSize: 9.5,
                fontFamily: 'monospace',
                color: accent ? const Color(0xCC4BE1FF) : Colors.white54)),
      );

  Widget _cell(int ri, int ci) {
    final g = widget.grid;
    final v = g.data[ri][ci];
    final d = widget.result?.delta[ri][ci] ?? 0;
    final sel = widget.selRi == ri && widget.selCi == ci;
    final heat = getHeatmapColor(v, g.vmin, g.vmax).withValues(alpha: 0.55);
    final hasFix = d != 0;

    return InkWell(
      onTap: () => widget.onCell?.call(ri, ci),
      child: Container(
        constraints: const BoxConstraints(minWidth: 48, minHeight: 40),
        padding: const EdgeInsets.symmetric(horizontal: 4, vertical: 3),
        decoration: BoxDecoration(
          color: heat,
          border: Border.all(
            color: sel
                ? Colors.white
                : hasFix
                    ? Colors.lightBlueAccent
                    : Colors.white12,
            width: sel ? 1.6 : (hasFix ? 1.2 : 0.5),
          ),
        ),
        child: Column(
          mainAxisAlignment: MainAxisAlignment.center,
          children: [
            Text(v.toStringAsFixed(2),
                style: const TextStyle(fontSize: 10, fontFamily: 'monospace', color: Colors.white)),
            if (hasFix)
              Text((v + d).toStringAsFixed(2),
                  style: const TextStyle(
                      fontSize: 11,
                      fontWeight: FontWeight.bold,
                      fontFamily: 'monospace',
                      color: Colors.lightBlueAccent)),
          ],
        ),
      ),
    );
  }
}
"""

FILES["test/maps_lab_test.dart"] = r"""
import 'dart:typed_data';
import 'package:flutter_test/flutter_test.dart';
import 'package:__PKG__/maps_lab/maps_lab_core.dart';
import 'package:__PKG__/maps_lab/maps_lab_log.dart';

void main() {
  group('SafeExpr', () {
    test('uint8 timing: (x*.3515625)-20', () {
      final f = SafeExpr.compile('(x*.3515625)-20');
      expect(f(100), closeTo(15.15625, 0.0001));
      expect(f(0), -20.0);
    });
    test('estimated AFR: 14.7/(1+x*.0078125)', () {
      final f = SafeExpr.compile('14.7/(1+x*.0078125)');
      expect(f(80), closeTo(9.0461, 0.001));
    });
    test('float identity + garbage guard', () {
      expect(SafeExpr.compile('x')(42.5), 42.5);
      expect(SafeExpr.compile('eval(x)+1')(3), 3);
    });
  });

  const baseXml = '''
<rom>
  <romid><xmlid>32BITBASE</xmlid></romid>
  <scaling name="Timing8" units="deg" toexpr="(x*.3515625)-20" frexpr="(x+20)/.3515625" format="%.2f" storagetype="uint8" endian="big"/>
  <scaling name="RPM" units="RPM" toexpr="x" frexpr="x" format="%.0f" storagetype="float" endian="big"/>
  <table name="Base Timing" category="Ignition" type="3D" level="4" scaling="Timing8">
    <table name="Engine Load" type="X Axis" elements="2" scaling="RPM"/>
    <table name="Engine Speed" type="Y Axis" elements="3" scaling="RPM"/>
  </table>
</rom>''';

  const derivedXml = '''
<rom>
  <romid><xmlid>TEST_ROM</xmlid><internalidaddress>2000</internalidaddress><internalidstring>TEST_ROM</internalidstring></romid>
  <include>32BITBASE</include>
  <table name="Base Timing" address="1000">
    <table name="X" address="2400" elements="4"/>
    <table name="Y" address="2500"/>
  </table>
</rom>''';

  test('merge include-цепочки: адрес у производного, скейлинг наследуется', () {
    final defs = DefSet.build({'32BITBASE': baseXml, 'TEST_ROM': derivedXml}, 'TEST_ROM');
    expect(defs.chain, ['32BITBASE', 'TEST_ROM']);
    final t = defs.tables['Base Timing']!;
    expect(t.dataAddress, 0x1000);
    expect(t.dataScaling!.units, 'deg');
    expect(t.axes[0].name, 'Engine Load');
    expect(t.axes[0].elements, 4);
    expect(t.axes[0].address, 0x2400);
    expect(t.axes[1].elements, 3);
    expect(defs.isReadable3D(t), isTrue);
  });

  test('ROM: декод float32-осей и uint8-данных big-endian', () {
    final rom = Uint8List(0x3000);
    final bd = ByteData.view(rom.buffer);
    for (var i = 0; i < 8; i++) {
      bd.setUint8(0x2000 + i, 'TEST_ROM'.codeUnitAt(i));
    }
    final xs = [1000.0, 2000.0, 3000.0, 4000.0];
    for (var i = 0; i < 4; i++) {
      bd.setFloat32(0x2400 + i * 4, xs[i], Endian.big);
    }
    final ys = [0.8, 1.6, 2.4];
    for (var i = 0; i < 3; i++) {
      bd.setFloat32(0x2500 + i * 4, ys[i], Endian.big);
    }
    for (var i = 0; i < 12; i++) {
      bd.setUint8(0x1000 + i, 100);
    }

    final defs = DefSet.build({'32BITBASE': baseXml, 'TEST_ROM': derivedXml}, 'TEST_ROM');
    final parser = RomParser(rom);
    expect(parser.readRomId(0x2000), 'TEST_ROM');
    final grid = parser.extract(defs.tables['Base Timing']!)!;
    expect(grid.rows, 3);
    expect(grid.cols, 4);
    expect(grid.x.values.last, 4000);
    expect(grid.y.values.first, closeTo(0.8, 0.001));
    expect(grid.data[2][3], closeTo(15.15625, 0.001));
    expect(grid.kind, 'timing');
  });

  group('LogData + Analyzer', () {
    LogData syntheticLog({required double fbkc}) {
      final rpm = <double?>[], load = <double?>[], f = <double?>[], fl = <double?>[];
      for (var i = 0; i < 600; i++) {
        rpm.add(4000 + (i % 80) - 40);
        load.add(3.0);
        f.add(fbkc);
        fl.add(fbkc * 0.4);
      }
      return LogData()
        ..cols['rpm'] = rpm
        ..cols['load'] = load
        ..cols['fbkc'] = f
        ..cols['flkc'] = fl;
    }

    MapGrid smallGrid() => MapGrid(
          name: 'Base Timing Primary Non-Cruise',
          kind: 'timing',
          units: 'deg',
          addr: 0,
          x: AxisVals('Engine Speed', [3000, 4000, 5000], 'rpm'),
          y: AxisVals('Engine Load', [2.0, 3.0], 'г/об·бар'),
          data: [
            [20, 18, 16],
            [16, 14, 12],
          ],
        );

    test('детон-кластер → отрицательная дельта в своей ячейке и веер сглаживания', () {
      final res = Analyzer(const AnalyzerConfig()).analyzeMap(smallGrid(), syntheticLog(fbkc: -4.0))!;
      final d = res.delta[1][1];
      expect(d, lessThan(0));
      expect(d, greaterThanOrEqualTo(-3.0));
      expect(res.delta[0][0], -0.5);
      expect(res.info[1][1]!.why, contains('детон'));
    });

    test('чистый лог при выключенном allowTimingAdd → ноль правок', () {
      final res = Analyzer(const AnalyzerConfig()).analyzeMap(smallGrid(), syntheticLog(fbkc: 0))!;
      final anyDelta = res.delta.expand((r) => r).any((v) => v != 0);
      expect(anyDelta, isFalse);
    });

    test('CSV: автоопределение ; и десятичной запятой, алиасы колонок', () {
      const csv = 'Time (s);Engine Speed (RPM);Feedback Knock Correction;Load_4B\r\n'
          '0,0;3000;0,0;3,00\r\n'
          '0,1;3100;-2,5;3,10\r\n';
      final log = LogData.parse(csv);
      expect(log['rpm']![1], 3100);
      expect(log['fbkc']![1], -2.5);
      expect(log['load']![1], 3.1);
    });
  });
}
"""

# ── pubspec / manifest / write ──────────────────────────────────────────────
pub = APP / "pubspec.yaml"
text = pub.read_text(encoding="utf-8")

if "assets/defs/" not in text:
    try:
        if re.search(r"(?m)^  assets:\s*$", text):
            text = re.sub(r"(?m)^(  assets:\s*\n)", r"\1    - assets/defs/\n", text, count=1)
        elif re.search(r"(?m)^flutter:\s*$", text):
            text = re.sub(r"(?m)^(flutter:\s*\n)", r"\1  assets:\n    - assets/defs/\n", text, count=1)
        else:
            text += "\nflutter:\n  uses-material-design: true\n  assets:\n    - assets/defs/\n"
        pub.write_text(text, encoding="utf-8")
        print("[pubspec] assets/defs добавлены")
    except Exception as e:
        print(f"[pubspec] {e}")

man = APP / "android/app/src/main/AndroidManifest.xml"
try:
    if man.exists():
        m = man.read_text(encoding="utf-8")
        if "MANAGE_EXTERNAL_STORAGE" not in m:
            m = m.replace(
                "<application",
                '    <uses-permission android:name="android.permission.MANAGE_EXTERNAL_STORAGE"/>\n    <application',
                1,
            )
            man.write_text(m, encoding="utf-8")
            print("[manifest] MANAGE_EXTERNAL_STORAGE добавлен")
except Exception as e:
    print(f"[manifest] {e}")

pkg_match = re.search(r"(?m)^name:\s*(\S+)", text)
PKG = pkg_match.group(1) if pkg_match else "subaru_ssm2_fixed"

for rel, src in FILES.items():
    dest = APP / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_text(src.replace("__PKG__", PKG) + "\n", encoding="utf-8")
    print("[dart]", rel)

RAW = "https://raw.githubusercontent.com/TD-D/SubaruDefs/Stable/ECUFlash/subaru%20metric"
DEFS = {
    "A2TB100B.xml": f"{RAW}/Legacy%20GT/A2TB100B.xml",
    "A2TB100K.xml": f"{RAW}/Legacy%20GT%20spec.B/A2TB100K.xml",
    "32BITBASE.xml": f"{RAW}/Bases/32BITBASE.xml",
}
defs_dir = APP / "assets/defs"
defs_dir.mkdir(parents=True, exist_ok=True)
for name, url in DEFS.items():
    dest = defs_dir / name
    if not dest.exists() or dest.stat().st_size < 1000:
        urllib.request.urlretrieve(url, dest)
    print("[asset]", name, f"{dest.stat().st_size/1024:.0f} КБ")

(APP / "lib/maplab_link.dart").write_text(
    "import 'maps_lab/maps_lab_page.dart';\n"
    "export 'maps_lab/maps_lab_page.dart' show MapLabPage;\n"
    "typedef MapLabTab = MapLabPage;\n",
    encoding="utf-8",
)
print("\nГотово: 3D-вращение (EagerPan) + зум 2D таблицы. Запускайте ячейку 3/3.")

[pubspec] assets/defs добавлены
[manifest] MANAGE_EXTERNAL_STORAGE добавлен
[dart] lib/maps_lab/maps_lab_core.dart
[dart] lib/maps_lab/maps_lab_log.dart
[dart] lib/maps_lab/map3d_view.dart
[dart] lib/maps_lab/maps_lab_page.dart
[dart] test/maps_lab_test.dart
[asset] A2TB100B.xml 2 КБ
[asset] A2TB100K.xml 28 КБ
[asset] 32BITBASE.xml 489 КБ

Готово: 3D-вращение (EagerPan) + зум 2D таблицы. Запускайте ячейку 3/3.


In [4]:
# @title 2b/3-fix | Map Lab UI: большая карта + скролл всей страницы
from pathlib import Path
import json, re

CONFIG = Path("/content/ssm2_fixed_env.json")
CFG = json.loads(CONFIG.read_text(encoding="utf-8"))
APP = Path(CFG["app"])
pub = (APP / "pubspec.yaml").read_text(encoding="utf-8")
PKG = re.search(r"(?m)^name:\s*(\S+)", pub).group(1)

PAGE = r'''
/// Экран Map Lab: крупная карта + прокрутка всей страницы (журнал виден).
library;

import 'dart:convert';
import 'dart:io';

import 'package:flutter/material.dart';
import 'package:flutter/services.dart' show rootBundle;
import 'package:path_provider/path_provider.dart';
import 'package:permission_handler/permission_handler.dart';
import 'package:share_plus/share_plus.dart';

import 'map3d_view.dart';
import 'maps_lab_core.dart';
import 'maps_lab_log.dart';

/// Тепловая карта: зелёный → жёлтый → оранжевый → красный.
Color getHeatmapColor(double v, double min, double max) {
  if (max <= min) return const Color(0xFF4CAF50);
  final t = ((v - min) / (max - min)).clamp(0.0, 1.0);
  if (t < 0.33) {
    return Color.lerp(const Color(0xFF4CAF50), const Color(0xFFFFEB3B), t / 0.33)!;
  } else if (t < 0.66) {
    return Color.lerp(const Color(0xFFFFEB3B), const Color(0xFFFF9800), (t - 0.33) / 0.33)!;
  } else {
    return Color.lerp(const Color(0xFFFF9800), const Color(0xFFF44336), (t - 0.66) / 0.34)!;
  }
}

class MapLabPage extends StatefulWidget {
  const MapLabPage({super.key});

  @override
  State<MapLabPage> createState() => _MapLabPageState();
}

class _MapLabPageState extends State<MapLabPage> {
  static const _defKeys = ['A2TB100B', 'A2TB100K', '32BITBASE'];
  /// Высота окна карты (крупнее, чем «сжатый» Expanded).
  static const double _mapHeight = 480;

  final _journal = <String>[];
  DefSet? _defs;
  Map<String, MapGrid>? _maps;
  LogData? _log;
  Map<String, MapResult>? _results;
  String? _mapName;
  int? _selRi, _selCi;
  bool _view3D = false;
  String _defsSource = '—';

  @override
  void initState() {
    super.initState();
    WidgetsBinding.instance.addPostFrameCallback((_) => _bootstrapDefs());
  }

  void _say(String s) {
    if (!mounted) return;
    setState(() => _journal.add('${TimeOfDay.now().format(context)}  $s'));
  }

  Future<Directory> _cacheDir() async {
    final docs = await getApplicationDocumentsDirectory();
    final dir = Directory('${docs.path}/defs');
    if (!dir.existsSync()) dir.createSync(recursive: true);
    return dir;
  }

  Future<Map<String, String>> _tryAssets() async {
    final out = <String, String>{};
    for (final k in _defKeys) {
      try {
        final s = await rootBundle.loadString('assets/defs/$k.xml');
        if (s.trim().length > 200) out[k] = s;
      } catch (_) {}
    }
    return out;
  }

  Future<Map<String, String>> _tryCache() async {
    final out = <String, String>{};
    final dir = await _cacheDir();
    for (final k in _defKeys) {
      final f = File('${dir.path}/$k.xml');
      if (f.existsSync() && f.lengthSync() > 200) {
        try {
          out[k] = await f.readAsString();
        } catch (_) {}
      }
    }
    return out;
  }

  Future<void> _saveCache(Map<String, String> xmlById) async {
    final dir = await _cacheDir();
    for (final e in xmlById.entries) {
      try {
        await File('${dir.path}/${e.key}.xml').writeAsString(e.value, flush: true);
      } catch (_) {}
    }
  }

  bool _applyDefs(Map<String, String> xmlById, String source) {
    if (xmlById.length < 3) return false;
    try {
      _defs = DefSet.build(xmlById, 'A2TB100B');
      _defsSource = source;
      _say('дефиниции [$source]: ${_defs!.chain.join(' → ')} · '
          'ecuid ${_defs!.meta.ecuid} · ${_defs!.tables.length} таблиц');
      return true;
    } catch (e) {
      _say('!! ошибка разбора дефиниций: $e');
      return false;
    }
  }

  Future<void> _bootstrapDefs() async {
    var xml = await _tryAssets();
    if (_applyDefs(xml, 'APK assets')) {
      await _saveCache(xml);
      setState(() {});
      return;
    }
    xml = await _tryCache();
    if (_applyDefs(xml, 'кэш телефона')) {
      setState(() {});
      return;
    }
    _say('!! дефиниций нет. Нажмите «0 · Дефиниции» и укажите папку с XML.');
    setState(() {});
  }

  Future<Map<String, String>> _scanDefsInDir(Directory root) async {
    final found = <String, File>{};
    bool matchName(String name, String key) {
      final n = name.toLowerCase();
      return n == '${key.toLowerCase()}.xml' || n.contains(key.toLowerCase());
    }

    void consider(File f) {
      final name = f.uri.pathSegments.last;
      for (final k in _defKeys) {
        if (found.containsKey(k)) continue;
        if (matchName(name, k) && f.lengthSync() > 200) found[k] = f;
      }
    }

    try {
      for (final e in root.listSync(followLinks: false)) {
        if (e is File && e.path.toLowerCase().endsWith('.xml')) consider(e);
      }
      for (final e in root.listSync(followLinks: false)) {
        if (e is! Directory) continue;
        try {
          for (final f in e.listSync(followLinks: false)) {
            if (f is File && f.path.toLowerCase().endsWith('.xml')) consider(f);
          }
        } catch (_) {}
      }
    } catch (_) {}

    final out = <String, String>{};
    for (final e in found.entries) {
      try {
        out[e.key] = await e.value.readAsString();
      } catch (_) {}
    }
    return out;
  }

  Future<void> _pickDefs() async {
    final result = await showModalBottomSheet<Object>(
      context: context,
      isScrollControlled: true,
      backgroundColor: Colors.transparent,
      builder: (ctx) => const _FilePickerSheet(
        extensions: ['.xml'],
        title: 'Дефиниции RomRaider (.xml)',
        allowPickFolderDefs: true,
      ),
    );
    if (result == null) return;

    var xml = <String, String>{};
    var source = 'локально';

    if (result is Directory) {
      xml = await _scanDefsInDir(result);
      source = 'папка ${result.path.split('/').last}';
      _say('сканирование ${result.path}: найдено ${xml.length}/3');
    } else if (result is File) {
      xml = await _scanDefsInDir(result.parent);
      if (xml.length < 3) {
        final name = result.uri.pathSegments.last.toUpperCase();
        for (final k in _defKeys) {
          if (name.contains(k)) {
            try {
              xml[k] = await result.readAsString();
            } catch (_) {}
          }
        }
      }
      source = 'файл+папка';
    } else if (result is Map) {
      xml = Map<String, String>.from(result);
    }

    if (xml.length < 3) {
      final miss = _defKeys.where((k) => !xml.containsKey(k)).join(', ');
      _say('!! не хватает XML: $miss');
      return;
    }

    if (_applyDefs(xml, source)) {
      await _saveCache(xml);
      setState(() {});
    }
  }

  Future<File?> _chooseFile(List<String> exts, String title) async {
    final r = await showModalBottomSheet<Object>(
      context: context,
      isScrollControlled: true,
      backgroundColor: Colors.transparent,
      builder: (ctx) => _FilePickerSheet(extensions: exts, title: title),
    );
    return r is File ? r : null;
  }

  Future<void> _pickRom() async {
    if (_defs == null) {
      _say('сначала загрузите дефиниции');
      return;
    }
    final f = await _chooseFile(['.bin', '.hex', '.rom'], 'Выберите прошивку (.bin)');
    if (f == null) return;
    try {
      final bytes = await f.readAsBytes();
      setState(() {
        _results = null;
        _selRi = _selCi = null;
      });
      final parser = RomParser(bytes);
      final idAddr = int.tryParse(_defs?.meta.internalIdAddress ?? '2000', radix: 16) ?? 0x2000;
      final romId = parser.readRomId(idAddr);
      final expected = _defs?.meta.internalIdString ?? '';
      final matched = expected.isEmpty || romId == expected;
      _say('ROM: ${f.uri.pathSegments.last} · ${(bytes.length / 1024).toStringAsFixed(0)} КБ · '
          'ID="$romId"${matched ? '' : '  (ожидался $expected)'}');
      _maps = parser.extractKeys(_defs!);
      _mapName = _maps!.keys.firstOrNull;
      _say('карт извлечено: ${_maps!.length} из ${RomParser.keyTables.length}');
      setState(() {});
    } catch (e) {
      _say('!! ошибка парсинга ROM: $e');
    }
  }

  Future<void> _pickLog() async {
    final f = await _chooseFile(['.csv', '.txt', '.log'], 'Выберите файл лога (.csv)');
    if (f == null) return;
    try {
      final text = utf8.decode(await f.readAsBytes(), allowMalformed: true);
      _log = LogData.parse(text);
      final health = LogAudit.check(_log!);
      _say('лог: ${f.uri.pathSegments.last} · ${_log!.rows} строк');
      for (final n in health.notes) {
        _say('   $n');
      }
      setState(() {});
    } catch (e) {
      _say('!! ошибка чтения лога: $e');
    }
  }

  void _runAnalysis() {
    if (_maps == null || _log == null) return;
    const cfg = AnalyzerConfig();
    _results = Analyzer(cfg).run(_maps!, _log!);
    var dec = 0, inc = 0;
    for (final r in _results!.values) {
      dec += r.dec;
      inc += r.inc;
    }
    _say('анализ: вердикты по ${_results!.length} картам · убавить $dec, прибавить $inc ячеек');
    setState(() {});
  }

  Future<void> _export() async {
    if (_maps == null || _results == null) return;
    final dir = await getApplicationDocumentsDirectory();
    final stamp = DateTime.now().toIso8601String().replaceAll(':', '-').split('.').first;
    final outDir = Directory('${dir.path}/maplab_$stamp')..createSync(recursive: true);
    final files = <XFile>[];
    final md = StringBuffer('# Map Lab A2TB100B · рекомендации\n');
    for (final e in _maps!.entries) {
      final g = e.value;
      final res = _results![e.key];
      if (res == null) continue;
      final head = '\t${g.x.values.map((v) => v.toStringAsFixed(2)).join('\t')}';
      final rows = <String>[
        for (var ri = 0; ri < g.rows; ri++)
          '${g.y.values[ri].toStringAsFixed(2)}\t${[
            for (var ci = 0; ci < g.cols; ci++)
              (g.data[ri][ci] + res.delta[ri][ci]).toStringAsFixed(3),
          ].join('\t')}',
      ];
      final tsv = '$head\n${rows.join('\n')}\n';
      final file = File('${outDir.path}/${e.key.replaceAll(RegExp(r'[/ ]'), '_')}_recommended.tsv')
        ..writeAsStringSync(tsv);
      files.add(XFile(file.path));
      md.writeln('\n## ${e.key} (${g.units})');
      for (var ri = 0; ri < g.rows; ri++) {
        for (var ci = 0; ci < g.cols; ci++) {
          final d = res.delta[ri][ci];
          if (d == 0) continue;
          final inf = res.info[ri][ci];
          md.writeln('- ${g.x.values[ci]} × ${g.y.values[ri]}: ${g.data[ri][ci]} → '
              '${(g.data[ri][ci] + d).toStringAsFixed(2)} (${d > 0 ? '+' : ''}$d)'
              '${inf != null && inf.why.isNotEmpty ? ' — ${inf.why}' : ''}');
        }
      }
    }
    final mf = File('${outDir.path}/recommendations.md')..writeAsStringSync(md.toString());
    files.insert(0, XFile(mf.path));
    _say('экспорт: ${files.length} файлов → ${outDir.path}');
    await Share.shareXFiles(files, text: 'Map Lab A2TB100B — рекомендации');
  }

  @override
  Widget build(BuildContext context) {
    final theme = Theme.of(context);
    final map = _maps == null ? null : _maps![_mapName];
    MapResult? res;
    if (map != null) res = _results?[map.name];

    // Весь экран снова в одном ListView — журнал всегда можно долистать.
    // Карта фиксированной большой высоты; 3D крутится через EagerPan внутри map3d_view.
    return Scaffold(
      appBar: AppBar(
        title: const Text('Map Lab · A2TB100B'),
        actions: [
          if (_results != null)
            IconButton(icon: const Icon(Icons.ios_share), onPressed: _export, tooltip: 'Экспорт'),
        ],
      ),
      body: ListView(
        padding: const EdgeInsets.fromLTRB(12, 10, 12, 24),
        children: [
          Wrap(
            spacing: 8,
            runSpacing: 8,
            children: [
              _stepBtn('0 · Дефиниции', Icons.folder_special, _pickDefs, ok: _defs != null),
              _stepBtn('1 · Прошивка', Icons.memory, _pickRom, ok: _maps != null),
              _stepBtn('2 · Лог', Icons.description, _pickLog, ok: _log != null),
              _stepBtn('3 · Анализ', Icons.psychology,
                  (_maps != null && _log != null) ? _runAnalysis : null,
                  ok: _results != null),
              _stepBtn('Экспорт', Icons.ios_share, _results != null ? _export : null),
            ],
          ),
          const SizedBox(height: 6),
          Text(
            _defs != null
                ? 'XML: $_defsSource · ${_defs!.meta.xmlid}'
                : 'Нужны A2TB100B.xml, A2TB100K.xml, 32BITBASE.xml',
            style: theme.textTheme.bodySmall?.copyWith(
              color: _defs != null ? Colors.greenAccent : Colors.amber,
            ),
          ),
          if (_maps != null) ...[
            const SizedBox(height: 10),
            SizedBox(
              height: 40,
              child: ListView(
                scrollDirection: Axis.horizontal,
                children: [
                  for (final e in _maps!.entries)
                    Padding(
                      padding: const EdgeInsets.only(right: 8),
                      child: ChoiceChip(
                        selected: _mapName == e.key,
                        onSelected: (_) => setState(() {
                          _mapName = e.key;
                          _selRi = _selCi = null;
                        }),
                        label: Text(e.key, style: const TextStyle(fontSize: 11)),
                      ),
                    ),
                ],
              ),
            ),
            const SizedBox(height: 6),
            Row(
              children: [
                Expanded(
                  child: Text(
                    map == null ? '' : '${map.name} · ${map.rows}×${map.cols}',
                    style: theme.textTheme.bodySmall,
                    overflow: TextOverflow.ellipsis,
                  ),
                ),
                SegmentedButton<bool>(
                  segments: const [
                    ButtonSegment(value: false, icon: Icon(Icons.table_chart, size: 16), label: Text('2D')),
                    ButtonSegment(value: true, icon: Icon(Icons.threed_rotation, size: 16), label: Text('3D')),
                  ],
                  selected: {_view3D},
                  onSelectionChanged: (s) => setState(() => _view3D = s.first),
                ),
              ],
            ),
          ],
          if (map != null) ...[
            const SizedBox(height: 8),
            SizedBox(
              height: _mapHeight,
              child: Card(
                clipBehavior: Clip.antiAlias,
                child: _view3D
                    ? Map3DView(
                        grid: map,
                        result: res,
                        selRi: _selRi,
                        selCi: _selCi,
                        onCell: (ri, ci) => setState(() {
                          _selRi = ri;
                          _selCi = ci;
                        }),
                      )
                    : _Table2D(
                        grid: map,
                        result: res,
                        selRi: _selRi,
                        selCi: _selCi,
                        onCell: (ri, ci) => setState(() {
                          _selRi = ri;
                          _selCi = ci;
                        }),
                      ),
              ),
            ),
            if (_selRi != null && _selCi != null) ...[
              const SizedBox(height: 8),
              _inspector(map, res),
            ],
            const SizedBox(height: 8),
            Wrap(spacing: 12, runSpacing: 4, children: [
              _legend(const Color(0xFF4CAF50), 'низ'),
              _legend(const Color(0xFFFFEB3B), 'середина'),
              _legend(const Color(0xFFF44336), 'верх'),
              _legend(Colors.lightBlueAccent, 'правка'),
            ]),
          ],
          const SizedBox(height: 16),
          Text('Журнал', style: theme.textTheme.titleSmall),
          const SizedBox(height: 6),
          Container(
            width: double.infinity,
            constraints: const BoxConstraints(minHeight: 180),
            padding: const EdgeInsets.all(10),
            decoration: BoxDecoration(
              color: theme.colorScheme.surfaceContainerHighest.withValues(alpha: 0.35),
              borderRadius: BorderRadius.circular(8),
            ),
            child: _journal.isEmpty
                ? const Text('— пока пусто —', style: TextStyle(color: Colors.white38, fontSize: 12))
                : Column(
                    crossAxisAlignment: CrossAxisAlignment.start,
                    children: [
                      for (final j in _journal)
                        Padding(
                          padding: const EdgeInsets.only(bottom: 2),
                          child: Text(j, style: const TextStyle(fontFamily: 'monospace', fontSize: 11)),
                        ),
                    ],
                  ),
          ),
          const SizedBox(height: 12),
          const Text(
            'Листайте экран вниз, чтобы читать журнал. '
            'На 3D-карте тяните пальцем — страница при этом не уезжает.',
            style: TextStyle(fontSize: 11, color: Colors.white38),
          ),
        ],
      ),
    );
  }

  Widget _stepBtn(String label, IconData icon, VoidCallback? onTap, {bool ok = false}) {
    return FilledButton.tonalIcon(
      onPressed: onTap,
      icon: Icon(ok ? Icons.check_circle : icon, size: 18),
      label: Text(label, style: const TextStyle(fontSize: 12)),
    );
  }

  Widget _legend(Color c, String t) => Row(mainAxisSize: MainAxisSize.min, children: [
        Container(width: 12, height: 8, decoration: BoxDecoration(color: c, borderRadius: BorderRadius.circular(2))),
        const SizedBox(width: 6),
        Text(t, style: const TextStyle(fontSize: 10.5)),
      ]);

  Widget _inspector(MapGrid g, MapResult? res) {
    final ri = _selRi!, ci = _selCi!;
    final v = g.data[ri][ci];
    final d = res?.delta[ri][ci] ?? 0;
    final inf = res?.info[ri][ci];
    final col = d < 0 ? const Color(0xFFFF5C5C) : d > 0 ? const Color(0xFFA8FF3E) : null;
    return Card(
      child: Padding(
        padding: const EdgeInsets.all(10),
        child: Column(crossAxisAlignment: CrossAxisAlignment.start, children: [
          Text('Ячейка: ${g.x.name}=${g.x.values[ci]} · ${g.y.name}=${g.y.values[ri]}',
              style: const TextStyle(fontFamily: 'monospace', fontSize: 12)),
          const SizedBox(height: 4),
          Row(children: [
            Text(v.toStringAsFixed(2),
                style: const TextStyle(
                    fontSize: 18, fontFamily: 'monospace', decoration: TextDecoration.lineThrough)),
            const SizedBox(width: 8),
            const Icon(Icons.arrow_forward, size: 16),
            const SizedBox(width: 8),
            Text('${(v + d).toStringAsFixed(2)} ${g.units}',
                style: TextStyle(fontSize: 22, fontFamily: 'monospace', color: col)),
          ]),
          if (inf != null && inf.why.isNotEmpty)
            Padding(
              padding: const EdgeInsets.only(top: 4),
              child: Text(inf.why, style: const TextStyle(fontSize: 12)),
            ),
        ]),
      ),
    );
  }
}

class _FilePickerSheet extends StatefulWidget {
  const _FilePickerSheet({
    required this.extensions,
    required this.title,
    this.allowPickFolderDefs = false,
  });
  final List<String> extensions;
  final String title;
  final bool allowPickFolderDefs;

  @override
  State<_FilePickerSheet> createState() => _FilePickerSheetState();
}

class _FilePickerSheetState extends State<_FilePickerSheet> {
  Directory _currentDir = Directory('/storage/emulated/0');
  bool _hasPermission = false;
  List<FileSystemEntity> _items = [];

  @override
  void initState() {
    super.initState();
    _checkPermissionAndRefresh();
  }

  Future<void> _checkPermissionAndRefresh() async {
    var ok = false;
    try {
      ok = await Permission.manageExternalStorage.isGranted;
    } catch (_) {}
    if (!ok) {
      final docs = await getApplicationDocumentsDirectory();
      _currentDir = docs;
    } else if (!_currentDir.existsSync()) {
      _currentDir = Directory('/storage/emulated/0');
    }
    setState(() => _hasPermission = ok);
    _refreshList();
  }

  Future<void> _requestPermission() async {
    try {
      await Permission.manageExternalStorage.request();
    } catch (_) {}
    try {
      await openAppSettings();
    } catch (_) {}
    _checkPermissionAndRefresh();
  }

  void _refreshList() {
    try {
      if (!_currentDir.existsSync()) {
        setState(() => _items = []);
        return;
      }
      final list = _currentDir.listSync(followLinks: false);
      final dirs = <Directory>[];
      final files = <File>[];
      for (final e in list) {
        final name = e.path.split('/').last;
        if (name.startsWith('.')) continue;
        if (e is Directory) {
          dirs.add(e);
        } else if (e is File) {
          if (widget.extensions.any((x) => name.toLowerCase().endsWith(x))) files.add(e);
        }
      }
      dirs.sort((a, b) => a.path.toLowerCase().compareTo(b.path.toLowerCase()));
      files.sort((a, b) => b.lastModifiedSync().compareTo(a.lastModifiedSync()));
      setState(() => _items = [...dirs, ...files]);
    } catch (_) {
      setState(() => _items = []);
    }
  }

  void _goUp() {
    final parent = _currentDir.parent;
    if (parent.path.length >= 4) {
      setState(() => _currentDir = parent);
      _refreshList();
    }
  }

  void _goTo(String path) {
    final d = Directory(path);
    if (d.existsSync()) {
      setState(() => _currentDir = d);
      _refreshList();
    }
  }

  @override
  Widget build(BuildContext context) {
    return Container(
      height: MediaQuery.of(context).size.height * 0.85,
      decoration: const BoxDecoration(
        color: Color(0xFF141921),
        borderRadius: BorderRadius.vertical(top: Radius.circular(16)),
      ),
      child: Column(
        children: [
          Padding(
            padding: const EdgeInsets.symmetric(horizontal: 16, vertical: 12),
            child: Row(
              children: [
                Expanded(
                  child: Text(widget.title, style: const TextStyle(fontWeight: FontWeight.bold, fontSize: 16)),
                ),
                IconButton(icon: const Icon(Icons.close), onPressed: () => Navigator.of(context).pop()),
              ],
            ),
          ),
          if (!_hasPermission)
            Padding(
              padding: const EdgeInsets.symmetric(horizontal: 12),
              child: TextButton(onPressed: _requestPermission, child: const Text('РАЗРЕШИТЬ доступ к файлам')),
            ),
          SingleChildScrollView(
            scrollDirection: Axis.horizontal,
            padding: const EdgeInsets.symmetric(horizontal: 12, vertical: 4),
            child: Row(children: [
              _chip('Память', '/storage/emulated/0'),
              _chip('Downloads', '/storage/emulated/0/Download'),
              _chip('Telegram', '/storage/emulated/0/Telegram'),
              _chip('Documents', '/storage/emulated/0/Documents'),
            ]),
          ),
          Container(
            padding: const EdgeInsets.symmetric(horizontal: 8, vertical: 4),
            color: Colors.black26,
            child: Row(
              children: [
                IconButton(icon: const Icon(Icons.arrow_upward, size: 18), onPressed: _goUp),
                Expanded(
                  child: Text(_currentDir.path,
                      style: const TextStyle(fontFamily: 'monospace', fontSize: 11, color: Colors.cyanAccent),
                      overflow: TextOverflow.ellipsis),
                ),
                if (widget.allowPickFolderDefs)
                  FilledButton.tonal(
                    onPressed: () => Navigator.of(context).pop(_currentDir),
                    child: const Text('Взять XML из ЭТОЙ папки', style: TextStyle(fontSize: 11)),
                  ),
              ],
            ),
          ),
          const Divider(height: 1),
          Expanded(
            child: ListView.builder(
              itemCount: _items.length,
              itemBuilder: (ctx, i) {
                final item = _items[i];
                final name = item.path.split('/').last;
                if (item is Directory) {
                  return ListTile(
                    dense: true,
                    leading: const Icon(Icons.folder, color: Colors.amber),
                    title: Text(name),
                    onTap: () {
                      setState(() => _currentDir = item);
                      _refreshList();
                    },
                  );
                } else if (item is File) {
                  return ListTile(
                    dense: true,
                    leading: const Icon(Icons.insert_drive_file, color: Colors.lightBlueAccent),
                    title: Text(name, style: const TextStyle(fontFamily: 'monospace', fontSize: 13)),
                    onTap: () => Navigator.of(ctx).pop(item),
                  );
                }
                return const SizedBox.shrink();
              },
            ),
          ),
        ],
      ),
    );
  }

  Widget _chip(String label, String path) {
    return Padding(
      padding: const EdgeInsets.only(right: 6),
      child: ActionChip(
        visualDensity: VisualDensity.compact,
        label: Text(label, style: const TextStyle(fontSize: 11)),
        onPressed: () => _goTo(path),
      ),
    );
  }
}

/// 2D таблица с pinch-zoom и кнопками +/−.
class _Table2D extends StatefulWidget {
  const _Table2D({required this.grid, this.result, this.selRi, this.selCi, this.onCell});
  final MapGrid grid;
  final MapResult? result;
  final int? selRi, selCi;
  final void Function(int, int)? onCell;

  @override
  State<_Table2D> createState() => _Table2DState();
}

class _Table2DState extends State<_Table2D> {
  final _tc = TransformationController();
  static const _minS = 0.25;
  static const _maxS = 6.0;

  @override
  void dispose() {
    _tc.dispose();
    super.dispose();
  }

  void _zoomBy(double factor) {
    final m = _tc.value.clone();
    final s = m.getMaxScaleOnAxis();
    final next = (s * factor).clamp(_minS, _maxS);
    final f = next / s;
    final child = context.findRenderObject() as RenderBox?;
    if (child == null) {
      _tc.value = m.scaled(f);
      return;
    }
    final center = child.size.center(Offset.zero);
    final scene = _tc.toScene(center);
    m.translate(scene.dx, scene.dy);
    m.scale(f);
    m.translate(-scene.dx, -scene.dy);
    _tc.value = m;
    setState(() {});
  }

  void _reset() {
    _tc.value = Matrix4.identity();
    setState(() {});
  }

  @override
  Widget build(BuildContext context) {
    final g = widget.grid;
    return Stack(
      children: [
        InteractiveViewer(
          transformationController: _tc,
          minScale: _minS,
          maxScale: _maxS,
          boundaryMargin: const EdgeInsets.all(200),
          constrained: false,
          panEnabled: true,
          scaleEnabled: true,
          child: Padding(
            padding: const EdgeInsets.all(8),
            child: Table(
              defaultColumnWidth: const IntrinsicColumnWidth(),
              children: [
                TableRow(children: [
                  _head('${g.y.guess}↓ ${g.x.guess}→'),
                  for (final v in g.x.values) _head(v.toStringAsFixed(0), accent: true),
                ]),
                for (var ri = 0; ri < g.rows; ri++)
                  TableRow(children: [
                    _head(g.y.values[ri].toStringAsFixed(2), accent: true),
                    for (var ci = 0; ci < g.cols; ci++) _cell(ri, ci),
                  ]),
              ],
            ),
          ),
        ),
        Positioned(
          right: 8,
          bottom: 8,
          child: Column(
            children: [
              _zBtn(Icons.add, () => _zoomBy(1.25)),
              const SizedBox(height: 6),
              _zBtn(Icons.remove, () => _zoomBy(1 / 1.25)),
              const SizedBox(height: 6),
              _zBtn(Icons.center_focus_strong, _reset),
            ],
          ),
        ),
        const Positioned(
          left: 8,
          bottom: 8,
          child: Text('pinch / кнопки = зум', style: TextStyle(fontSize: 10, color: Colors.white38)),
        ),
      ],
    );
  }

  Widget _zBtn(IconData icon, VoidCallback onTap) {
    return Material(
      color: Colors.black54,
      shape: const CircleBorder(),
      child: InkWell(
        customBorder: const CircleBorder(),
        onTap: onTap,
        child: Padding(
          padding: const EdgeInsets.all(8),
          child: Icon(icon, size: 18, color: Colors.white),
        ),
      ),
    );
  }

  Widget _head(String t, {bool accent = false}) => Container(
        padding: const EdgeInsets.symmetric(horizontal: 6, vertical: 4),
        decoration: BoxDecoration(border: Border.all(color: Colors.white12)),
        child: Text(t,
            style: TextStyle(
                fontSize: 9.5,
                fontFamily: 'monospace',
                color: accent ? const Color(0xCC4BE1FF) : Colors.white54)),
      );

  Widget _cell(int ri, int ci) {
    final g = widget.grid;
    final v = g.data[ri][ci];
    final d = widget.result?.delta[ri][ci] ?? 0;
    final sel = widget.selRi == ri && widget.selCi == ci;
    final heat = getHeatmapColor(v, g.vmin, g.vmax).withValues(alpha: 0.55);
    final hasFix = d != 0;

    return InkWell(
      onTap: () => widget.onCell?.call(ri, ci),
      child: Container(
        constraints: const BoxConstraints(minWidth: 48, minHeight: 40),
        padding: const EdgeInsets.symmetric(horizontal: 4, vertical: 3),
        decoration: BoxDecoration(
          color: heat,
          border: Border.all(
            color: sel
                ? Colors.white
                : hasFix
                    ? Colors.lightBlueAccent
                    : Colors.white12,
            width: sel ? 1.6 : (hasFix ? 1.2 : 0.5),
          ),
        ),
        child: Column(
          mainAxisAlignment: MainAxisAlignment.center,
          children: [
            Text(v.toStringAsFixed(2),
                style: const TextStyle(fontSize: 10, fontFamily: 'monospace', color: Colors.white)),
            if (hasFix)
              Text((v + d).toStringAsFixed(2),
                  style: const TextStyle(
                      fontSize: 11,
                      fontWeight: FontWeight.bold,
                      fontFamily: 'monospace',
                      color: Colors.lightBlueAccent)),
          ],
        ),
      ),
    );
  }
}
'''

dest = APP / "lib/maps_lab/maps_lab_page.dart"
dest.write_text(PAGE.replace("__PKG__", PKG) + "\n", encoding="utf-8")
print("[ok] maps_lab_page.dart: карта 480px + ListView (журнал долистывается)")
print("Дальше: ячейка 3/3 (сборка APK).")

[ok] maps_lab_page.dart: карта 480px + ListView (журнал долистывается)
Дальше: ячейка 3/3 (сборка APK).


In [5]:
# @title Быстрый автофикс неиспользуемых импортов
from pathlib import Path

APP = Path("/content/subaru_ssm2_fixed")

# 1. lib/analyzer.dart
analyzer_f = APP / "lib/analyzer.dart"
if analyzer_f.exists():
    text = analyzer_f.read_text(encoding="utf-8")
    text = text.replace("import 'pids.dart';\n", "").replace("import 'protocol.dart';\n", "")
    analyzer_f.write_text(text, encoding="utf-8")

# 2. lib/model.dart
model_f = APP / "lib/model.dart"
if model_f.exists():
    text = model_f.read_text(encoding="utf-8")
    text = text.replace("import 'identity.dart';\n", "")
    model_f.write_text(text, encoding="utf-8")

# 3. test/protocol_test.dart
proto_t = APP / "test/protocol_test.dart"
if proto_t.exists():
    text = proto_t.read_text(encoding="utf-8")
    for imp in ["analyzer.dart", "derived.dart", "elm.dart", "engine.dart", "identity.dart"]:
        text = text.replace(f"import 'package:subaru_ssm2/{imp}';\n", "")
    proto_t.write_text(text, encoding="utf-8")

print("[OK] Все неиспользуемые импорты вычищены!")

[OK] Все неиспользуемые импорты вычищены!


In [6]:
# @title 3/3 | SSM2 0.7 FIXED - Полная очистка, анализ, тесты и автоскачивание APK { display-mode: "form" }

import hashlib
import json
import os
from pathlib import Path
import re
import shutil
import subprocess
import sys
import time
import urllib.request

BUILD_VARIANT = "release"  # @param ["release", "debug", "profile"]
TRANSPORT = "from_cell_2"  # @param ["from_cell_2", "bluetooth_classic", "flutter_bluetooth_serial"]
CLEAN_BEFORE = True  # @param {type:"boolean"}

# --- 0. Полная предварительная очистка старых сборок ---
print("=== 0. Полная очистка предыдущих сборок ===")
content_dir = Path("/content")
for old_apk in content_dir.glob("ssm2_fixed_*.apk"):
    try:
        old_apk.unlink()
        print(f"[Удален старый APK] {old_apk.name}")
    except Exception as e:
        print(f"[Ошибка удаления {old_apk.name}]: {e}")

for old_item in content_dir.glob("ssm2_build_*"):
    try:
        if old_item.is_dir():
            shutil.rmtree(old_item, ignore_errors=True)
        else:
            old_item.unlink(missing_ok=True)
        print(f"[Удален старый отчёт] {old_item.name}")
    except Exception as e:
        print(f"[Ошибка очистки {old_item.name}]: {e}")

APP = Path("/content/subaru_ssm2_fixed")
CONFIG = APP / "build_config.json"
if not CONFIG.exists():
    raise RuntimeError("Не найден новый проект. Выполните ячейки 1 и 2.")
cfg = json.loads(CONFIG.read_text(encoding="utf-8"))
FLUTTER = Path(cfg["flutter"])
SDK = Path(cfg["sdk"])
JAVA = Path(cfg["java"])
bt = cfg["bt_package"] if TRANSPORT == "from_cell_2" else TRANSPORT
os.environ.update(JAVA_HOME=str(JAVA), ANDROID_HOME=str(SDK), ANDROID_SDK_ROOT=str(SDK))
os.environ["PATH"] = os.pathsep.join([str(JAVA / "bin"), str(FLUTTER / "bin"),
                                    str(SDK / "cmdline-tools/latest/bin"), os.environ.get("PATH", "")])
stamp = time.strftime("%Y%m%d_%H%M%S")
REPORT = Path(f"/content/ssm2_build_{stamp}")
REPORT.mkdir(parents=True, exist_ok=True)
report = {"revision": "0.7", "transport": bt, "variant": BUILD_VARIANT,
          "flutter": cfg["flutter_version"], "stages": {}, "hardware_tested": False}


def save_report():
    (REPORT / "report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")


def stage(name, args, timeout=3600):
    logfile = REPORT / f"{name}.log"
    print(f"\n=== {name} ===\n> {' '.join(map(str, args))}\nЖурнал: {logfile}")
    started = time.monotonic()
    try:
        with logfile.open("w", encoding="utf-8") as out:
            p = subprocess.run(list(map(str, args)), cwd=APP, text=True,
                               stdout=out, stderr=subprocess.STDOUT, timeout=timeout)
        text = logfile.read_text(encoding="utf-8", errors="replace")
        print(text[-7000:])
        report["stages"][name] = {"exit_code": p.returncode, "seconds": round(time.monotonic() - started, 1)}
        save_report()
        if p.returncode:
            raise RuntimeError(f"Этап {name} не прошел. Полный журнал: {logfile}")
        return text
    except subprocess.TimeoutExpired:
        report["stages"][name] = {"error": "timeout"}
        save_report()
        raise RuntimeError(f"Таймаут этапа {name}. Журнал: {logfile}")


print("SSM2 0.7 | Проверка перед сборкой")
required = [FLUTTER / "bin/flutter", FLUTTER / "bin/dart", JAVA / "bin/java",
            SDK / "platforms/android-36/android.jar", SDK / "build-tools/35.0.0/aapt",
            SDK / "build-tools/35.0.0/apksigner", SDK / "ndk/27.0.12077973/source.properties",
            APP / "transport_templates/classic.dart.txt", APP / "transport_templates/serial.dart.txt",
            APP / "test/protocol_test.dart", APP / "tool/prepare_bt.py",
            APP / "lib/derived.dart", APP / "lib/analyzer.dart",
            APP / "lib/identity.dart", APP / "lib/maplab_link.dart", APP / "lib/pids.dart"]
for file in required:
    if not file.exists(): raise RuntimeError(f"Отсутствует: {file}. Повторите соответствующую ячейку.")
    print("[OK]", file)

version_log = stage("flutter_version", [FLUTTER / "bin/flutter", "--version", "--machine"], timeout=600)
actual_flutter = json.JSONDecoder().raw_decode(version_log[version_log.index("{"):])[0]
report["flutter"] = actual_flutter["frameworkVersion"]
save_report()

properties_file = APP / "android/local.properties"
properties = {}
if properties_file.exists():
    for line in properties_file.read_text(encoding="utf-8").splitlines():
        if "=" in line and not line.lstrip().startswith("#"):
            key, value = line.split("=", 1)
            properties[key.strip()] = value
properties.update({"sdk.dir": str(SDK), "flutter.sdk": str(FLUTTER)})
properties_file.write_text("\n".join(f"{key}={value}" for key, value in properties.items()) + "\n", encoding="utf-8")
gradlew = APP / "android/gradlew"
gradlew.chmod(gradlew.stat().st_mode | 0o111)

stage("prepare_bt", [sys.executable, APP / "tool/prepare_bt.py", bt], timeout=600)
cfg["bt_package"] = bt
CONFIG.write_text(json.dumps(cfg, indent=2), encoding="utf-8")

if CLEAN_BEFORE:
    stage("clean", [FLUTTER / "bin/flutter", "clean"], timeout=600)

stage("pub_get", [FLUTTER / "bin/flutter", "pub", "get"], timeout=1200)
stage("format", [FLUTTER / "bin/dart", "format", "lib", "test"], timeout=600)

# Флаг --no-fatal-warnings предотвратит сбой при предупреждениях синтаксиса
stage("analyze", [FLUTTER / "bin/flutter", "analyze", "--no-pub", "--no-fatal-infos", "--no-fatal-warnings"], timeout=1200)
stage("tests", [FLUTTER / "bin/flutter", "test", "--no-pub", "--reporter", "expanded"], timeout=1200)

apk = APP / f"build/app/outputs/flutter-apk/app-{BUILD_VARIANT}.apk"
apk.unlink(missing_ok=True)
stage("build", [FLUTTER / "bin/flutter", "build", "apk", f"--{BUILD_VARIANT}", "--no-pub"], timeout=4800)
if not apk.exists() or apk.stat().st_size < 1024 * 1024:
    raise RuntimeError(f"Новый APK не найден: {apk}")

stage("apk_signature", [SDK / "build-tools/35.0.0/apksigner", "verify", "--verbose", apk], timeout=120)
digest = hashlib.sha256(apk.read_bytes()).hexdigest()
short = "A" if bt == "bluetooth_classic" else "B"
destination = Path(f"/content/ssm2_fixed_{short}_{BUILD_VARIANT}.apk")
shutil.copy2(apk, destination)
report.update({"apk": str(destination), "sha256": digest, "apk_verified": True})
save_report()

print(f"\n[OK] Проверки и сборка успешно завершены: {destination}")
print(f"SHA256: {digest}")

# --- Автоматическое скачивание готового APK ---
try:
    from google.colab import files
    print(f"\n[АВТО-СКАЧИВАНИЕ] Файл {destination.name} отправлен на ваш ПК/телефон...")
    files.download(str(destination))
except Exception as e:
    print(f"\nСкачайте файл из боковой панели 'Файлы' Colab: {destination.name}")

=== 0. Полная очистка предыдущих сборок ===
SSM2 0.7 | Проверка перед сборкой
[OK] /content/flutter/bin/flutter
[OK] /content/flutter/bin/dart
[OK] /usr/lib/jvm/java-17-openjdk-amd64/bin/java
[OK] /content/android-sdk/platforms/android-36/android.jar
[OK] /content/android-sdk/build-tools/35.0.0/aapt
[OK] /content/android-sdk/build-tools/35.0.0/apksigner
[OK] /content/android-sdk/ndk/27.0.12077973/source.properties
[OK] /content/subaru_ssm2_fixed/transport_templates/classic.dart.txt
[OK] /content/subaru_ssm2_fixed/transport_templates/serial.dart.txt
[OK] /content/subaru_ssm2_fixed/test/protocol_test.dart
[OK] /content/subaru_ssm2_fixed/tool/prepare_bt.py
[OK] /content/subaru_ssm2_fixed/lib/derived.dart
[OK] /content/subaru_ssm2_fixed/lib/analyzer.dart
[OK] /content/subaru_ssm2_fixed/lib/identity.dart
[OK] /content/subaru_ssm2_fixed/lib/maplab_link.dart
[OK] /content/subaru_ssm2_fixed/lib/pids.dart

=== flutter_version ===
> /content/flutter/bin/flutter --version --machine
Журнал: /conte

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>